In [ ]:
import sys
print(sys.executable)

In [ ]:
import os
import gc
import json
import zipfile
import random
from pathlib import Path
from collections import Counter, defaultdict, deque
from io import BytesIO

import numpy as np
import torch

from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageDraw


# Reproducibility
SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", device)
print("Random seed:", SEED)

In [ ]:
DATA_ROOT = Path(
    os.environ.get(
        "V2X_SIM_ROOT",
        Path.cwd() / "data" / "V2X-Sim-2.0-final"
    )
)

OUTPUT_DIR = Path(
    os.environ.get(
        "V2X_OUTPUT_DIR",
        Path.cwd() / "outputs"
    )
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT:", DATA_ROOT)
print("DATA_ROOT exists:", DATA_ROOT.exists())
print("OUTPUT_DIR:", OUTPUT_DIR)
print("OUTPUT_DIR exists:", OUTPUT_DIR.exists())

if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f"DATA_ROOT not found: {DATA_ROOT}\n"
        "Please place V2X-Sim under ./data/V2X-Sim-2.0-final "
        "or set the V2X_SIM_ROOT environment variable."
    )

In [ ]:
# Camera and LiDAR archive files


zip_files = [
    p for p in DATA_ROOT.iterdir()
    if p.is_file() and p.suffix.lower() == ".zip"
]

camera_zip_files = sorted([
    p for p in zip_files
    if p.stem.lower().startswith("cam")
])

lidar_zip_files = sorted([
    p for p in zip_files
    if p.stem.lower().startswith("lidar")
])

print(f"Camera archives found: {len(camera_zip_files)}")
print(f"LiDAR archives found: {len(lidar_zip_files)}")

if len(camera_zip_files) == 0:
    raise FileNotFoundError("No camera ZIP archives were found in DATA_ROOT.")

if len(lidar_zip_files) == 0:
    raise FileNotFoundError("No LiDAR ZIP archives were found in DATA_ROOT.")

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}

camera_image_index = []

for zip_path in camera_zip_files:
    with zipfile.ZipFile(zip_path, "r") as z:
        for name in z.namelist():
            if Path(name).suffix.lower() in IMAGE_EXTENSIONS:
                camera_image_index.append(
                    (
                        zip_path,
                        zip_path.name,
                        zip_path.stem,
                        name,
                        Path(name).name,
                    )
                )

print(f"Camera images indexed: {len(camera_image_index)}")

if len(camera_image_index) == 0:
    raise RuntimeError("No camera images were found in the camera archives.")

In [ ]:
count_per_zip = defaultdict(int)

for item in camera_image_index:
    zip_path, zip_name, zip_stem, inner_path, file_name = item
    count_per_zip[zip_name] += 1

print("Images per camera archive:")

for zip_name in sorted(count_per_zip):
    print(f"{zip_name:20s}: {count_per_zip[zip_name]}")

In [ ]:
# Identify camera sensor folders inside each archive
sensor_folders_by_zip = defaultdict(set)

for zip_path in camera_zip_files:
    with zipfile.ZipFile(zip_path, "r") as z:
        names = z.namelist()

    for name in names:
        parts = Path(name).parts

        if len(parts) >= 2 and parts[0] == "sweeps":
            sensor_folder = parts[1]

            if sensor_folder.startswith("CAM"):
                sensor_folders_by_zip[zip_path.name].add(sensor_folder)

print("Camera sensor folders found:")

for zip_name in sorted(sensor_folders_by_zip):
    print(f"\n{zip_name}")

    for sensor in sorted(sensor_folders_by_zip[zip_name]):
        print(f"  {sensor}")

In [ ]:
# Cell 8: Build global camera index for vehicle and RSU

import re

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}

camera_index = {}
duplicate_keys = defaultdict(list)

# Vehicle camera:
# CAM_BACK_RIGHT_id_1
pattern_vehicle_sensor = re.compile(
    r"^(CAM_[A-Z_]+)_id_(\d+)$"
)

# Infrastructure camera example:
# CAM_id_0_0, CAM_id_0_1, CAM_id_0_2, CAM_id_0_3
pattern_rsu_sensor = re.compile(
    r"^CAM_id_(\d+)_(\d+)$"
)

# Frame:
# scene_1_000082.jpg
pattern_frame = re.compile(
    r"scene_(\d+)_(\d+)\.(jpg|jpeg|png)$",
    re.IGNORECASE
)

num_images = 0
bad_entries = 0

for zip_path in camera_zip_files:

    with zipfile.ZipFile(zip_path, "r") as z:

        for name in z.namelist():

            if Path(name).suffix.lower() not in IMAGE_EXTENSIONS:
                continue

            parts = Path(name).parts

            if len(parts) < 3:
                bad_entries += 1
                continue

            sensor_folder = parts[1]
            file_name = parts[-1]

            m_frame = pattern_frame.match(file_name)

            if m_frame is None:
                bad_entries += 1
                continue

            scene_id = int(m_frame.group(1))
            frame_id = int(m_frame.group(2))


            m_vehicle = pattern_vehicle_sensor.match(sensor_folder)

            if m_vehicle is not None:

                cam_view = m_vehicle.group(1)
                agent_id = int(m_vehicle.group(2))
                node_type = "vehicle"

            else:

                m_rsu = pattern_rsu_sensor.match(sensor_folder)

                if m_rsu is not None:

                    agent_id = int(m_rsu.group(1))
                    rsu_cam_id = int(m_rsu.group(2))

                    cam_view = f"CAM_INT_{rsu_cam_id}"
                    node_type = "rsu"

                else:
                    bad_entries += 1
                    continue

            key = (
                agent_id,
                scene_id,
                frame_id,
                cam_view
            )

            item = {
                "zip_path": zip_path,
                "zip_name": zip_path.name,
                "inner_path": name,
                "sensor_folder": sensor_folder,
                "cam_view": cam_view,
                "agent_id": agent_id,
                "scene_id": scene_id,
                "frame_id": frame_id,
                "node_type": node_type
            }

            if key in camera_index:
                duplicate_keys[key].append(item)
            else:
                camera_index[key] = item

            num_images += 1


print(f"Parsed camera images: {num_images}")
print(f"Unparsed entries: {bad_entries}")
print(f"Unique camera keys: {len(camera_index)}")
print(f"Duplicate keys: {len(duplicate_keys)}")

In [ ]:
views_per_agent = defaultdict(set)
frames_per_agent_view = defaultdict(int)
node_type_per_agent = defaultdict(set)

for (agent_id, scene_id, frame_id, cam_view), item in camera_index.items():

    views_per_agent[agent_id].add(cam_view)

    frames_per_agent_view[
        (agent_id, cam_view)
    ] += 1

    node_type_per_agent[agent_id].add(
        item["node_type"]
    )

print("Camera views per agent:")

for agent_id in sorted(views_per_agent):

    print(f"\nAgent ID: {agent_id}")
    print(
        "Node type:",
        sorted(node_type_per_agent[agent_id])
    )

    for view in sorted(views_per_agent[agent_id]):

        count = frames_per_agent_view[
            (agent_id, view)
        ]

        print(
            f"  {view:22s}: {count} images"
        )

In [ ]:
vehicle_views = [
    "CAM_FRONT",
    "CAM_FRONT_RIGHT",
    "CAM_BACK_RIGHT",
    "CAM_BACK",
    "CAM_BACK_LEFT",
    "CAM_FRONT_LEFT"
]

rsu_views = [
    "CAM_INT_0",
    "CAM_INT_1",
    "CAM_INT_2",
    "CAM_INT_3"
]

sample_to_views = defaultdict(set)

for (agent_id, scene_id, frame_id, cam_view), item in camera_index.items():
    sample_key = (agent_id, scene_id, frame_id)
    sample_to_views[sample_key].add(cam_view)

complete_vehicle_samples = []
complete_rsu_samples = []

for sample_key, views in sample_to_views.items():

    if all(view in views for view in vehicle_views):
        complete_vehicle_samples.append(sample_key)

    if all(view in views for view in rsu_views):
        complete_rsu_samples.append(sample_key)

complete_vehicle_samples = sorted(complete_vehicle_samples)
complete_rsu_samples = sorted(complete_rsu_samples)

print(
    "Complete 6-camera vehicle samples:",
    len(complete_vehicle_samples)
)

print(
    "Complete 4-camera RSU samples:",
    len(complete_rsu_samples)
)

In [ ]:
# Match ego-vehicle samples with synchronized RSU samples

RSU_AGENT_ID = 0

rsu_scene_frame_set = {
    (scene_id, frame_id)
    for agent_id, scene_id, frame_id in complete_rsu_samples
    if agent_id == RSU_AGENT_ID
}

matched_ego_rsu_samples = []

for ego_agent_id, scene_id, frame_id in complete_vehicle_samples:

    if (scene_id, frame_id) in rsu_scene_frame_set:

        matched_ego_rsu_samples.append(
            (
                ego_agent_id,
                RSU_AGENT_ID,
                scene_id,
                frame_id
            )
        )

matched_ego_rsu_samples = sorted(
    matched_ego_rsu_samples
)

print(
    "Matched synchronized ego-RSU samples:",
    len(matched_ego_rsu_samples)
)

In [ ]:
def load_image_from_zip(zip_path, inner_path):
    """Load one RGB image directly from a ZIP archive."""
    with zipfile.ZipFile(zip_path, "r") as z:
        img_bytes = z.read(inner_path)

    return Image.open(BytesIO(img_bytes)).convert("RGB")


def load_rgb_image_from_zip(
    zip_path,
    inner_path,
    image_size=(256, 256)
):
    """Load, resize, normalize, and convert an RGB image to a tensor."""

    img = load_image_from_zip(
        zip_path,
        inner_path
    )

    img = img.resize(
        image_size,
        Image.BILINEAR
    )

    arr = (
        np.asarray(img)
        .astype(np.float32)
        / 255.0
    )

    # [H, W, 3] -> [3, H, W]
    tensor = torch.from_numpy(
        arr
    ).permute(2, 0, 1)

    return tensor


#  Select one synchronized sample for validation
ego_agent_id, rsu_agent_id, scene_id, frame_id = (
    matched_ego_rsu_samples[0]
)


#  Load the six ego-vehicle camera views
ego_camera_tensors = []

for view in vehicle_views:

    key = (
        ego_agent_id,
        scene_id,
        frame_id,
        view
    )

    item = camera_index[key]

    img_tensor = load_rgb_image_from_zip(
        zip_path=item["zip_path"],
        inner_path=item["inner_path"],
        image_size=(256, 256)
    )

    ego_camera_tensors.append(
        img_tensor
    )

ego_cameras = torch.stack(
    ego_camera_tensors,
    dim=0
)


# Load the four RSU camera views
rsu_camera_tensors = []

for view in rsu_views:

    key = (
        rsu_agent_id,
        scene_id,
        frame_id,
        view
    )

    item = camera_index[key]

    img_tensor = load_rgb_image_from_zip(
        zip_path=item["zip_path"],
        inner_path=item["inner_path"],
        image_size=(256, 256)
    )

    rsu_camera_tensors.append(
        img_tensor
    )

rsu_cameras = torch.stack(
    rsu_camera_tensors,
    dim=0
)

# Validate tensor dimensions
print("Ego camera tensor shape:", ego_cameras.shape)
print("RSU camera tensor shape:", rsu_cameras.shape)

assert ego_cameras.shape == (6, 3, 256, 256)
assert rsu_cameras.shape == (4, 3, 256, 256)

In [ ]:
# Pad RSU camera views to six common camera slots

# Ego vehicle already provides six valid camera views
ego_cameras_6 = ego_cameras

# RSU provides four valid views and is padded to six slots
rsu_cameras_6 = torch.zeros(
    (6, 3, 256, 256),
    dtype=rsu_cameras.dtype
)

rsu_cameras_6[:4] = rsu_cameras

# Camera-validity masks
ego_cam_mask = torch.tensor(
    [1, 1, 1, 1, 1, 1],
    dtype=torch.float32
)

rsu_cam_mask = torch.tensor(
    [1, 1, 1, 1, 0, 0],
    dtype=torch.float32
)

print("Ego camera tensor:", ego_cameras_6.shape)
print("RSU padded camera tensor:", rsu_cameras_6.shape)
print("Ego camera mask:", ego_cam_mask)
print("RSU camera mask:", rsu_cam_mask)

In [ ]:
# Multi-view camera feature encoder and attention aggregation

import torch.nn as nn
import torch.nn.functional as F

class ConvBNReLU(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size=3,
        stride=1,
        padding=None
    ):
        super().__init__()

        if padding is None:
            padding = kernel_size // 2

        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=kernel_size,
                stride=stride,
                padding=padding,
                bias=False
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class CameraBranchEncoder(nn.Module):
    """
    Shared encoder applied independently to all valid camera views.
    """

    def __init__(
        self,
        in_channels=3,
        feature_channels=64
    ):
        super().__init__()

        self.encoder = nn.Sequential(
            ConvBNReLU(
                in_channels,
                32,
                kernel_size=5,
                stride=2
            ),
            ConvBNReLU(
                32,
                32,
                kernel_size=3,
                stride=1
            ),
            nn.MaxPool2d(2),

            ConvBNReLU(
                32,
                64,
                kernel_size=3,
                stride=1
            ),
            ConvBNReLU(
                64,
                feature_channels,
                kernel_size=3,
                stride=1
            ),
            nn.MaxPool2d(2),

            ConvBNReLU(
                feature_channels,
                feature_channels,
                kernel_size=3,
                stride=1
            )
        )

    def forward(self, images):
        # images: [B, N, 3, H, W]

        B, N, C, H, W = images.shape

        x = images.reshape(
            B * N,
            C,
            H,
            W
        )

        feat = self.encoder(x)

        _, C_feat, H_feat, W_feat = feat.shape

        feat = feat.reshape(
            B,
            N,
            C_feat,
            H_feat,
            W_feat
        )

        return feat

class MultiCameraAttentionAggregator(nn.Module):
    """
    Masked spatial attention across camera views.
    """

    def __init__(
        self,
        feature_channels=64
    ):
        super().__init__()

        self.score_net = nn.Sequential(
            nn.Conv2d(
                feature_channels,
                feature_channels // 2,
                kernel_size=1
            ),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                feature_channels // 2,
                1,
                kernel_size=1
            )
        )

    def forward(
        self,
        cam_feats,
        cam_mask
    ):
        # cam_feats: [B, N, C, H, W]
        # cam_mask:  [B, N]

        B, N, C, H, W = cam_feats.shape

        x = cam_feats.reshape(
            B * N,
            C,
            H,
            W
        )

        scores = self.score_net(x)

        scores = scores.reshape(
            B,
            N,
            1,
            H,
            W
        )

        # Exclude padded/invalid camera views
        cam_mask = cam_mask.float().view(
            B,
            N,
            1,
            1,
            1
        )

        scores = scores.masked_fill(
            cam_mask == 0,
            -1e9
        )

        # Normalize attention across camera views
        weights = torch.softmax(
            scores,
            dim=1
        )

        fused = torch.sum(
            weights * cam_feats,
            dim=1
        )

        return fused, weights


class CameraBranch(nn.Module):
    """
    Complete multi-view camera feature branch.
    """

    def __init__(
        self,
        feature_channels=64
    ):
        super().__init__()

        self.encoder = CameraBranchEncoder(
            in_channels=3,
            feature_channels=feature_channels
        )

        self.aggregator = (
            MultiCameraAttentionAggregator(
                feature_channels=feature_channels
            )
        )

    def forward(
        self,
        cameras,
        cam_mask
    ):
        cam_feats = self.encoder(cameras)

        fused_cam_feat, cam_weights = (
            self.aggregator(
                cam_feats,
                cam_mask
            )
        )

        return {
            "fused_camera_feature": fused_cam_feat,
            "camera_features": cam_feats,
            "camera_attention_weights": cam_weights
        }


camera_branch = CameraBranch(
    feature_channels=64
).to(device)

print("Camera branch initialized.")

In [ ]:
camera_branch.eval()

ego_batch = ego_cameras_6.unsqueeze(0).to(device)
rsu_batch = rsu_cameras_6.unsqueeze(0).to(device)

ego_mask_batch = ego_cam_mask.unsqueeze(0).to(device)
rsu_mask_batch = rsu_cam_mask.unsqueeze(0).to(device)

with torch.no_grad():

    ego_cam_out = camera_branch(
        cameras=ego_batch,
        cam_mask=ego_mask_batch
    )

    rsu_cam_out = camera_branch(
        cameras=rsu_batch,
        cam_mask=rsu_mask_batch
    )

print(
    "Ego fused camera feature:",
    ego_cam_out["fused_camera_feature"].shape
)

print(
    "RSU fused camera feature:",
    rsu_cam_out["fused_camera_feature"].shape
)

print(
    "Ego attention normalization:",
    ego_cam_out["camera_attention_weights"]
    .sum(dim=1)
    .mean()
    .item()
)

print(
    "RSU attention normalization:",
    rsu_cam_out["camera_attention_weights"]
    .sum(dim=1)
    .mean()
    .item()
)

assert ego_cam_out["fused_camera_feature"].shape == (
    1, 64, 32, 32
)

assert rsu_cam_out["fused_camera_feature"].shape == (
    1, 64, 32, 32
)

In [ ]:
print("RSU attention per camera:")

rsu_w = (
    rsu_cam_out["camera_attention_weights"]
    .detach()
    .cpu()
)

for i in range(6):
    mean_attention = rsu_w[0, i].mean().item()
    print(f"Camera {i}: mean attention = {mean_attention:.6f}")

In [ ]:
# LiDAR archive validation

lidar_archive_info = {}

for zip_path in lidar_zip_files:
    with zipfile.ZipFile(zip_path, "r") as z:
        names = z.namelist()

    if len(names) == 0:
        raise RuntimeError(
            f"Empty LiDAR archive: {zip_path.name}"
        )

    suffix_counter = Counter(
        Path(name).suffix.lower()
        for name in names
    )

    lidar_archive_info[zip_path.name] = {
        "num_files": len(names),
        "extensions": dict(suffix_counter)
    }

    print(
        f"{zip_path.name}: "
        f"{len(names)} files, "
        f"extensions = {dict(suffix_counter)}"
    )

In [ ]:
# LiDAR sensor folder discovery
lidar_sensor_folders_by_zip = defaultdict(set)

for zip_path in lidar_zip_files:

    # Skip semantic-label archive if present
    if zip_path.name.lower() == "lidarseg.zip":
        continue

    with zipfile.ZipFile(zip_path, "r") as z:

        for name in z.namelist():

            parts = Path(name).parts

            if len(parts) >= 2 and parts[0] == "sweeps":

                sensor_folder = parts[1]

                if "LIDAR" in sensor_folder:
                    lidar_sensor_folders_by_zip[
                        zip_path.name
                    ].add(sensor_folder)


print("LiDAR sensor folders found:")

for zip_name in sorted(lidar_sensor_folders_by_zip):

    print(f"\n{zip_name}")

    for sensor in sorted(
        lidar_sensor_folders_by_zip[zip_name]
    ):
        print(f"  {sensor}")

In [ ]:
lidar_index = {}
lidar_duplicates = defaultdict(list)

# Sensor folder:
pattern_lidar_sensor = re.compile(
    r"^([A-Z]*LIDAR_TOP)_id_(\d+)$"
)

# Frame:
pattern_lidar_frame = re.compile(
    r"scene_(\d+)_(\d+)\.pcd\.bin$",
    re.IGNORECASE
)

num_lidar_files = 0
bad_lidar_entries = 0

for zip_path in lidar_zip_files:

    if zip_path.name.lower() == "lidarseg.zip":
        continue

    with zipfile.ZipFile(zip_path, "r") as z:

        for name in z.namelist():

            if not name.lower().endswith(".pcd.bin"):
                continue

            parts = Path(name).parts

            if len(parts) < 3:
                bad_lidar_entries += 1
                continue

            sensor_folder = parts[1]
            file_name = parts[-1]

            m_sensor = pattern_lidar_sensor.match(sensor_folder)
            m_frame = pattern_lidar_frame.match(file_name)

            if m_sensor is None or m_frame is None:
                bad_lidar_entries += 1
                continue

            lidar_type = m_sensor.group(1)
            agent_id = int(m_sensor.group(2))

            scene_id = int(m_frame.group(1))
            frame_id = int(m_frame.group(2))

            key = (
                agent_id,
                scene_id,
                frame_id,
                lidar_type
            )

            item = {
                "zip_path": zip_path,
                "zip_name": zip_path.name,
                "inner_path": name,
                "sensor_folder": sensor_folder,
                "lidar_type": lidar_type,
                "agent_id": agent_id,
                "scene_id": scene_id,
                "frame_id": frame_id
            }

            if key in lidar_index:
                lidar_duplicates[key].append(item)
            else:
                lidar_index[key] = item

            num_lidar_files += 1

print("Parsed LiDAR files:", num_lidar_files)
print("Unparsed LiDAR entries:", bad_lidar_entries)
print("Unique LiDAR keys:", len(lidar_index))
print("Duplicate LiDAR keys:", len(lidar_duplicates))

In [ ]:
lidar_types_per_agent = defaultdict(set)
lidar_count_per_agent_type = defaultdict(int)

for (agent_id, scene_id, frame_id, lidar_type), item in lidar_index.items():

    lidar_types_per_agent[agent_id].add(lidar_type)

    lidar_count_per_agent_type[
        (agent_id, lidar_type)
    ] += 1


print("LiDAR availability per agent:")

for agent_id in sorted(lidar_types_per_agent):

    print(f"\nAgent ID: {agent_id}")

    for lidar_type in sorted(
        lidar_types_per_agent[agent_id]
    ):

        count = lidar_count_per_agent_type[
            (agent_id, lidar_type)
        ]

        print(
            f"  {lidar_type:18s}: {count} files"
        )

In [ ]:
print("Selected sample:")
print("ego:", ego_agent_id, "rsu:", rsu_agent_id, "scene:", scene_id, "frame:", frame_id)

print("\nRaw LiDAR exists:")
print("ego LIDAR_TOP:", (ego_agent_id, scene_id, frame_id, "LIDAR_TOP") in lidar_index)
print("rsu LIDAR_TOP:", (rsu_agent_id, scene_id, frame_id, "LIDAR_TOP") in lidar_index)

print("\nSemantic LiDAR exists:")
print("ego SEMLIDAR_TOP:", (ego_agent_id, scene_id, frame_id, "SEMLIDAR_TOP") in lidar_index)
print("rsu SEMLIDAR_TOP:", (rsu_agent_id, scene_id, frame_id, "SEMLIDAR_TOP") in lidar_index)

In [ ]:
def read_lidar_bin_from_zip(zip_path, inner_path):
    with zipfile.ZipFile(zip_path, "r") as z:
        raw = z.read(inner_path)

    return np.frombuffer(raw, dtype=np.float32)


def inspect_lidar_item(item, title):
    print(f"\n{title}")
    print("Zip:", item["zip_name"])
    print("Inner:", item["inner_path"])
    print("LiDAR type:", item["lidar_type"])
    print("Agent:", item["agent_id"])
    print("Scene:", item["scene_id"])
    print("Frame:", item["frame_id"])

    raw_arr = read_lidar_bin_from_zip(
        item["zip_path"],
        item["inner_path"]
    )

    print("Raw float32 length:", raw_arr.shape[0])

    if raw_arr.shape[0] % 5 == 0:
        pts = raw_arr.reshape(-1, 5)

        print("Point shape:", pts.shape)
        print("First point:", pts[0])
        print("Min:", pts.min(axis=0))
        print("Max:", pts.max(axis=0))

    else:
        print("Cannot reshape to [-1, 5]")


raw_key = (
    ego_agent_id,
    scene_id,
    frame_id,
    "LIDAR_TOP"
)

sem_key = (
    ego_agent_id,
    scene_id,
    frame_id,
    "SEMLIDAR_TOP"
)

raw_item = lidar_index[raw_key]
sem_item = lidar_index[sem_key]

inspect_lidar_item(
    raw_item,
    "RAW LiDAR input"
)

inspect_lidar_item(
    sem_item,
    "Semantic LiDAR reference"
)

In [ ]:
lidar_scene_frame_agent = defaultdict(set)

for (agent_id, scene_id, frame_id, lidar_type), item in lidar_index.items():
    lidar_scene_frame_agent[
        (agent_id, scene_id, frame_id)
    ].add(lidar_type)


matched_camera_lidar_vehicle_samples = []

for ego_agent_id, scene_id, frame_id in complete_vehicle_samples:

    lidar_types = lidar_scene_frame_agent.get(
        (ego_agent_id, scene_id, frame_id),
        set()
    )

    # Require geometric LiDAR input
    if "LIDAR_TOP" in lidar_types:
        matched_camera_lidar_vehicle_samples.append(
            (
                ego_agent_id,
                scene_id,
                frame_id,
                sorted(lidar_types)
            )
        )


print(
    "Vehicle samples with complete 6-camera input + LIDAR_TOP:",
    len(matched_camera_lidar_vehicle_samples)
)

In [ ]:
# Match synchronized ego-RSU camera and LiDAR samples

RAW_LIDAR_TYPE = "LIDAR_TOP"

matched_ego_rsu_camera_lidar_samples = []

for (
    ego_agent_id,
    rsu_agent_id,
    scene_id,
    frame_id
) in matched_ego_rsu_samples:

    ego_lidar_key = (
        ego_agent_id,
        scene_id,
        frame_id,
        RAW_LIDAR_TYPE
    )

    rsu_lidar_key = (
        rsu_agent_id,
        scene_id,
        frame_id,
        RAW_LIDAR_TYPE
    )

    if (
        ego_lidar_key in lidar_index
        and rsu_lidar_key in lidar_index
    ):
        matched_ego_rsu_camera_lidar_samples.append(
            (
                ego_agent_id,
                rsu_agent_id,
                scene_id,
                frame_id
            )
        )

matched_ego_rsu_camera_lidar_samples = sorted(
    matched_ego_rsu_camera_lidar_samples
)

print(
    "Matched synchronized ego-RSU "
    "camera + LiDAR samples:",
    len(matched_ego_rsu_camera_lidar_samples)
)

In [ ]:
def read_lidar_points_from_zip(zip_path, inner_path):
    """
    Read one V2X-Sim LiDAR file from a ZIP archive.

    The stored point records contain five float32 values.
    The first four channels are used as:
    x, y, z, intensity.
    """

    with zipfile.ZipFile(zip_path, "r") as z:
        raw = z.read(inner_path)

    arr = np.frombuffer(
        raw,
        dtype=np.float32
    ).copy()

    if arr.size % 5 != 0:
        raise ValueError(
            f"Cannot reshape LiDAR file to [N, 5]. "
            f"Raw length = {arr.size}"
        )

    points = arr.reshape(-1, 5)

    # Retain geometric coordinates and intensity
    points = points[:, :4]

    return points


#  Select one synchronized ego-RSU multimodal sample

(
    ego_agent_id,
    rsu_agent_id,
    scene_id,
    frame_id
) = matched_ego_rsu_camera_lidar_samples[0]


ego_lidar_item = lidar_index[
    (
        ego_agent_id,
        scene_id,
        frame_id,
        "LIDAR_TOP"
    )
]

rsu_lidar_item = lidar_index[
    (
        rsu_agent_id,
        scene_id,
        frame_id,
        "LIDAR_TOP"
    )
]


#  Load raw LiDAR point clouds

ego_points = read_lidar_points_from_zip(
    ego_lidar_item["zip_path"],
    ego_lidar_item["inner_path"]
)

rsu_points = read_lidar_points_from_zip(
    rsu_lidar_item["zip_path"],
    rsu_lidar_item["inner_path"]
)


print("Ego LiDAR points:", ego_points.shape)
print("RSU LiDAR points:", rsu_points.shape)

assert ego_points.shape[1] == 4
assert rsu_points.shape[1] == 4

In [ ]:
#  Convert LiDAR point clouds to 4-channel BEV tensors

def lidar_points_to_bev(
    points,
    x_range=(-50.0, 50.0),
    y_range=(-50.0, 50.0),
    z_range=(-3.0, 5.0),
    bev_size=(256, 256)
):
    """
    Convert a LiDAR point cloud into a four-channel BEV tensor:
    occupancy, normalized height, density, and intensity.
    """

    H, W = bev_size

    x = points[:, 0]
    y = points[:, 1]
    z = points[:, 2]

    # Keep points inside the predefined BEV region
    valid = (
        (x >= x_range[0]) & (x <= x_range[1]) &
        (y >= y_range[0]) & (y <= y_range[1]) &
        (z >= z_range[0]) & (z <= z_range[1])
    )

    points = points[valid]

    bev = np.zeros(
        (4, H, W),
        dtype=np.float32
    )

    if points.shape[0] == 0:
        return torch.from_numpy(bev)

    x = points[:, 0]
    y = points[:, 1]
    z = points[:, 2]

    intensity = (
        points[:, 3]
        if points.shape[1] > 3
        else np.ones_like(x, dtype=np.float32)
    )

    x_idx = (
        (x - x_range[0]) /
        (x_range[1] - x_range[0]) *
        (W - 1)
    ).astype(np.int32)

    y_idx = (
        (y - y_range[0]) /
        (y_range[1] - y_range[0]) *
        (H - 1)
    ).astype(np.int32)

    x_idx = np.clip(x_idx, 0, W - 1)
    y_idx = np.clip(y_idx, 0, H - 1)

    # Flip vertical axis for BEV image coordinates
    y_idx = H - 1 - y_idx

    # Channel 0: occupancy
    bev[0, y_idx, x_idx] = 1.0

    # Channel 1: normalized maximum height
    z_norm = (
        (z - z_range[0]) /
        (z_range[1] - z_range[0])
    )

    z_norm = np.clip(
        z_norm,
        0.0,
        1.0
    )

    np.maximum.at(
        bev[1],
        (y_idx, x_idx),
        z_norm
    )

    # Channel 2: point density
    counts = np.zeros(
        (H, W),
        dtype=np.float32
    )

    np.add.at(
        counts,
        (y_idx, x_idx),
        1.0
    )

    bev[2] = np.clip(
        np.log1p(counts) / np.log(64),
        0.0,
        1.0
    )

    # Channel 3: normalized intensity
    intensity = np.nan_to_num(
        intensity,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    ).astype(np.float32)

    if intensity.max() > intensity.min():
        intensity = (
            (intensity - intensity.min()) /
            (intensity.max() - intensity.min())
        )
    else:
        intensity = np.zeros_like(intensity)

    intensity_map = np.zeros(
        (H, W),
        dtype=np.float32
    )

    np.maximum.at(
        intensity_map,
        (y_idx, x_idx),
        intensity
    )

    bev[3] = intensity_map

    return torch.from_numpy(bev)


# Convert synchronized ego and RSU LiDAR samples

ego_bev = lidar_points_to_bev(
    ego_points,
    bev_size=(256, 256)
)

rsu_bev = lidar_points_to_bev(
    rsu_points,
    bev_size=(256, 256)
)

print("Ego BEV tensor:", ego_bev.shape)
print("RSU BEV tensor:", rsu_bev.shape)

assert ego_bev.shape == (4, 256, 256)
assert rsu_bev.shape == (4, 256, 256)

In [ ]:
print("ego occupied cells:", ego_bev[0].sum().item())
print("rsu occupied cells:", rsu_bev[0].sum().item())

In [ ]:
# Figure Vehicle 6-camera views

from PIL import Image, ImageDraw, ImageFont
from IPython.display import Image as IPImage, display

debug_dir = DATA_ROOT / "debug_outputs"
debug_dir.mkdir(exist_ok=True)

def make_camera_grid(camera_images, title, save_name):
    cell_w, cell_h = 300, 220
    cols, rows = 3, 2

    fig = Image.new("RGB", (cols * cell_w, rows * cell_h + 50), "white")
    draw = ImageDraw.Draw(fig)

    try:
        font_title = ImageFont.truetype("arial.ttf", 24)
        font_panel = ImageFont.truetype("arial.ttf", 16)
    except:
        font_title = None
        font_panel = None

    draw.text((260, 12), title, fill=(0, 0, 0), font=font_title)

    for i, (view, img) in enumerate(camera_images[:6]):
        img = img.convert("RGB").resize((280, 160))

        col = i % cols
        row = i // cols

        x = col * cell_w
        y = row * cell_h + 50

        panel = Image.new("RGB", (cell_w, cell_h), "white")
        pdraw = ImageDraw.Draw(panel)

        pdraw.text((12, 10), view, fill=(0, 0, 0), font=font_panel)
        panel.paste(img, (10, 45))
        pdraw.rectangle([5, 5, cell_w - 5, cell_h - 5], outline=(160, 160, 160), width=2)

        fig.paste(panel, (x, y))

    save_path = debug_dir / save_name
    fig.save(save_path)

    print("Saved:", save_path)
    display(IPImage(filename=str(save_path)))

make_camera_grid(
    ego_camera_images,
    "",
    "Figure_A_Vehicle_6_Camera_Views.png"
)

In [ ]:
# Figure RSU 4-camera views

from PIL import Image, ImageDraw, ImageFont
from IPython.display import Image as IPImage, display

debug_dir = DATA_ROOT / "debug_outputs"
debug_dir.mkdir(exist_ok=True)

def make_rsu_camera_grid(rsu_camera_images, title, save_name):
    cell_w, cell_h = 340, 240
    cols, rows = 2, 2

    fig = Image.new("RGB", (cols * cell_w, rows * cell_h + 50), "white")
    draw = ImageDraw.Draw(fig)

    try:
        font_title = ImageFont.truetype("arial.ttf", 24)
        font_panel = ImageFont.truetype("arial.ttf", 16)
    except:
        font_title = None
        font_panel = None

    draw.text((170, 12), title, fill=(0, 0, 0), font=font_title)

    for i, (view, img) in enumerate(rsu_camera_images[:4]):
        img = img.convert("RGB").resize((310, 175))

        col = i % cols
        row = i // cols

        x = col * cell_w
        y = row * cell_h + 50

        panel = Image.new("RGB", (cell_w, cell_h), "white")
        pdraw = ImageDraw.Draw(panel)

        pdraw.text((12, 10), view, fill=(0, 0, 0), font=font_panel)
        panel.paste(img, (15, 45))
        pdraw.rectangle([5, 5, cell_w - 5, cell_h - 5], outline=(160, 160, 160), width=2)

        fig.paste(panel, (x, y))

    save_path = debug_dir / save_name
    fig.save(save_path)

    print("Saved:", save_path)
    display(IPImage(filename=str(save_path)))

make_rsu_camera_grid(
    rsu_camera_images,
    "",
    "Figure_B_RSU_4_Camera_Views.png"
)

In [ ]:
# Figure Vehicle-RSU Cooperative LiDAR BEV

from PIL import Image, ImageDraw, ImageFont
import numpy as np
from IPython.display import Image as IPImage, display

debug_dir = DATA_ROOT / "debug_outputs"
debug_dir.mkdir(exist_ok=True)

def points_to_bev_image(points, size=(360, 360), color=(0, 0, 255), max_points=25000):
    pts = np.asarray(points)[:, :2]
    pts = pts[np.isfinite(pts).all(axis=1)]

    if len(pts) > max_points:
        idx = np.random.choice(len(pts), max_points, replace=False)
        pts = pts[idx]

    x = pts[:, 0]
    y = pts[:, 1]

    x_min, x_max = -70, 70
    y_min, y_max = -70, 70

    W, H = size
    img = Image.new("RGB", size, "white")
    draw = ImageDraw.Draw(img)

    xs = ((x - x_min) / (x_max - x_min + 1e-6) * (W - 1)).astype(int)
    ys = ((y_max - y) / (y_max - y_min + 1e-6) * (H - 1)).astype(int)

    for px, py in zip(xs, ys):
        if 0 <= px < W and 0 <= py < H:
            draw.point((int(px), int(py)), fill=color)

    return img

def overlay_bev(ego_points, rsu_points, size=(360, 360), max_points=25000):
    ego = np.asarray(ego_points)[:, :2]
    rsu = np.asarray(rsu_points)[:, :2]

    if len(ego) > max_points:
        ego = ego[np.random.choice(len(ego), max_points, replace=False)]

    if len(rsu) > max_points:
        rsu = rsu[np.random.choice(len(rsu), max_points, replace=False)]

    x_min, x_max = -70, 70
    y_min, y_max = -70, 70
    W, H = size

    img = Image.new("RGB", size, "white")
    draw = ImageDraw.Draw(img)

    def draw_points(pts, color):
        x = pts[:, 0]
        y = pts[:, 1]

        xs = ((x - x_min) / (x_max - x_min + 1e-6) * (W - 1)).astype(int)
        ys = ((y_max - y) / (y_max - y_min + 1e-6) * (H - 1)).astype(int)

        for px, py in zip(xs, ys):
            if 0 <= px < W and 0 <= py < H:
                draw.point((int(px), int(py)), fill=color)

    draw_points(rsu, (220, 60, 60))   # RSU = red
    draw_points(ego, (40, 80, 220))   # Vehicle = blue

    return img

def make_panel(title, img, cell_size=(400, 430)):
    panel = Image.new("RGB", cell_size, "white")
    draw = ImageDraw.Draw(panel)

    try:
        font_panel = ImageFont.truetype("arial.ttf", 18)
    except:
        font_panel = None

    draw.text((15, 15), title, fill=(0, 0, 0), font=font_panel)
    panel.paste(img, (20, 55))
    draw.rectangle([6, 6, cell_size[0] - 6, cell_size[1] - 6], outline=(150, 150, 150), width=2)

    return panel

ego_bev_img = points_to_bev_image(ego_points, color=(40, 80, 220))
rsu_bev_img = points_to_bev_image(rsu_points, color=(220, 60, 60))
coop_bev_img = overlay_bev(ego_points, rsu_points)

panel_a = make_panel("(a) Vehicle LiDAR BEV", ego_bev_img)
panel_b = make_panel("(b) RSU LiDAR BEV", rsu_bev_img)
panel_c = make_panel("(c) Cooperative BEV Overlay", coop_bev_img)

fig = Image.new("RGB", (1200, 500), "white")
draw = ImageDraw.Draw(fig)

try:
    font_main = ImageFont.truetype("arial.ttf", 26)
    font_legend = ImageFont.truetype("arial.ttf", 16)
except:
    font_main = None
    font_legend = None

draw.text((350, 15), "", fill=(0, 0, 0), font=font_main)

fig.paste(panel_a, (0, 60))
fig.paste(panel_b, (400, 60))
fig.paste(panel_c, (800, 60))

# Legend
draw.rectangle([840, 455, 855, 470], fill=(40, 80, 220))
draw.text((860, 452), "Vehicle LiDAR", fill=(0, 0, 0), font=font_legend)

draw.rectangle([980, 455, 995, 470], fill=(220, 60, 60))
draw.text((1000, 452), "RSU LiDAR", fill=(0, 0, 0), font=font_legend)

save_path = debug_dir / "Figure_C_Cooperative_LiDAR_BEV.png"
fig.save(save_path)

print("Saved:", save_path)
display(IPImage(filename=str(save_path)))

In [ ]:
# LiDAR BEV feature encoder

class LiDARBranchEncoder(nn.Module):
    """
    LiDAR BEV feature encoder.

    Input:
        lidar_bev: [B, 4, 256, 256]

    Output:
        lidar_feature: [B, 64, 32, 32]
    """

    def __init__(
        self,
        in_channels=4,
        feature_channels=64
    ):
        super().__init__()

        self.encoder = nn.Sequential(
            ConvBNReLU(
                in_channels,
                32,
                kernel_size=5,
                stride=2
            ),
            ConvBNReLU(
                32,
                32,
                kernel_size=3,
                stride=1
            ),
            nn.MaxPool2d(2),

            ConvBNReLU(
                32,
                64,
                kernel_size=3,
                stride=1
            ),
            ConvBNReLU(
                64,
                feature_channels,
                kernel_size=3,
                stride=1
            ),
            nn.MaxPool2d(2),

            ConvBNReLU(
                feature_channels,
                feature_channels,
                kernel_size=3,
                stride=1
            )
        )

    def forward(self, lidar_bev):
        return self.encoder(lidar_bev)


lidar_branch = LiDARBranchEncoder(
    in_channels=4,
    feature_channels=64
).to(device)

print("LiDAR branch initialized.")

In [ ]:
# Forward pass for ego and RSU LiDAR

lidar_branch.eval()

ego_lidar_batch = ego_bev.unsqueeze(0).to(device)
rsu_lidar_batch = rsu_bev.unsqueeze(0).to(device)

print("ego_lidar_batch:", ego_lidar_batch.shape)
print("rsu_lidar_batch:", rsu_lidar_batch.shape)

with torch.no_grad():
    ego_lidar_feat = lidar_branch(ego_lidar_batch)
    rsu_lidar_feat = lidar_branch(rsu_lidar_batch)

print("Ego LiDAR feature:", ego_lidar_feat.shape)
print("RSU LiDAR feature:", rsu_lidar_feat.shape)

In [ ]:
ego_cam_feat = ego_cam_out["fused_camera_feature"]
rsu_cam_feat = rsu_cam_out["fused_camera_feature"]

print("ego_cam_feat:", ego_cam_feat.shape)
print("rsu_cam_feat:", rsu_cam_feat.shape)

print("ego_lidar_feat:", ego_lidar_feat.shape)
print("rsu_lidar_feat:", rsu_lidar_feat.shape)

assert ego_cam_feat.shape == ego_lidar_feat.shape
assert rsu_cam_feat.shape == rsu_lidar_feat.shape

print("Camera and LiDAR features are compatible for local fusion.")

In [ ]:
# Local camera-LiDAR fusion module
class LocalCameraLiDARFusion(nn.Module):
    """
    Local camera-LiDAR fusion.

    Inputs:
        camera_feature: [B, C, H, W]
        lidar_feature:  [B, C, H, W]

    Outputs:
        fused_feature:  [B, C, H, W]
        fusion_weights: [B, 2, H, W]

    The two fusion-weight channels correspond to:
        0 -> camera contribution
        1 -> LiDAR contribution
    """

    def __init__(
        self,
        feature_channels=64
    ):
        super().__init__()

        # Modality-specific refinement
        self.camera_refine = nn.Sequential(
            ConvBNReLU(
                feature_channels,
                feature_channels,
                kernel_size=3
            ),
            ConvBNReLU(
                feature_channels,
                feature_channels,
                kernel_size=3
            )
        )

        self.lidar_refine = nn.Sequential(
            ConvBNReLU(
                feature_channels,
                feature_channels,
                kernel_size=3
            ),
            ConvBNReLU(
                feature_channels,
                feature_channels,
                kernel_size=3
            )
        )

        # Spatial modality-weight prediction
        self.weight_net = nn.Sequential(
            nn.Conv2d(
                feature_channels * 2,
                feature_channels,
                kernel_size=1
            ),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                feature_channels,
                2,
                kernel_size=1
            )
        )

        # Cross-modal fusion
        self.fusion_conv = nn.Sequential(
            ConvBNReLU(
                feature_channels * 2,
                feature_channels,
                kernel_size=3
            ),
            ConvBNReLU(
                feature_channels,
                feature_channels,
                kernel_size=3
            )
        )

        self.output_refine = nn.Sequential(
            ConvBNReLU(
                feature_channels,
                feature_channels,
                kernel_size=3
            )
        )

    def forward(
        self,
        camera_feature,
        lidar_feature
    ):

        cam = self.camera_refine(
            camera_feature
        )

        lidar = self.lidar_refine(
            lidar_feature
        )

        # Estimate spatial camera/LiDAR weights
        concat_feat = torch.cat(
            [cam, lidar],
            dim=1
        )

        fusion_logits = self.weight_net(
            concat_feat
        )

        fusion_weights = torch.softmax(
            fusion_logits,
            dim=1
        )

        cam_weight = fusion_weights[:, 0:1]
        lidar_weight = fusion_weights[:, 1:2]

        # Apply learned modality weights
        weighted_cam = cam_weight * cam
        weighted_lidar = lidar_weight * lidar

        fused_input = torch.cat(
            [weighted_cam, weighted_lidar],
            dim=1
        )

        fused_residual = self.fusion_conv(
            fused_input
        )

        # LiDAR-preserving residual fusion
        fused_feature = (
            fused_residual + lidar
        )

        fused_feature = self.output_refine(
            fused_feature
        )

        return {
            "fused_feature": fused_feature,
            "fusion_weights": fusion_weights,
            "camera_weight": cam_weight,
            "lidar_weight": lidar_weight
        }

local_fusion = LocalCameraLiDARFusion(
    feature_channels=64
).to(device)

print("Local camera-LiDAR fusion module initialized.")

In [ ]:
# Forward pass for ego and RSU local Camera-LiDAR fusion

local_fusion.eval()

with torch.no_grad():
    ego_local_out = local_fusion(
        camera_feature=ego_cam_feat,
        lidar_feature=ego_lidar_feat
    )

    rsu_local_out = local_fusion(
        camera_feature=rsu_cam_feat,
        lidar_feature=rsu_lidar_feat
    )

ego_local_fused_feat = ego_local_out["fused_feature"]
rsu_local_fused_feat = rsu_local_out["fused_feature"]

print("Ego local fused feature:", ego_local_fused_feat.shape)
print("RSU local fused feature:", rsu_local_fused_feat.shape)

print("\nEgo fusion weights:", ego_local_out["fusion_weights"].shape)
print("RSU fusion weights:", rsu_local_out["fusion_weights"].shape)

print("\nEgo camera weight mean:", ego_local_out["camera_weight"].mean().item())
print("Ego LiDAR weight mean:", ego_local_out["lidar_weight"].mean().item())
print("Ego weight sum:", ego_local_out["fusion_weights"].sum(dim=1).mean().item())

print("\nRSU camera weight mean:", rsu_local_out["camera_weight"].mean().item())
print("RSU LiDAR weight mean:", rsu_local_out["lidar_weight"].mean().item())
print("RSU weight sum:", rsu_local_out["fusion_weights"].sum(dim=1).mean().item())

In [ ]:
# Reliability-guided V2I cooperative fusion

class InitialV2IFusion(nn.Module):
    """
    Reliability-guided vehicle-to-infrastructure cooperative fusion.

    Inputs:
        ego_feature: [B, C, H, W]
        rsu_feature: [B, C, H, W]

    Outputs:
        cooperative_feature: [B, C, H, W]
        rsu_reliability_map: [B, 1, H, W]
    """

    def __init__(
        self,
        feature_channels=64
    ):
        super().__init__()

        # Agent-specific feature refinement
        self.ego_refine = nn.Sequential(
            ConvBNReLU(
                feature_channels,
                feature_channels,
                kernel_size=3
            ),
            ConvBNReLU(
                feature_channels,
                feature_channels,
                kernel_size=3
            )
        )

        self.rsu_refine = nn.Sequential(
            ConvBNReLU(
                feature_channels,
                feature_channels,
                kernel_size=3
            ),
            ConvBNReLU(
                feature_channels,
                feature_channels,
                kernel_size=3
            )
        )

        # Spatial RSU reliability estimation
        self.reliability_net = nn.Sequential(
            nn.Conv2d(
                feature_channels * 3,
                feature_channels,
                kernel_size=1
            ),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                feature_channels,
                feature_channels // 2,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                feature_channels // 2,
                1,
                kernel_size=1
            ),
            nn.Sigmoid()
        )

        # Cooperative residual fusion
        self.fusion_conv = nn.Sequential(
            ConvBNReLU(
                feature_channels * 2,
                feature_channels,
                kernel_size=3
            ),
            ConvBNReLU(
                feature_channels,
                feature_channels,
                kernel_size=3
            )
        )

        self.output_refine = nn.Sequential(
            ConvBNReLU(
                feature_channels,
                feature_channels,
                kernel_size=3
            )
        )

    def forward(
        self,
        ego_feature,
        rsu_feature
    ):

        ego = self.ego_refine(
            ego_feature
        )

        rsu = self.rsu_refine(
            rsu_feature
        )

        # Ego-RSU feature disagreement
        diff = torch.abs(
            ego - rsu
        )

        reliability_input = torch.cat(
            [ego, rsu, diff],
            dim=1
        )

        rsu_reliability_map = (
            self.reliability_net(
                reliability_input
            )
        )

        # Reliability-weighted RSU representation
        reliable_rsu = (
            rsu_reliability_map * rsu
        )

        fusion_input = torch.cat(
            [ego, reliable_rsu],
            dim=1
        )

        cooperative_residual = (
            self.fusion_conv(
                fusion_input
            )
        )

        # Ego-preserving residual cooperative fusion
        cooperative_feature = (
            ego + cooperative_residual
        )

        cooperative_feature = (
            self.output_refine(
                cooperative_feature
            )
        )

        return {
            "cooperative_feature": cooperative_feature,
            "rsu_reliability_map": rsu_reliability_map,
            "reliable_rsu_feature": reliable_rsu
        }


v2i_fusion = InitialV2IFusion(
    feature_channels=64
).to(device)

print("Reliability-guided V2I fusion module initialized.")

In [ ]:
# Cell 33: Validate initial V2I cooperative fusion

v2i_fusion.eval()

# Initial cooperative-fusion baseline:
# the RSU feature is fused before explicit RSU-to-ego alignment.
rsu_input_feat = rsu_local_fused_feat

with torch.no_grad():

    v2i_out = v2i_fusion(
        ego_feature=ego_local_fused_feat,
        rsu_feature=rsu_input_feat
    )


cooperative_v2i_feat = v2i_out[
    "cooperative_feature"
]

rsu_reliability_map = v2i_out[
    "rsu_reliability_map"
]


print(
    "Cooperative V2I feature:",
    cooperative_v2i_feat.shape
)

print(
    "RSU reliability map:",
    rsu_reliability_map.shape
)

print(
    "RSU reliability range:",
    f"{rsu_reliability_map.min().item():.4f} - "
    f"{rsu_reliability_map.max().item():.4f}"
)

print(
    "Mean RSU reliability:",
    f"{rsu_reliability_map.mean().item():.4f}"
)


assert cooperative_v2i_feat.shape == (
    1, 64, 32, 32
)

assert rsu_reliability_map.shape == (
    1, 1, 32, 32
)

In [ ]:
seg_zip_files = sorted([
    p for p in zip_files
    if p.stem.lower().startswith("seg")
])

if len(seg_zip_files) == 0:
    raise FileNotFoundError(
        "No segmentation ZIP archive was found in DATA_ROOT."
    )

seg_zip_path = seg_zip_files[0]

with zipfile.ZipFile(seg_zip_path, "r") as z:
    seg_names = z.namelist()

seg_npz_files = [
    name for name in seg_names
    if name.lower().endswith(".npz")
]

print("Segmentation archive:", seg_zip_path.name)
print("Segmentation label files:", len(seg_npz_files))

if len(seg_npz_files) == 0:
    raise RuntimeError(
        "No .npz segmentation label files were found."
    )

In [ ]:
test_seg_file = seg_npz_files[0]

with zipfile.ZipFile(seg_zip_path, "r") as z:
    raw = z.read(test_seg_file)

seg_data = np.load(
    BytesIO(raw)
)

print("Example segmentation file:", test_seg_file)
print("Available arrays:", seg_data.files)

for key in seg_data.files:
    arr = seg_data[key]

    print(
        f"{key}: "
        f"shape={arr.shape}, "
        f"dtype={arr.dtype}, "
        f"min={arr.min()}, "
        f"max={arr.max()}"
    )

In [ ]:
# Summarize segmentation sensor folders

seg_sensor_folders = defaultdict(int)

for name in seg_npz_files:

    parts = Path(name).parts

    if len(parts) >= 3 and parts[0] == "sweeps":

        sensor_folder = parts[1]

        seg_sensor_folders[
            sensor_folder
        ] += 1


print("Segmentation sensor folders:")

for folder, count in sorted(
    seg_sensor_folders.items()
):
    print(
        f"{folder:30s}: {count}"
    )

In [ ]:
#  LiDAR semantic-label index

lidarseg_candidates = [
    p for p in zip_files
    if p.name.lower() == "lidarseg.zip"
]

if len(lidarseg_candidates) == 0:
    raise FileNotFoundError(
        "lidarseg.zip was not found in DATA_ROOT."
    )

lidarseg_zip_path = lidarseg_candidates[0]

with zipfile.ZipFile(lidarseg_zip_path, "r") as z:
    lidarseg_names = z.namelist()

pattern_lidarseg = re.compile(
    r"scene_(\d+)_(\d+)_(\d+)\.pcd\.bin$",
    re.IGNORECASE
)

lidarseg_index = {}
bad_lidarseg_entries = 0

for name in lidarseg_names:

    if not name.lower().endswith(".pcd.bin"):
        continue

    file_name = Path(name).name
    match = pattern_lidarseg.match(file_name)

    if match is None:
        bad_lidarseg_entries += 1
        continue

    scene_id = int(match.group(1))
    agent_id = int(match.group(2))
    frame_id = int(match.group(3))

    key = (
        agent_id,
        scene_id,
        frame_id
    )

    lidarseg_index[key] = {
        "zip_path": lidarseg_zip_path,
        "inner_path": name,
        "agent_id": agent_id,
        "scene_id": scene_id,
        "frame_id": frame_id
    }


print(
    "Indexed LiDAR semantic-label files:",
    len(lidarseg_index)
)

print(
    "Unparsed LiDAR semantic-label entries:",
    bad_lidarseg_entries
)

if len(lidarseg_index) == 0:
    raise RuntimeError(
        "No LiDAR semantic-label files were indexed."
    )

In [ ]:
#  synchronized multimodal sample set

matched_multimodal_samples = []

for (
    ego_agent_id,
    rsu_agent_id,
    scene_id,
    frame_id
) in matched_ego_rsu_camera_lidar_samples:

    ego_label_key = (
        ego_agent_id,
        scene_id,
        frame_id
    )

    rsu_label_key = (
        rsu_agent_id,
        scene_id,
        frame_id
    )

    if (
        ego_label_key in lidarseg_index
        and rsu_label_key in lidarseg_index
    ):
        matched_multimodal_samples.append(
            (
                ego_agent_id,
                rsu_agent_id,
                scene_id,
                frame_id
            )
        )

matched_multimodal_samples = sorted(
    matched_multimodal_samples
)

print(
    "Complete synchronized multimodal samples:",
    len(matched_multimodal_samples)
)

assert len(matched_multimodal_samples) > 0

In [ ]:
# Validate LiDAR semantic labels

def read_lidarseg_labels_from_zip(zip_path, inner_path):
    """Read one uint8 semantic label per LiDAR point."""

    with zipfile.ZipFile(zip_path, "r") as z:
        raw = z.read(inner_path)

    return np.frombuffer(
        raw,
        dtype=np.uint8
    ).copy()


ego_agent_id, rsu_agent_id, scene_id, frame_id = (
    matched_train_samples[0]
)

ego_label_item = lidarseg_index[
    (ego_agent_id, scene_id, frame_id)
]

rsu_label_item = lidarseg_index[
    (rsu_agent_id, scene_id, frame_id)
]

ego_labels = read_lidarseg_labels_from_zip(
    ego_label_item["zip_path"],
    ego_label_item["inner_path"]
)

rsu_labels = read_lidarseg_labels_from_zip(
    rsu_label_item["zip_path"],
    rsu_label_item["inner_path"]
)

print("Ego semantic labels:", ego_labels.shape)
print("RSU semantic labels:", rsu_labels.shape)

print("Ego label IDs:", np.unique(ego_labels))
print("RSU label IDs:", np.unique(rsu_labels))

In [ ]:
# SEMLIDAR_TOP coordinates for BEV labels


def read_lidar_points_with_expected_count(zip_path, inner_path, expected_num_points=None):
    with zipfile.ZipFile(zip_path, "r") as z:
        raw = z.read(inner_path)

    arr = np.frombuffer(raw, dtype=np.float32).copy()

    if expected_num_points is not None:
        if arr.shape[0] % expected_num_points == 0:
            dim = arr.shape[0] // expected_num_points

            if dim in [3, 4, 5, 6]:
                points = arr.reshape(expected_num_points, dim)
                return points

    for dim in [5, 4, 6, 3]:
        if arr.shape[0] % dim == 0:
            points = arr.reshape(-1, dim)
            return points

    raise ValueError(f"Cannot infer point dimension. Raw float length = {arr.shape[0]}")


ego_agent_id, rsu_agent_id, scene_id, frame_id = matched_train_samples[0]

print("Selected train sample:")
print("ego_agent_id:", ego_agent_id)
print("rsu_agent_id:", rsu_agent_id)
print("scene_id:", scene_id)
print("frame_id:", frame_id)

# Read lidarseg labels

ego_label_item = lidarseg_index[(ego_agent_id, scene_id, frame_id)]
rsu_label_item = lidarseg_index[(rsu_agent_id, scene_id, frame_id)]

ego_labels = read_lidarseg_labels_from_zip(
    ego_label_item["zip_path"],
    ego_label_item["inner_path"]
)

rsu_labels = read_lidarseg_labels_from_zip(
    rsu_label_item["zip_path"],
    rsu_label_item["inner_path"]
)


# Read raw LIDAR_TOP for model input

ego_raw_lidar_item = lidar_index[(ego_agent_id, scene_id, frame_id, "LIDAR_TOP")]
rsu_raw_lidar_item = lidar_index[(rsu_agent_id, scene_id, frame_id, "LIDAR_TOP")]

ego_raw_points = read_lidar_points_from_zip(
    ego_raw_lidar_item["zip_path"],
    ego_raw_lidar_item["inner_path"]
)

rsu_raw_points = read_lidar_points_from_zip(
    rsu_raw_lidar_item["zip_path"],
    rsu_raw_lidar_item["inner_path"]
)

# Read SEMLIDAR_TOP for label coordinates

ego_sem_lidar_item = lidar_index[(ego_agent_id, scene_id, frame_id, "SEMLIDAR_TOP")]
rsu_sem_lidar_item = lidar_index[(rsu_agent_id, scene_id, frame_id, "SEMLIDAR_TOP")]

ego_label_points = read_lidar_points_with_expected_count(
    ego_sem_lidar_item["zip_path"],
    ego_sem_lidar_item["inner_path"],
    expected_num_points=len(ego_labels)
)

rsu_label_points = read_lidar_points_with_expected_count(
    rsu_sem_lidar_item["zip_path"],
    rsu_sem_lidar_item["inner_path"],
    expected_num_points=len(rsu_labels)
)


print("\nRaw LiDAR input points:")
print("ego_raw_points:", ego_raw_points.shape)
print("rsu_raw_points:", rsu_raw_points.shape)

print("\nSemantic LiDAR points for labels:")
print("ego_label_points:", ego_label_points.shape)
print("ego_labels:", ego_labels.shape)
print("Same count:", ego_label_points.shape[0] == ego_labels.shape[0])

print("\nrsu_label_points:", rsu_label_points.shape)
print("rsu_labels:", rsu_labels.shape)
print("Same count:", rsu_label_points.shape[0] == rsu_labels.shape[0])

print("\nEgo label unique:", np.unique(ego_labels)[:50])
print("RSU label unique:", np.unique(rsu_labels)[:50])

In [ ]:
# Convert point-level semantic labels to BEV targets

IGNORE_INDEX = 255


def lidar_points_labels_to_bev_label(
    points,
    labels,
    x_range=(-50.0, 50.0),
    y_range=(-50.0, 50.0),
    z_range=(-3.0, 5.0),
    bev_size=(256, 256),
    ignore_index=IGNORE_INDEX
):
    """
    Convert point-level semantic labels into a sparse BEV
    semantic target map.

    Class 0 ("Unlabeled") is retained as a valid semantic class.
    BEV cells without a valid projected semantic point are
    assigned the ignore index 255.

    Output:
        bev_label: torch.LongTensor [H, W]
    """

    H, W = bev_size

    if points.shape[0] != labels.shape[0]:
        raise ValueError(
            f"Point-label mismatch: "
            f"points={points.shape[0]}, "
            f"labels={labels.shape[0]}"
        )

    x = points[:, 0]
    y = points[:, 1]
    z = points[:, 2]

    # Keep points inside the BEV region
    valid = (
        (x >= x_range[0]) & (x <= x_range[1]) &
        (y >= y_range[0]) & (y <= y_range[1]) &
        (z >= z_range[0]) & (z <= z_range[1])
    )

    points_valid = points[valid]
    labels_valid = labels[valid].astype(np.int64)

    bev_label = np.full(
        (H, W),
        ignore_index,
        dtype=np.int64
    )

    if points_valid.shape[0] == 0:
        return torch.from_numpy(bev_label)

    x = points_valid[:, 0]
    y = points_valid[:, 1]
    z = points_valid[:, 2]

    x_idx = (
        (x - x_range[0]) /
        (x_range[1] - x_range[0]) *
        (W - 1)
    ).astype(np.int32)

    y_idx = (
        (y - y_range[0]) /
        (y_range[1] - y_range[0]) *
        (H - 1)
    ).astype(np.int32)

    x_idx = np.clip(x_idx, 0, W - 1)
    y_idx = np.clip(y_idx, 0, H - 1)

    # Convert to BEV image coordinates
    y_idx = H - 1 - y_idx

    # Sort by height so that the highest point is assigned last
    order = np.argsort(z)

    x_idx = x_idx[order]
    y_idx = y_idx[order]
    labels_valid = labels_valid[order]

    bev_label[y_idx, x_idx] = labels_valid

    return torch.from_numpy(bev_label)


# Build ego and RSU BEV semantic targets

ego_bev_label = lidar_points_labels_to_bev_label(
    points=ego_label_points,
    labels=ego_labels,
    bev_size=(256, 256),
    ignore_index=IGNORE_INDEX
)

rsu_bev_label = lidar_points_labels_to_bev_label(
    points=rsu_label_points,
    labels=rsu_labels,
    bev_size=(256, 256),
    ignore_index=IGNORE_INDEX
)


print("Ego BEV label:", ego_bev_label.shape)
print("RSU BEV label:", rsu_bev_label.shape)

print(
    "Ego valid labeled cells:",
    (ego_bev_label != IGNORE_INDEX).sum().item()
)

print(
    "RSU valid labeled cells:",
    (rsu_bev_label != IGNORE_INDEX).sum().item()
)

assert ego_bev_label.shape == (256, 256)
assert rsu_bev_label.shape == (256, 256)

In [ ]:
# Validate BEV labels and define segmentation classes

IGNORE_INDEX = 255
NUM_CLASSES = 23

valid_labels = ego_bev_label[
    ego_bev_label != IGNORE_INDEX
]

print(
    "Valid label IDs:",
    torch.unique(valid_labels)
)

print(
    "Valid label range:",
    valid_labels.min().item(),
    "to",
    valid_labels.max().item()
)

# The segmentation decoder predicts the full 23-class label space.
num_classes = NUM_CLASSES

# Verify that all valid labels fall within the decoder class range
assert valid_labels.min().item() >= 0
assert valid_labels.max().item() < NUM_CLASSES

# Prepare the BEV segmentation target
target_bev_label = (
    ego_bev_label
    .unsqueeze(0)
    .long()
    .to(device)
)

print(
    "Target BEV label shape:",
    target_bev_label.shape
)

print(
    "Valid BEV cells:",
    (target_bev_label != IGNORE_INDEX)
    .sum()
    .item()
)

print(
    "Ignored BEV cells:",
    (target_bev_label == IGNORE_INDEX)
    .sum()
    .item()
)

assert target_bev_label.shape == (
    1, 256, 256
)

In [ ]:
# BEV semantic segmentation decoder

class BEVSegmentationDecoder(nn.Module):
    """
    Decode the selected cooperative BEV feature into
    per-pixel semantic logits.

    Input:
        x: [B, 64, 32, 32]

    Output:
        logits: [B, 23, 256, 256]
    """

    def __init__(
        self,
        in_channels=64,
        num_classes=23
    ):
        super().__init__()

        self.decoder = nn.Sequential(

            ConvBNReLU(
                in_channels,
                128,
                kernel_size=3
            ),

            nn.Upsample(
                scale_factor=2,
                mode="bilinear",
                align_corners=False
            ),  # 32 -> 64

            ConvBNReLU(
                128,
                96,
                kernel_size=3
            ),

            nn.Upsample(
                scale_factor=2,
                mode="bilinear",
                align_corners=False
            ),  # 64 -> 128

            ConvBNReLU(
                96,
                64,
                kernel_size=3
            ),

            nn.Upsample(
                scale_factor=2,
                mode="bilinear",
                align_corners=False
            ),  # 128 -> 256

            ConvBNReLU(
                64,
                32,
                kernel_size=3
            ),

            nn.Conv2d(
                32,
                num_classes,
                kernel_size=1
            )
        )

    def forward(self, x):
        return self.decoder(x)


seg_decoder = BEVSegmentationDecoder(
    in_channels=64,
    num_classes=NUM_CLASSES
).to(device)

print("BEV segmentation decoder initialized.")

In [ ]:
SEED = 42

TRAIN_TARGET = 8000
VAL_TARGET = 1000
TEST_TARGET = 1000


def split_samples_by_scene_80_10_10(
    samples,
    train_target=8000,
    val_target=1000,
    test_target=1000,
    seed=42
):

    scene_to_samples = defaultdict(list)

    for sample in samples:
        ego_agent_id, rsu_agent_id, scene_id, frame_id = sample
        scene_to_samples[scene_id].append(sample)

    scene_ids = list(scene_to_samples.keys())

    rng = random.Random(seed)
    rng.shuffle(scene_ids)

    num_scenes = len(scene_ids)

    n_train_scenes = int(0.8 * num_scenes)
    n_val_scenes = int(0.1 * num_scenes)

    train_scene_ids = scene_ids[:n_train_scenes]

    val_scene_ids = scene_ids[
        n_train_scenes:
        n_train_scenes + n_val_scenes
    ]

    test_scene_ids = scene_ids[
        n_train_scenes + n_val_scenes:
    ]

    train_scenes = set(train_scene_ids)
    val_scenes = set(val_scene_ids)
    test_scenes = set(test_scene_ids)

    train_pool = []
    val_pool = []
    test_pool = []

    for scene_id in train_scenes:
        train_pool.extend(scene_to_samples[scene_id])

    for scene_id in val_scenes:
        val_pool.extend(scene_to_samples[scene_id])

    for scene_id in test_scenes:
        test_pool.extend(scene_to_samples[scene_id])

    rng.shuffle(train_pool)
    rng.shuffle(val_pool)
    rng.shuffle(test_pool)

    assert len(train_pool) >= train_target
    assert len(val_pool) >= val_target
    assert len(test_pool) >= test_target

    train_samples = train_pool[:train_target]
    val_samples = val_pool[:val_target]
    test_samples = test_pool[:test_target]

    all_used_samples = (
        train_samples +
        val_samples +
        test_samples
    )

    # Scene-level leakage checks
    assert train_scenes.isdisjoint(val_scenes)
    assert train_scenes.isdisjoint(test_scenes)
    assert val_scenes.isdisjoint(test_scenes)

    return (
        all_used_samples,
        train_samples,
        val_samples,
        test_samples,
        train_scenes,
        val_scenes,
        test_scenes
    )


(
    all_stage1_samples,
    train_samples,
    val_samples,
    test_samples,
    train_scenes,
    val_scenes,
    test_scenes
) = split_samples_by_scene_80_10_10(
    samples=matched_train_samples,
    train_target=TRAIN_TARGET,
    val_target=VAL_TARGET,
    test_target=TEST_TARGET,
    seed=SEED
)


print("Dataset split completed.")
print("Training samples:", len(train_samples))
print("Validation samples:", len(val_samples))
print("Testing samples:", len(test_samples))

print("Training scenes:", len(train_scenes))
print("Validation scenes:", len(val_scenes))
print("Testing scenes:", len(test_scenes))

print(
    "Scene leakage:",
    train_scenes.isdisjoint(val_scenes)
    and train_scenes.isdisjoint(test_scenes)
    and val_scenes.isdisjoint(test_scenes)
)

In [ ]:
# Multimodal ego-RSU dataset

class Stage1V2XDataset(Dataset):

    def __init__(
        self,
        samples,
        image_size=(256, 256),
        bev_size=(256, 256),
        ignore_index=255
    ):
        self.samples = samples
        self.image_size = image_size
        self.bev_size = bev_size
        self.ignore_index = ignore_index

    def __len__(self):
        return len(self.samples)

    def _load_vehicle_cameras(
        self,
        agent_id,
        scene_id,
        frame_id
    ):
        tensors = []

        for view in vehicle_views:
            key = (
                agent_id,
                scene_id,
                frame_id,
                view
            )

            item = camera_index[key]

            img_tensor = load_rgb_image_from_zip(
                zip_path=item["zip_path"],
                inner_path=item["inner_path"],
                image_size=self.image_size
            )

            tensors.append(img_tensor)

        cameras = torch.stack(
            tensors,
            dim=0
        )

        mask = torch.tensor(
            [1, 1, 1, 1, 1, 1],
            dtype=torch.float32
        )

        return cameras, mask

    def _load_rsu_cameras(
        self,
        agent_id,
        scene_id,
        frame_id
    ):
        tensors = []

        for view in rsu_views:
            key = (
                agent_id,
                scene_id,
                frame_id,
                view
            )

            item = camera_index[key]

            img_tensor = load_rgb_image_from_zip(
                zip_path=item["zip_path"],
                inner_path=item["inner_path"],
                image_size=self.image_size
            )

            tensors.append(img_tensor)

        rsu_4 = torch.stack(
            tensors,
            dim=0
        )

        rsu_6 = torch.zeros(
            (
                6,
                3,
                self.image_size[0],
                self.image_size[1]
            ),
            dtype=rsu_4.dtype
        )

        rsu_6[:4] = rsu_4

        mask = torch.tensor(
            [1, 1, 1, 1, 0, 0],
            dtype=torch.float32
        )

        return rsu_6, mask

    def _load_lidar_bev(
        self,
        agent_id,
        scene_id,
        frame_id
    ):
        item = lidar_index[
            (
                agent_id,
                scene_id,
                frame_id,
                "LIDAR_TOP"
            )
        ]

        points = read_lidar_points_from_zip(
            item["zip_path"],
            item["inner_path"]
        )

        return lidar_points_to_bev(
            points,
            bev_size=self.bev_size
        )

    def _load_ego_bev_label(
        self,
        agent_id,
        scene_id,
        frame_id
    ):
        label_item = lidarseg_index[
            (
                agent_id,
                scene_id,
                frame_id
            )
        ]

        labels = read_lidarseg_labels_from_zip(
            label_item["zip_path"],
            label_item["inner_path"]
        )

        sem_lidar_item = lidar_index[
            (
                agent_id,
                scene_id,
                frame_id,
                "SEMLIDAR_TOP"
            )
        ]

        label_points = read_lidar_points_with_expected_count(
            sem_lidar_item["zip_path"],
            sem_lidar_item["inner_path"],
            expected_num_points=len(labels)
        )

        return lidar_points_labels_to_bev_label(
            points=label_points,
            labels=labels,
            bev_size=self.bev_size,
            ignore_index=self.ignore_index
        )

    def __getitem__(self, idx):
        (
            ego_agent_id,
            rsu_agent_id,
            scene_id,
            frame_id
        ) = self.samples[idx]

        ego_cameras, ego_cam_mask = (
            self._load_vehicle_cameras(
                ego_agent_id,
                scene_id,
                frame_id
            )
        )

        rsu_cameras, rsu_cam_mask = (
            self._load_rsu_cameras(
                rsu_agent_id,
                scene_id,
                frame_id
            )
        )

        ego_lidar_bev = self._load_lidar_bev(
            ego_agent_id,
            scene_id,
            frame_id
        )

        rsu_lidar_bev = self._load_lidar_bev(
            rsu_agent_id,
            scene_id,
            frame_id
        )

        target = self._load_ego_bev_label(
            ego_agent_id,
            scene_id,
            frame_id
        )

        return {
            "ego_cameras": ego_cameras,
            "rsu_cameras": rsu_cameras,
            "ego_cam_mask": ego_cam_mask,
            "rsu_cam_mask": rsu_cam_mask,
            "ego_lidar_bev": ego_lidar_bev,
            "rsu_lidar_bev": rsu_lidar_bev,
            "target": target,
            "sample_info": torch.tensor(
                [
                    ego_agent_id,
                    rsu_agent_id,
                    scene_id,
                    frame_id
                ],
                dtype=torch.long
            )
        }

In [ ]:
camera_index_multi = defaultdict(list)

# Primary camera entries
for key, item in camera_index.items():
    camera_index_multi[key].append(item)

# Additional duplicate entries, if available
if "duplicate_keys" in globals():
    for key, items in duplicate_keys.items():
        camera_index_multi[key].extend(items)

num_multi_keys = sum(
    1 for items in camera_index_multi.values()
    if len(items) > 1
)

print("Camera keys indexed:", len(camera_index_multi))
print("Camera keys with multiple candidate files:", num_multi_keys)

In [ ]:
#  Camera image loader

bad_camera_entries = defaultdict(int)


def load_rgb_image_from_camera_key(
    key,
    image_size=(256, 256)
):

    candidates = camera_index_multi.get(key, [])

    if not candidates:
        raise KeyError(
            f"No camera candidates found for key: {key}"
        )

    last_error = None

    for item in candidates:
        try:
            with zipfile.ZipFile(
                item["zip_path"],
                "r"
            ) as z:
                img_bytes = z.read(
                    item["inner_path"]
                )

            img = Image.open(
                BytesIO(img_bytes)
            ).convert("RGB")

            img = img.resize(
                image_size,
                Image.BILINEAR
            )

            arr = (
                np.asarray(img)
                .astype(np.float32)
                / 255.0
            )

            tensor = torch.from_numpy(
                arr
            ).permute(2, 0, 1)

            return tensor

        except Exception as exc:
            last_error = exc

            bad_camera_entries[
                (
                    item["zip_name"],
                    item["inner_path"]
                )
            ] += 1

    raise RuntimeError(
        f"All camera candidates failed for key={key}. "
        f"Last error: {last_error!r}"
    )

In [ ]:
class Stage1V2XDatasetSafe(Stage1V2XDataset):
    """
    Stage-1 dataset using duplicate-aware robust camera loading.
    """

    def _load_vehicle_cameras(
        self,
        agent_id,
        scene_id,
        frame_id
    ):
        tensors = []

        for view in vehicle_views:
            key = (
                agent_id,
                scene_id,
                frame_id,
                view
            )

            img_tensor = load_rgb_image_from_camera_key(
                key=key,
                image_size=self.image_size
            )

            tensors.append(img_tensor)

        cameras = torch.stack(tensors, dim=0)

        mask = torch.tensor(
            [1, 1, 1, 1, 1, 1],
            dtype=torch.float32
        )

        return cameras, mask

    def _load_rsu_cameras(
        self,
        agent_id,
        scene_id,
        frame_id
    ):
        tensors = []

        for view in rsu_views:
            key = (
                agent_id,
                scene_id,
                frame_id,
                view
            )

            img_tensor = load_rgb_image_from_camera_key(
                key=key,
                image_size=self.image_size
            )

            tensors.append(img_tensor)

        rsu_4 = torch.stack(tensors, dim=0)

        rsu_6 = torch.zeros(
            (
                6,
                3,
                self.image_size[0],
                self.image_size[1]
            ),
            dtype=rsu_4.dtype
        )

        rsu_6[:4] = rsu_4

        mask = torch.tensor(
            [1, 1, 1, 1, 0, 0],
            dtype=torch.float32
        )

        return rsu_6, mask

In [ ]:
train_dataset = Stage1V2XDatasetSafe(
    samples=train_samples,
    image_size=(256, 256),
    bev_size=(256, 256),
    ignore_index=IGNORE_INDEX
)

val_dataset = Stage1V2XDatasetSafe(
    samples=val_samples,
    image_size=(256, 256),
    bev_size=(256, 256),
    ignore_index=IGNORE_INDEX
)

train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

In [ ]:
# Stage-1 V2I fusion and segmentation model

class Stage1V2IFusionSegmentationModel(nn.Module):
    def __init__(self, feature_channels=64, num_classes=23):
        super().__init__()

        self.camera_branch = CameraBranch(
            feature_channels=feature_channels
        )

        self.lidar_branch = LiDARBranchEncoder(
            in_channels=4,
            feature_channels=feature_channels
        )

        self.local_fusion = LocalCameraLiDARFusion(
            feature_channels=feature_channels
        )

        self.v2i_fusion = InitialV2IFusion(
            feature_channels=feature_channels
        )

        self.seg_decoder = BEVSegmentationDecoder(
            in_channels=feature_channels,
            num_classes=num_classes
        )

    def forward(
        self,
        ego_cameras,
        rsu_cameras,
        ego_cam_mask,
        rsu_cam_mask,
        ego_lidar_bev,
        rsu_lidar_bev
    ):
        # Camera encoding
        ego_cam_out = self.camera_branch(
            cameras=ego_cameras,
            cam_mask=ego_cam_mask
        )

        rsu_cam_out = self.camera_branch(
            cameras=rsu_cameras,
            cam_mask=rsu_cam_mask
        )

        ego_cam_feat = ego_cam_out["fused_camera_feature"]
        rsu_cam_feat = rsu_cam_out["fused_camera_feature"]

        # LiDAR encoding
        ego_lidar_feat = self.lidar_branch(
            ego_lidar_bev
        )

        rsu_lidar_feat = self.lidar_branch(
            rsu_lidar_bev
        )

        # Local camera-LiDAR fusion
        ego_local_out = self.local_fusion(
            camera_feature=ego_cam_feat,
            lidar_feature=ego_lidar_feat
        )

        rsu_local_out = self.local_fusion(
            camera_feature=rsu_cam_feat,
            lidar_feature=rsu_lidar_feat
        )

        ego_local_feat = ego_local_out["fused_feature"]
        rsu_local_feat = rsu_local_out["fused_feature"]

        # Initial cooperative fusion before geometric alignment
        v2i_out = self.v2i_fusion(
            ego_feature=ego_local_feat,
            rsu_feature=rsu_local_feat
        )

        cooperative_feat = v2i_out["cooperative_feature"]

        # BEV semantic segmentation
        segmentation_logits = self.seg_decoder(
            cooperative_feat
        )

        return {
            "segmentation_logits": segmentation_logits,
            "ego_cam_out": ego_cam_out,
            "rsu_cam_out": rsu_cam_out,
            "ego_local_out": ego_local_out,
            "rsu_local_out": rsu_local_out,
            "v2i_out": v2i_out
        }


stage1_model = Stage1V2IFusionSegmentationModel(
    feature_channels=64,
    num_classes=NUM_CLASSES
).to(device)

print("Stage-1 model initialized.")

In [ ]:
# Stage-1 training and validation DataLoaders


STAGE1_BATCH_SIZE = 1
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

train_dataset = Stage1V2XDatasetSafe(
    samples=train_samples,
    image_size=(256, 256),
    bev_size=(256, 256),
    ignore_index=IGNORE_INDEX
)

val_dataset = Stage1V2XDatasetSafe(
    samples=val_samples,
    image_size=(256, 256),
    bev_size=(256, 256),
    ignore_index=IGNORE_INDEX
)

loader_generator = torch.Generator()
loader_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=STAGE1_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    generator=loader_generator
)

val_loader = DataLoader(
    val_dataset,
    batch_size=STAGE1_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Batch size:", STAGE1_BATCH_SIZE)

In [ ]:
# Compute class weights from training labels

CLASS_WEIGHT_MAX_SAMPLES = None


def compute_class_frequencies_from_samples(
    samples,
    num_classes,
    ignore_index=IGNORE_INDEX,
    max_samples=None
):
    class_counts = torch.zeros(
        num_classes,
        dtype=torch.float64
    )

    use_samples = (
        samples
        if max_samples is None
        else samples[:max_samples]
    )

    for i, sample in enumerate(use_samples):
        (
            ego_agent_id,
            rsu_agent_id,
            scene_id,
            frame_id
        ) = sample

        label_item = lidarseg_index[
            (ego_agent_id, scene_id, frame_id)
        ]

        labels = read_lidarseg_labels_from_zip(
            label_item["zip_path"],
            label_item["inner_path"]
        )

        sem_lidar_item = lidar_index[
            (
                ego_agent_id,
                scene_id,
                frame_id,
                "SEMLIDAR_TOP"
            )
        ]

        label_points = read_lidar_points_with_expected_count(
            sem_lidar_item["zip_path"],
            sem_lidar_item["inner_path"],
            expected_num_points=len(labels)
        )

        bev_label = lidar_points_labels_to_bev_label(
            points=label_points,
            labels=labels,
            bev_size=(256, 256),
            ignore_index=ignore_index
        )

        valid = bev_label != ignore_index
        values = bev_label[valid]

        if values.numel() > 0:
            counts = torch.bincount(
                values.long(),
                minlength=num_classes
            ).double()

            class_counts += counts

        if (i + 1) % 500 == 0:
            print(
                f"Processed {i + 1}/{len(use_samples)} "
                "training samples"
            )

    return class_counts


class_counts = compute_class_frequencies_from_samples(
    samples=train_samples,
    num_classes=NUM_CLASSES,
    ignore_index=IGNORE_INDEX,
    max_samples=CLASS_WEIGHT_MAX_SAMPLES
)

present = class_counts > 0

frequencies = (
    class_counts
    / class_counts.sum().clamp(min=1)
)

class_weights = torch.zeros(
    NUM_CLASSES,
    dtype=torch.float32
)

median_frequency = frequencies[present].median()

class_weights[present] = torch.sqrt(
    median_frequency / frequencies[present]
).float()

class_weights = torch.clamp(
    class_weights,
    min=0.25,
    max=5.0
)

print("Computed class weights for training.")
print("Classes present:", int(present.sum().item()))

In [ ]:
def move_batch_to_device(batch, device):
    return {
        key: value.to(device) if torch.is_tensor(value) else value
        for key, value in batch.items()
    }

In [ ]:
# Stage-1 training/validation epoch

def run_one_epoch_weighted(
    model,
    loader,
    optimizer=None,
    train=True,
    ignore_index=IGNORE_INDEX,
    class_weights=None
):
    model.train() if train else model.eval()

    weights = (
        class_weights.to(device)
        if class_weights is not None
        else None
    )

    criterion = nn.CrossEntropyLoss(
        weight=weights,
        ignore_index=ignore_index
    )

    total_loss = 0.0
    total_valid_pixels = 0
    total_correct = 0
    used_batches = 0

    for batch_idx, batch in enumerate(loader):

        batch = move_batch_to_device(
            batch,
            device
        )

        target = batch["target"]

        valid_mask = target != ignore_index
        valid_pixels = valid_mask.sum().item()

        if valid_pixels == 0:
            continue

        if train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(train):

            out = model(
                ego_cameras=batch["ego_cameras"],
                rsu_cameras=batch["rsu_cameras"],
                ego_cam_mask=batch["ego_cam_mask"],
                rsu_cam_mask=batch["rsu_cam_mask"],
                ego_lidar_bev=batch["ego_lidar_bev"],
                rsu_lidar_bev=batch["rsu_lidar_bev"]
            )

            logits = out["segmentation_logits"]

            loss = criterion(
                logits,
                target
            )

            if train:
                loss.backward()

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=5.0
                )

                optimizer.step()

        with torch.no_grad():

            pred = torch.argmax(
                logits,
                dim=1
            )

            correct = (
                (pred == target)
                & valid_mask
            ).sum().item()

        total_loss += loss.item()
        total_valid_pixels += valid_pixels
        total_correct += correct
        used_batches += 1

        if train and (batch_idx + 1) % 100 == 0:
            print(
                f"Batch {batch_idx + 1}/{len(loader)} | "
                f"Loss: {loss.item():.4f}"
            )

    avg_loss = (
        total_loss
        / max(used_batches, 1)
    )

    pixel_acc = (
        total_correct
        / max(total_valid_pixels, 1)
    )

    return avg_loss, pixel_acc

In [ ]:
# Train Stage-1 perception model


stage1_model_v2 = Stage1V2IFusionSegmentationModel(
    feature_channels=64,
    num_classes=NUM_CLASSES
).to(device)

optimizer = torch.optim.AdamW(
    stage1_model_v2.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

MAX_EPOCHS = 50
EARLY_STOP_PATIENCE = 4

best_val_loss = float("inf")
epochs_without_improvement = 0

stage1_weighted_history = []

history_path = (
    OUTPUT_DIR
    / "stage1_weighted_training_history.json"
)

best_path = (
    OUTPUT_DIR
    / "stage1_weighted_best_checkpoint.pth"
)

last_path = (
    OUTPUT_DIR
    / "stage1_weighted_last_checkpoint.pth"
)


for epoch in range(1, MAX_EPOCHS + 1):

    print(
        f"\nStage-1 Epoch {epoch}/{MAX_EPOCHS}"
    )

    train_loss, train_acc = run_one_epoch_weighted(
        model=stage1_model_v2,
        loader=train_loader,
        optimizer=optimizer,
        train=True,
        ignore_index=IGNORE_INDEX,
        class_weights=class_weights
    )

    val_loss, val_acc = run_one_epoch_weighted(
        model=stage1_model_v2,
        loader=val_loader,
        optimizer=None,
        train=False,
        ignore_index=IGNORE_INDEX,
        class_weights=class_weights
    )

    scheduler.step(val_loss)

    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"Train Loss: {train_loss:.4f} | "
        f"Train Pixel Acc: {train_acc:.4f}"
    )

    print(
        f"Val Loss: {val_loss:.4f} | "
        f"Val Pixel Acc: {val_acc:.4f} | "
        f"LR: {current_lr:.8f}"
    )

    epoch_record = {
        "epoch": epoch,
        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "train_acc": float(train_acc),
        "val_acc": float(val_acc),
        "lr": float(current_lr)
    }

    stage1_weighted_history.append(
        epoch_record
    )

    with open(history_path, "w") as f:
        json.dump(
            stage1_weighted_history,
            f,
            indent=4
        )

    checkpoint = {
        "epoch": epoch,
        "model_state_dict":
            stage1_model_v2.state_dict(),
        "optimizer_state_dict":
            optimizer.state_dict(),
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_acc": train_acc,
        "val_acc": val_acc,
        "num_classes": NUM_CLASSES,
        "class_weights": class_weights,
        "history": stage1_weighted_history
    }

    # Save latest checkpoint
    torch.save(
        checkpoint,
        last_path
    )

    # Save model with minimum validation loss
    if val_loss < best_val_loss:

        best_val_loss = val_loss
        epochs_without_improvement = 0

        torch.save(
            checkpoint,
            best_path
        )

        print(
            "Saved new best Stage-1 checkpoint."
        )

    else:

        epochs_without_improvement += 1

        print(
            "No validation improvement: "
            f"{epochs_without_improvement}/"
            f"{EARLY_STOP_PATIENCE}"
        )

    if (
        epochs_without_improvement
        >= EARLY_STOP_PATIENCE
    ):
        print("Early stopping triggered.")
        break


print("Stage-1 training completed.")
print(
    "Best validation loss:",
    best_val_loss
)

In [ ]:
#  Load best Stage-1 checkpoint

best_weighted_path = (
    OUTPUT_DIR
    / "stage1_weighted_best_checkpoint.pth"
)

checkpoint = torch.load(
    best_weighted_path,
    map_location=device
)

stage1_model_v2.load_state_dict(
    checkpoint["model_state_dict"]
)

stage1_model_v2.to(device)
stage1_model_v2.eval()

print(
    "Loaded best Stage-1 checkpoint."
)

print(
    "Best epoch:",
    checkpoint["epoch"]
)

print(
    "Validation loss:",
    checkpoint["val_loss"]
)

print(
    "Validation pixel accuracy:",
    checkpoint["val_acc"]
)

In [ ]:
# Test DataLoader

test_dataset = Stage1V2XDatasetSafe(
    samples=test_samples,
    image_size=(256, 256),
    bev_size=(256, 256),
    ignore_index=IGNORE_INDEX
)

test_loader = DataLoader(
    test_dataset,
    batch_size=STAGE1_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

print("Test batches:", len(test_loader))

In [ ]:
# Stage-1 mIoU evaluation

def evaluate_stage1_miou(
    model,
    loader,
    num_classes,
    device,
    ignore_index=IGNORE_INDEX
):
    model.eval()

    confusion = torch.zeros(
        (num_classes, num_classes),
        dtype=torch.int64
    )

    total_correct = 0
    total_valid = 0

    with torch.inference_mode():

        for batch in loader:

            batch_device = move_batch_to_device(
                batch,
                device
            )

            out = model(
                ego_cameras=batch_device["ego_cameras"],
                ego_camera_mask=batch_device["ego_camera_mask"],
                ego_lidar_bev=batch_device["ego_lidar_bev"],
                rsu_cameras=batch_device["rsu_cameras"],
                rsu_camera_mask=batch_device["rsu_camera_mask"],
                rsu_lidar_bev=batch_device["rsu_lidar_bev"]
            )

            logits = out["segmentation_logits"]
            target = batch_device["target"]

            pred = torch.argmax(
                logits,
                dim=1
            )

            valid_mask = (
                target != ignore_index
            )

            valid_target = target[
                valid_mask
            ]

            valid_pred = pred[
                valid_mask
            ]

            total_correct += int(
                (
                    valid_target
                    == valid_pred
                ).sum().item()
            )

            total_valid += int(
                valid_target.numel()
            )

            valid_target = (
                valid_target
                .detach()
                .cpu()
                .long()
            )

            valid_pred = (
                valid_pred
                .detach()
                .cpu()
                .long()
            )

            indices = (
                num_classes
                * valid_target
                + valid_pred
            )

            confusion += torch.bincount(
                indices,
                minlength=(
                    num_classes
                    * num_classes
                )
            ).reshape(
                num_classes,
                num_classes
            )

    intersection = torch.diag(
        confusion
    ).float()

    gt_pixels_per_class = (
        confusion.sum(dim=1).float()
    )

    pred_pixels_per_class = (
        confusion.sum(dim=0).float()
    )

    union = (
        gt_pixels_per_class
        + pred_pixels_per_class
        - intersection
    )

    gt_present_classes = (
        gt_pixels_per_class > 0
    )

    iou_per_class = torch.zeros(
        num_classes,
        dtype=torch.float32
    )

    iou_per_class[
        gt_present_classes
    ] = (
        intersection[
            gt_present_classes
        ]
        /
        union[
            gt_present_classes
        ].clamp(min=1)
    )

    if gt_present_classes.any():

        miou = float(
            iou_per_class[
                gt_present_classes
            ].mean().item()
        )

    else:

        miou = 0.0

    pixel_accuracy = (
        total_correct
        / max(total_valid, 1)
    )

    result = {
        "pixel_accuracy":
            float(pixel_accuracy),

        "mIoU":
            float(miou),

        "total_valid_pixels":
            int(total_valid),

        "total_correct_pixels":
            int(total_correct),

        "valid_classes": [
            int(i)
            for i in range(num_classes)
            if bool(
                gt_present_classes[i]
            )
        ],

        "per_class_iou": {
            str(i):
                float(
                    iou_per_class[i].item()
                )
            for i in range(num_classes)
            if bool(
                gt_present_classes[i]
            )
        }
    }

    return result, confusion

In [ ]:
# Full Stage-1 validation evaluation

stage1_eval_results, stage1_confusion = evaluate_stage1_miou(
    model=stage1_model_v2,
    loader=val_loader,
    num_classes=NUM_CLASSES,
    device=device,
    ignore_index=IGNORE_INDEX
)

print("Stage-1 validation results:")
print(
    "Pixel accuracy:",
    stage1_eval_results["pixel_accuracy"]
)
print(
    "mIoU:",
    stage1_eval_results["mIoU"]
)

stage1_eval_path = (
    OUTPUT_DIR
    / "stage1_weighted_eval_results.json"
)

stage1_confusion_path = (
    OUTPUT_DIR
    / "stage1_weighted_confusion_matrix.pt"
)

with open(stage1_eval_path, "w") as f:
    json.dump(
        stage1_eval_results,
        f,
        indent=4
    )

torch.save(
    stage1_confusion,
    stage1_confusion_path
)

print("Saved Stage-1 validation results.")

In [ ]:
# Full Stage-1 test evaluation

stage1_test_results, stage1_test_confusion = evaluate_stage1_miou(
    model=stage1_model_v2,
    loader=test_loader,
    num_classes=NUM_CLASSES,
    device=device,
    ignore_index=IGNORE_INDEX
)

print("Stage-1 test results:")
print(
    "Pixel accuracy:",
    stage1_test_results["pixel_accuracy"]
)
print(
    "mIoU:",
    stage1_test_results["mIoU"]
)

stage1_test_path = (
    OUTPUT_DIR
    / "stage1_weighted_test_results.json"
)

stage1_test_confusion_path = (
    OUTPUT_DIR
    / "stage1_weighted_test_confusion_matrix.pt"
)

with open(stage1_test_path, "w") as f:
    json.dump(
        stage1_test_results,
        f,
        indent=4
    )

torch.save(
    stage1_test_confusion,
    stage1_test_confusion_path
)

print("Saved Stage-1 test results.")

In [ ]:
# Stage 2:Learnable RSU-to-ego feature alignment

class RSUToEgoFeatureAlignment(nn.Module):
    """
    Learnable affine alignment of RSU BEV features to the ego frame.

    Input:
        ego_feature: [B, C, H, W]
        rsu_feature: [B, C, H, W]

    Output:
        aligned_rsu_feature: [B, C, H, W]
        theta: [B, 2, 3]
        alignment_params: [B, 3]
            normalized x-translation,
            normalized y-translation,
            rotation angle
    """

    def __init__(
        self,
        feature_channels=64,
        hidden_dim=128,
        max_translation=0.20,
        max_rotation_deg=10.0,
        padding_mode="border"
    ):
        super().__init__()

        self.max_translation = max_translation
        self.max_rotation_rad = math.radians(
            max_rotation_deg
        )
        self.padding_mode = padding_mode

        self.spatial_encoder = nn.Sequential(
            nn.Conv2d(
                feature_channels * 3,
                hidden_dim,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(hidden_dim),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                hidden_dim,
                hidden_dim,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(hidden_dim),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                hidden_dim,
                hidden_dim,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(hidden_dim),
            nn.ReLU(inplace=True)
        )

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.regressor = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, 3)
        )

        # Initialize close to the identity transform
        nn.init.zeros_(
            self.regressor[-1].weight
        )
        nn.init.zeros_(
            self.regressor[-1].bias
        )

    def forward(
        self,
        ego_feature,
        rsu_feature
    ):
        B, C, H, W = ego_feature.shape

        diff = torch.abs(
            ego_feature - rsu_feature
        )

        x = torch.cat(
            [
                ego_feature,
                rsu_feature,
                diff
            ],
            dim=1
        )

        spatial_feat = self.spatial_encoder(
            x
        )

        pooled_feat = self.pool(
            spatial_feat
        ).view(B, -1)

        raw_params = torch.tanh(
            self.regressor(
                pooled_feat
            )
        )

        dx = (
            raw_params[:, 0]
            * self.max_translation
        )

        dy = (
            raw_params[:, 1]
            * self.max_translation
        )

        angle = (
            raw_params[:, 2]
            * self.max_rotation_rad
        )

        cos_a = torch.cos(angle)
        sin_a = torch.sin(angle)

        theta = torch.zeros(
            (B, 2, 3),
            dtype=ego_feature.dtype,
            device=ego_feature.device
        )

        theta[:, 0, 0] = cos_a
        theta[:, 0, 1] = -sin_a
        theta[:, 0, 2] = dx

        theta[:, 1, 0] = sin_a
        theta[:, 1, 1] = cos_a
        theta[:, 1, 2] = dy

        grid = F.affine_grid(
            theta,
            size=rsu_feature.size(),
            align_corners=False
        )

        aligned_rsu_feature = F.grid_sample(
            rsu_feature,
            grid,
            mode="bilinear",
            padding_mode=self.padding_mode,
            align_corners=False
        )

        alignment_params = torch.stack(
            [
                dx,
                dy,
                angle
            ],
            dim=1
        )

        return {
            "aligned_rsu_feature":
                aligned_rsu_feature,
            "theta":
                theta,
            "alignment_params":
                alignment_params
        }


print(
    "RSU-to-ego feature alignment module ready."
)

In [ ]:
# Stage-2 aligned V2I segmentation model

class Stage2AlignedV2IFusionSegmentationModel(nn.Module):
    """
    Stage-2 model with learnable RSU-to-ego feature alignment.

    Pipeline:
        Ego camera + LiDAR -> ego local feature
        RSU camera + LiDAR -> RSU local feature
        RSU local feature -> ego-frame alignment
        Ego + aligned RSU -> cooperative V2I fusion
        Cooperative feature -> BEV segmentation
    """

    def __init__(
        self,
        feature_channels=64,
        num_classes=23
    ):
        super().__init__()

        self.camera_branch = CameraBranch(
            feature_channels=feature_channels
        )

        self.lidar_branch = LiDARBranchEncoder(
            in_channels=4,
            feature_channels=feature_channels
        )

        self.local_fusion = LocalCameraLiDARFusion(
            feature_channels=feature_channels
        )

        self.rsu_alignment = RSUToEgoFeatureAlignment(
            feature_channels=feature_channels
        )

        self.v2i_fusion = InitialV2IFusion(
            feature_channels=feature_channels
        )

        self.seg_decoder = BEVSegmentationDecoder(
            in_channels=feature_channels,
            num_classes=num_classes
        )

    def forward(
        self,
        ego_cameras,
        rsu_cameras,
        ego_cam_mask,
        rsu_cam_mask,
        ego_lidar_bev,
        rsu_lidar_bev
    ):

        ego_cam_out = self.camera_branch(
            cameras=ego_cameras,
            cam_mask=ego_cam_mask
        )

        rsu_cam_out = self.camera_branch(
            cameras=rsu_cameras,
            cam_mask=rsu_cam_mask
        )

        ego_cam_feat = ego_cam_out["fused_camera_feature"]
        rsu_cam_feat = rsu_cam_out["fused_camera_feature"]

        ego_lidar_feat = self.lidar_branch(
            ego_lidar_bev
        )

        rsu_lidar_feat = self.lidar_branch(
            rsu_lidar_bev
        )

        ego_local_out = self.local_fusion(
            camera_feature=ego_cam_feat,
            lidar_feature=ego_lidar_feat
        )

        rsu_local_out = self.local_fusion(
            camera_feature=rsu_cam_feat,
            lidar_feature=rsu_lidar_feat
        )

        ego_local_feat = ego_local_out["fused_feature"]
        rsu_local_feat = rsu_local_out["fused_feature"]

        alignment_out = self.rsu_alignment(
            ego_feature=ego_local_feat,
            rsu_feature=rsu_local_feat
        )

        aligned_rsu_feat = (
            alignment_out["aligned_rsu_feature"]
        )

        v2i_out = self.v2i_fusion(
            ego_feature=ego_local_feat,
            rsu_feature=aligned_rsu_feat
        )

        segmentation_logits = self.seg_decoder(
            v2i_out["cooperative_feature"]
        )

        return {
            "segmentation_logits": segmentation_logits,
            "ego_cam_out": ego_cam_out,
            "rsu_cam_out": rsu_cam_out,
            "ego_local_out": ego_local_out,
            "rsu_local_out": rsu_local_out,
            "alignment_out": alignment_out,
            "v2i_out": v2i_out
        }


stage2_model = Stage2AlignedV2IFusionSegmentationModel(
    feature_channels=64,
    num_classes=NUM_CLASSES
).to(device)

print("Stage-2 aligned V2I model initialized.")


# Initialize Stage 2 from the best Stage-1 checkpoint

stage1_weighted_path = (
    OUTPUT_DIR
    / "stage1_weighted_best_checkpoint.pth"
)

assert stage1_weighted_path.exists(), (
    f"Stage-1 checkpoint not found: {stage1_weighted_path}"
)

stage1_checkpoint = torch.load(
    stage1_weighted_path,
    map_location="cpu"
)

stage1_state_dict = (
    stage1_checkpoint["model_state_dict"]
    if "model_state_dict" in stage1_checkpoint
    else stage1_checkpoint
)

missing_keys, unexpected_keys = (
    stage2_model.load_state_dict(
        stage1_state_dict,
        strict=False
    )
)

# Only the newly added alignment module should be missing
bad_missing = [
    key
    for key in missing_keys
    if not key.startswith("rsu_alignment.")
]

assert not bad_missing, (
    "Unexpected missing Stage-2 parameters:\n"
    + "\n".join(bad_missing)
)

assert not unexpected_keys, (
    "Unexpected checkpoint parameters:\n"
    + "\n".join(unexpected_keys)
)

stage2_model.to(device)

print("Stage-1 weights transferred successfully.")
print(
    "New alignment parameters initialized:",
    len(missing_keys)
)


# Stage-2 trainability configuration

FREEZE_STAGE2_BACKBONE = False

if FREEZE_STAGE2_BACKBONE:
    for parameter in stage2_model.camera_branch.parameters():
        parameter.requires_grad = False

    for parameter in stage2_model.lidar_branch.parameters():
        parameter.requires_grad = False


total_params = sum(
    p.numel()
    for p in stage2_model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in stage2_model.parameters()
    if p.requires_grad
)

print("Total Stage-2 parameters:", total_params)
print("Trainable Stage-2 parameters:", trainable_params)

In [ ]:
def run_one_epoch_stage2(
    model,
    loader,
    optimizer=None,
    train=True,
    ignore_index=IGNORE_INDEX,
    class_weights=None,
    alignment_reg_weight=1e-4
):
    """
    Run one Stage-2 training or validation epoch.

    Returns:
        avg_loss
        avg_seg_loss
        avg_align_reg
        pixel_acc
        align_abs_mean
    """

    model.train() if train else model.eval()

    weights = (
        class_weights.to(device)
        if class_weights is not None
        else None
    )

    criterion = nn.CrossEntropyLoss(
        weight=weights,
        ignore_index=ignore_index
    )

    total_loss = 0.0
    total_seg_loss = 0.0
    total_align_reg = 0.0
    total_valid_pixels = 0
    total_correct = 0
    used_batches = 0

    align_abs_sum = torch.zeros(
        3,
        dtype=torch.float64
    )

    align_count = 0

    with torch.set_grad_enabled(train):

        for batch_idx, batch in enumerate(loader):

            batch = move_batch_to_device(
                batch,
                device
            )

            target = batch["target"]

            valid_mask = target != ignore_index
            valid_pixels = valid_mask.sum().item()

            if valid_pixels == 0:
                continue

            if train:
                optimizer.zero_grad(
                    set_to_none=True
                )

            out = model(
                ego_cameras=batch["ego_cameras"],
                rsu_cameras=batch["rsu_cameras"],
                ego_cam_mask=batch["ego_cam_mask"],
                rsu_cam_mask=batch["rsu_cam_mask"],
                ego_lidar_bev=batch["ego_lidar_bev"],
                rsu_lidar_bev=batch["rsu_lidar_bev"]
            )

            logits = out["segmentation_logits"]

            alignment_params = (
                out["alignment_out"]
                ["alignment_params"]
            )

            seg_loss = criterion(
                logits,
                target
            )

            align_reg = torch.mean(
                alignment_params ** 2
            )

            loss = (
                seg_loss
                + alignment_reg_weight
                * align_reg
            )

            if train:
                loss.backward()

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=5.0
                )

                optimizer.step()

            with torch.no_grad():

                pred = torch.argmax(
                    logits,
                    dim=1
                )

                correct = (
                    (pred == target)
                    & valid_mask
                ).sum().item()

                align_abs_sum += (
                    alignment_params
                    .detach()
                    .abs()
                    .sum(dim=0)
                    .cpu()
                    .double()
                )

                align_count += (
                    alignment_params.shape[0]
                )

            total_loss += loss.item()
            total_seg_loss += seg_loss.item()
            total_align_reg += align_reg.item()

            total_valid_pixels += valid_pixels
            total_correct += correct
            used_batches += 1

            if train and (batch_idx + 1) % 100 == 0:
                print(
                    f"Batch {batch_idx + 1}/{len(loader)} | "
                    f"Loss: {loss.item():.4f} | "
                    f"Seg: {seg_loss.item():.4f}"
                )

    avg_loss = (
        total_loss
        / max(used_batches, 1)
    )

    avg_seg_loss = (
        total_seg_loss
        / max(used_batches, 1)
    )

    avg_align_reg = (
        total_align_reg
        / max(used_batches, 1)
    )

    pixel_acc = (
        total_correct
        / max(total_valid_pixels, 1)
    )

    align_abs_mean = (
        align_abs_sum
        / max(align_count, 1)
    ).tolist()

    return (
        avg_loss,
        avg_seg_loss,
        avg_align_reg,
        pixel_acc,
        align_abs_mean
    )

In [ ]:
# Train Stage-2 aligned V2I model

STAGE2_ALIGN_REG_WEIGHT = 1e-4
NUM_EPOCHS_STAGE2 = 40
EARLY_STOP_PATIENCE_STAGE2 = 3

stage2_history_path = (
    OUTPUT_DIR
    / "stage2_aligned_training_history.json"
)

stage2_last_path = (
    OUTPUT_DIR
    / "stage2_aligned_last_checkpoint.pth"
)

stage2_best_path = (
    OUTPUT_DIR
    / "stage2_aligned_best_checkpoint.pth"
)


# Differential learning rates:
# smaller rates for Stage-1 initialized modules,
# larger rate for the newly introduced alignment module.
optimizer_stage2 = torch.optim.AdamW(
    [
        {
            "params": stage2_model.camera_branch.parameters(),
            "lr": 2e-5
        },
        {
            "params": stage2_model.lidar_branch.parameters(),
            "lr": 2e-5
        },
        {
            "params": stage2_model.local_fusion.parameters(),
            "lr": 2e-5
        },
        {
            "params": stage2_model.rsu_alignment.parameters(),
            "lr": 1e-4
        },
        {
            "params": stage2_model.v2i_fusion.parameters(),
            "lr": 5e-5
        },
        {
            "params": stage2_model.seg_decoder.parameters(),
            "lr": 5e-5
        }
    ],
    weight_decay=1e-4
)

scheduler_stage2 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_stage2,
    mode="min",
    factor=0.5,
    patience=1
)

best_val_loss = float("inf")
epochs_without_improvement = 0
stage2_history = []


print("Starting Stage-2 training.")


for epoch in range(1, NUM_EPOCHS_STAGE2 + 1):

    print(
        f"\nStage-2 Epoch "
        f"{epoch}/{NUM_EPOCHS_STAGE2}"
    )

    (
        train_loss,
        train_seg_loss,
        train_align_reg,
        train_acc,
        train_align_abs
    ) = run_one_epoch_stage2(
        model=stage2_model,
        loader=train_loader,
        optimizer=optimizer_stage2,
        train=True,
        ignore_index=IGNORE_INDEX,
        class_weights=class_weights,
        alignment_reg_weight=STAGE2_ALIGN_REG_WEIGHT
    )

    (
        val_loss,
        val_seg_loss,
        val_align_reg,
        val_acc,
        val_align_abs
    ) = run_one_epoch_stage2(
        model=stage2_model,
        loader=val_loader,
        optimizer=None,
        train=False,
        ignore_index=IGNORE_INDEX,
        class_weights=class_weights,
        alignment_reg_weight=STAGE2_ALIGN_REG_WEIGHT
    )

    scheduler_stage2.step(
        val_loss
    )

    current_lrs = [
        float(group["lr"])
        for group in optimizer_stage2.param_groups
    ]

    print(
        f"Train Loss: {train_loss:.4f} | "
        f"Seg: {train_seg_loss:.4f} | "
        f"Acc: {train_acc:.4f}"
    )

    print(
        f"Val Loss: {val_loss:.4f} | "
        f"Seg: {val_seg_loss:.4f} | "
        f"Acc: {val_acc:.4f}"
    )

    epoch_record = {
        "epoch": int(epoch),

        "train_loss": float(train_loss),
        "train_seg_loss": float(train_seg_loss),
        "train_align_reg": float(train_align_reg),
        "train_acc": float(train_acc),
        "train_align_abs_mean": [
            float(x)
            for x in train_align_abs
        ],

        "val_loss": float(val_loss),
        "val_seg_loss": float(val_seg_loss),
        "val_align_reg": float(val_align_reg),
        "val_acc": float(val_acc),
        "val_align_abs_mean": [
            float(x)
            for x in val_align_abs
        ],

        "learning_rates": current_lrs,
        "alignment_reg_weight":
            float(STAGE2_ALIGN_REG_WEIGHT)
    }

    stage2_history.append(
        epoch_record
    )

    with open(
        stage2_history_path,
        "w"
    ) as f:
        json.dump(
            stage2_history,
            f,
            indent=4
        )

    checkpoint = {
        "epoch": int(epoch),

        "model_state_dict":
            stage2_model.state_dict(),

        "optimizer_state_dict":
            optimizer_stage2.state_dict(),

        "train_loss": float(train_loss),
        "train_seg_loss": float(train_seg_loss),
        "train_align_reg": float(train_align_reg),
        "train_acc": float(train_acc),

        "val_loss": float(val_loss),
        "val_seg_loss": float(val_seg_loss),
        "val_align_reg": float(val_align_reg),
        "val_acc": float(val_acc),

        "num_classes": int(NUM_CLASSES),

        "class_weights":
            class_weights.detach().cpu(),

        "history":
            stage2_history,

        "alignment_reg_weight":
            float(STAGE2_ALIGN_REG_WEIGHT)
    }

    # Save latest checkpoint
    torch.save(
        checkpoint,
        stage2_last_path
    )

    # Save checkpoint with minimum validation loss
    if val_loss < best_val_loss:

        best_val_loss = val_loss
        epochs_without_improvement = 0

        torch.save(
            checkpoint,
            stage2_best_path
        )

        print(
            "Saved new best Stage-2 checkpoint."
        )

    else:

        epochs_without_improvement += 1

        print(
            "No validation improvement: "
            f"{epochs_without_improvement}/"
            f"{EARLY_STOP_PATIENCE_STAGE2}"
        )

    if (
        epochs_without_improvement
        >= EARLY_STOP_PATIENCE_STAGE2
    ):
        print("Stage-2 early stopping triggered.")
        break


print("\nStage-2 training completed.")
print(
    "Best validation loss:",
    best_val_loss
)

In [ ]:
# Load best Stage-2 checkpoint

best_stage2_path = (
    OUTPUT_DIR
    / "stage2_aligned_best_checkpoint.pth"
)

assert best_stage2_path.exists(), (
    f"Best Stage-2 checkpoint not found: "
    f"{best_stage2_path}"
)

stage2_checkpoint = torch.load(
    best_stage2_path,
    map_location=device
)

stage2_model.load_state_dict(
    stage2_checkpoint["model_state_dict"]
)

stage2_model.to(device)
stage2_model.eval()

print("Loaded best Stage-2 checkpoint.")
print("Best epoch:", stage2_checkpoint["epoch"])
print("Validation loss:", stage2_checkpoint["val_loss"])
print("Validation pixel accuracy:", stage2_checkpoint["val_acc"])

In [ ]:
# Stage-2 mIoU evaluation function

def evaluate_stage2_miou(
    model,
    loader,
    num_classes,
    device,
    ignore_index=IGNORE_INDEX
):
    """
    Evaluate Stage-2 BEV segmentation and alignment statistics.

    mIoU is computed over ground-truth-present semantic classes.
    """

    confusion = torch.zeros(
        (num_classes, num_classes),
        dtype=torch.int64
    )

    total_valid_pixels = 0
    total_correct_pixels = 0

    align_sum = torch.zeros(
        3,
        dtype=torch.float64
    )

    align_sq_sum = torch.zeros(
        3,
        dtype=torch.float64
    )

    align_abs_sum = torch.zeros(
        3,
        dtype=torch.float64
    )

    align_count = 0

    model.eval()

    with torch.inference_mode():

        for batch in loader:

            batch = move_batch_to_device(
                batch,
                device
            )

            target = batch["target"]

            out = model(
                ego_cameras=batch["ego_cameras"],
                rsu_cameras=batch["rsu_cameras"],
                ego_cam_mask=batch["ego_cam_mask"],
                rsu_cam_mask=batch["rsu_cam_mask"],
                ego_lidar_bev=batch["ego_lidar_bev"],
                rsu_lidar_bev=batch["rsu_lidar_bev"]
            )

            logits = out["segmentation_logits"]

            pred = torch.argmax(
                logits,
                dim=1
            )

            valid_mask = (
                target != ignore_index
            )

            if not valid_mask.any():
                continue

            valid_target = target[
                valid_mask
            ]

            valid_pred = pred[
                valid_mask
            ]

            total_valid_pixels += int(
                valid_target.numel()
            )

            total_correct_pixels += int(
                (
                    valid_target
                    == valid_pred
                ).sum().item()
            )

            valid_target_cpu = (
                valid_target
                .detach()
                .cpu()
                .long()
            )

            valid_pred_cpu = (
                valid_pred
                .detach()
                .cpu()
                .long()
            )

            keep = (
                (valid_target_cpu >= 0)
                & (valid_target_cpu < num_classes)
                & (valid_pred_cpu >= 0)
                & (valid_pred_cpu < num_classes)
            )

            valid_target_cpu = (
                valid_target_cpu[keep]
            )

            valid_pred_cpu = (
                valid_pred_cpu[keep]
            )

            indices = (
                num_classes
                * valid_target_cpu
                + valid_pred_cpu
            )

            confusion += torch.bincount(
                indices,
                minlength=(
                    num_classes
                    * num_classes
                )
            ).reshape(
                num_classes,
                num_classes
            )

            alignment_params = (
                out["alignment_out"]
                ["alignment_params"]
                .detach()
                .cpu()
                .double()
            )

            align_sum += (
                alignment_params.sum(dim=0)
            )

            align_sq_sum += (
                (alignment_params ** 2)
                .sum(dim=0)
            )

            align_abs_sum += (
                alignment_params
                .abs()
                .sum(dim=0)
            )

            align_count += (
                alignment_params.shape[0]
            )

    intersection = torch.diag(
        confusion
    ).float()

    gt_pixels_per_class = (
        confusion.sum(dim=1).float()
    )

    pred_pixels_per_class = (
        confusion.sum(dim=0).float()
    )

    union = (
        gt_pixels_per_class
        + pred_pixels_per_class
        - intersection
    )

    # mIoU over classes present in the ground truth
    gt_present_classes = (
        gt_pixels_per_class > 0
    )

    iou_per_class = torch.zeros(
        num_classes,
        dtype=torch.float32
    )

    iou_per_class[
        gt_present_classes
    ] = (
        intersection[
            gt_present_classes
        ]
        /
        union[
            gt_present_classes
        ].clamp(min=1)
    )

    if gt_present_classes.any():

        miou = float(
            iou_per_class[
                gt_present_classes
            ].mean().item()
        )

    else:

        miou = 0.0

    pixel_acc = (
        total_correct_pixels
        / max(total_valid_pixels, 1)
    )

    align_mean = (
        align_sum
        / max(align_count, 1)
    )

    align_abs_mean = (
        align_abs_sum
        / max(align_count, 1)
    )

    align_var = (
        align_sq_sum
        / max(align_count, 1)
        - align_mean ** 2
    )

    align_std = torch.sqrt(
        torch.clamp(
            align_var,
            min=0.0
        )
    )

    results = {
        "pixel_accuracy": float(pixel_acc),
        "mIoU": float(miou),

        "total_valid_pixels":
            int(total_valid_pixels),

        "total_correct_pixels":
            int(total_correct_pixels),

        "valid_classes": [
            int(i)
            for i in range(num_classes)
            if bool(gt_present_classes[i])
        ],

        "per_class_iou": {
            str(i):
                float(iou_per_class[i].item())
            for i in range(num_classes)
            if bool(gt_present_classes[i])
        },

        "alignment_statistics": {
            "dx_mean":
                float(align_mean[0].item()),

            "dy_mean":
                float(align_mean[1].item()),

            "angle_mean":
                float(align_mean[2].item()),

            "dx_std":
                float(align_std[0].item()),

            "dy_std":
                float(align_std[1].item()),

            "angle_std":
                float(align_std[2].item()),

            "dx_abs_mean":
                float(align_abs_mean[0].item()),

            "dy_abs_mean":
                float(align_abs_mean[1].item()),

            "angle_abs_mean":
                float(align_abs_mean[2].item())
        }
    }

    return results, confusion

In [ ]:
# Full Stage-2 validation evaluation

stage2_val_results, stage2_val_confusion = (
    evaluate_stage2_miou(
        model=stage2_model,
        loader=val_loader,
        num_classes=NUM_CLASSES,
        device=device,
        ignore_index=IGNORE_INDEX
    )
)

print("Stage-2 validation results:")
print("Pixel accuracy:", stage2_val_results["pixel_accuracy"])
print("mIoU:", stage2_val_results["mIoU"])

In [ ]:
# Full Stage-2 test evaluation

stage2_test_results, stage2_test_confusion = (
    evaluate_stage2_miou(
        model=stage2_model,
        loader=test_loader,
        num_classes=NUM_CLASSES,
        device=device,
        ignore_index=IGNORE_INDEX
    )
)

print("Stage-2 test results:")
print("Pixel accuracy:", stage2_test_results["pixel_accuracy"])
print("mIoU:", stage2_test_results["mIoU"])

In [ ]:
# Full Stage-2 validation evaluation

stage2_eval_results, stage2_confusion = evaluate_stage2_miou(
    model=stage2_model,
    loader=val_loader,
    num_classes=NUM_CLASSES,
    device=device,
    ignore_index=IGNORE_INDEX
)

print("Stage-2 validation results:")
print("Pixel accuracy:", stage2_eval_results["pixel_accuracy"])
print("mIoU:", stage2_eval_results["mIoU"])
print("Valid classes:", stage2_eval_results["valid_classes"])
print(
    "Alignment statistics:",
    stage2_eval_results["alignment_statistics"]
)

stage2_eval_save = {
    "checkpoint": str(best_stage2_path),
    "best_epoch": int(stage2_checkpoint["epoch"]),
    "train_loss": float(stage2_checkpoint["train_loss"]),
    "val_loss": float(stage2_checkpoint["val_loss"]),
    "train_acc": float(stage2_checkpoint["train_acc"]),
    "val_acc": float(stage2_checkpoint["val_acc"]),
    "validation_pixel_accuracy": float(
        stage2_eval_results["pixel_accuracy"]
    ),
    "validation_miou": float(
        stage2_eval_results["mIoU"]
    ),
    "total_valid_pixels": int(
        stage2_eval_results["total_valid_pixels"]
    ),
    "total_correct_pixels": int(
        stage2_eval_results["total_correct_pixels"]
    ),
    "valid_classes": stage2_eval_results["valid_classes"],
    "per_class_iou": stage2_eval_results["per_class_iou"],
    "alignment_statistics":
        stage2_eval_results["alignment_statistics"]
}

stage2_eval_path = (
    OUTPUT_DIR
    / "stage2_aligned_validation_results.json"
)

stage2_confusion_path = (
    OUTPUT_DIR
    / "stage2_aligned_validation_confusion_matrix.pt"
)

with open(stage2_eval_path, "w") as f:
    json.dump(
        stage2_eval_save,
        f,
        indent=4
    )

torch.save(
    stage2_confusion,
    stage2_confusion_path
)

print("Saved Stage-2 validation results.")

In [ ]:
# Cell 87: Stage-3 temporal recovery configuration

HISTORY_LEN = 3

# Probability of current RSU feature unavailability during training
STAGE3_TRAIN_DROP_PROB = 0.5

# Weight of auxiliary feature-recovery loss
STAGE3_RECOVERY_LOSS_WEIGHT = 0.1

STAGE3_BATCH_SIZE = (
    4 if torch.cuda.is_available() else 1
)

STAGE3_NUM_EPOCHS = 10
STAGE3_EARLY_STOP_PATIENCE = 3

STAGE3_DIR = (
    OUTPUT_DIR
    / "stage3_temporal_recovery"
)

STAGE3_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Ensure scene-level separation is preserved
assert set(train_scenes).isdisjoint(
    set(val_scenes)
)

print("Stage-3 configuration:")
print("History length:", HISTORY_LEN)
print(
    "Training drop probability:",
    STAGE3_TRAIN_DROP_PROB
)
print(
    "Recovery-loss weight:",
    STAGE3_RECOVERY_LOSS_WEIGHT
)
print(
    "Batch size:",
    STAGE3_BATCH_SIZE
)
print(
    "Maximum epochs:",
    STAGE3_NUM_EPOCHS
)

In [ ]:
# Stage-3 temporal sequences

def build_temporal_sequences(
    samples,
    history_len=HISTORY_LEN
):
    """
    Build temporally continuous sequences:

        [t-history_len, ..., t-1] -> t

    Each sample is represented as:
        (ego_agent_id, rsu_agent_id, scene_id, frame_id)
    """

    sample_set = set(samples)
    sequences = []

    sorted_samples = sorted(
        samples,
        key=lambda x: (
            x[2],  # scene
            x[0],  # ego agent
            x[1],  # RSU agent
            x[3]   # frame
        )
    )

    for current in sorted_samples:

        (
            ego_agent_id,
            rsu_agent_id,
            scene_id,
            frame_id
        ) = current

        history = [
            (
                ego_agent_id,
                rsu_agent_id,
                scene_id,
                frame_id - h
            )
            for h in range(
                history_len,
                0,
                -1
            )
        ]

        if all(
            history_key in sample_set
            for history_key in history
        ):
            sequences.append({
                "current": current,
                "history": history
            })

    return sequences


#  Build all valid temporal sequences

all_stage3_sequences = build_temporal_sequences(
    samples=matched_train_samples,
    history_len=HISTORY_LEN
)



# Apply existing scene-level train/validation split

stage3_train_sequences = [
    seq
    for seq in all_stage3_sequences
    if seq["current"][2] in train_scenes
]

stage3_val_sequences = [
    seq
    for seq in all_stage3_sequences
    if seq["current"][2] in val_scenes
]


rng = random.Random(SEED)

rng.shuffle(stage3_train_sequences)
rng.shuffle(stage3_val_sequences)



# Leakage and availability checks

train_scene_set_stage3 = {
    seq["current"][2]
    for seq in stage3_train_sequences
}

val_scene_set_stage3 = {
    seq["current"][2]
    for seq in stage3_val_sequences
}

assert train_scene_set_stage3.isdisjoint(
    val_scene_set_stage3
), "Stage-3 scene leakage detected."

assert len(stage3_train_sequences) > 0, (
    "No Stage-3 training sequences found."
)

assert len(stage3_val_sequences) > 0, (
    "No Stage-3 validation sequences found."
)

# Build unique sample keys required for feature caching

stage3_unique_keys = set()

for seq in (
    stage3_train_sequences
    + stage3_val_sequences
):

    stage3_unique_keys.add(
        seq["current"]
    )

    stage3_unique_keys.update(
        seq["history"]
    )

stage3_unique_keys = sorted(
    stage3_unique_keys
)


print(
    "Stage-3 training sequences:",
    len(stage3_train_sequences)
)

print(
    "Stage-3 validation sequences:",
    len(stage3_val_sequences)
)

print(
    "Unique samples required for Stage-3 cache:",
    len(stage3_unique_keys)
)

print("No Stage-3 scene leakage detected.")

In [ ]:
# Precompute Stage-2 features for Stage-3 recovery

stage3_cache_path = (
    STAGE3_DIR
    / f"stage3_feature_cache_H{HISTORY_LEN}.pt"
)

FORCE_REBUILD_STAGE3_CACHE = False


def precompute_stage2_features_for_keys(
    sample_keys,
    cache_path,
    force_rebuild=False
):
    """
    Precompute Stage-2 ego-local and aligned-RSU features
    required by Stage-3 temporal recovery.
    """

    sample_keys = [
        tuple(int(x) for x in key)
        for key in sample_keys
    ]


    # Load existing cache when available

    if cache_path.exists() and not force_rebuild:

        print(
            "Loading existing Stage-3 feature cache:",
            cache_path
        )

        return torch.load(
            cache_path,
            map_location="cpu"
        )

    # Build feature dataset

    feature_dataset = Stage1V2XDatasetSafe(
        samples=sample_keys,
        image_size=(256, 256),
        bev_size=(256, 256),
        ignore_index=IGNORE_INDEX
    )

    feature_loader = DataLoader(
        feature_dataset,
        batch_size=1,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY
    )

    feature_cache = {}

    stage2_model.eval()

    # Extract frozen Stage-2 features

    with torch.inference_mode():

        for batch_idx, batch in enumerate(feature_loader):

            sample_key = sample_keys[batch_idx]

            batch_device = move_batch_to_device(
                batch,
                device
            )

            out = stage2_model(
                ego_cameras=batch_device["ego_cameras"],
                rsu_cameras=batch_device["rsu_cameras"],
                ego_cam_mask=batch_device["ego_cam_mask"],
                rsu_cam_mask=batch_device["rsu_cam_mask"],
                ego_lidar_bev=batch_device["ego_lidar_bev"],
                rsu_lidar_bev=batch_device["rsu_lidar_bev"]
            )

            ego_local_feature = (
                out["ego_local_out"]
                ["fused_feature"][0]
                .detach()
                .cpu()
                .half()
            )

            aligned_rsu_feature = (
                out["alignment_out"]
                ["aligned_rsu_feature"][0]
                .detach()
                .cpu()
                .half()
            )

            target = (
                batch["target"][0]
                .detach()
                .cpu()
                .long()
            )

            feature_cache[sample_key] = {
                "ego_feature":
                    ego_local_feature,

                "aligned_rsu_feature":
                    aligned_rsu_feature,

                "target":
                    target
            }

            if (batch_idx + 1) % 500 == 0:
                print(
                    f"Cached "
                    f"{batch_idx + 1}/"
                    f"{len(feature_loader)} samples"
                )

    # Save cache

    temp_path = cache_path.with_suffix(
        ".tmp.pt"
    )

    torch.save(
        feature_cache,
        temp_path
    )

    temp_path.replace(
        cache_path
    )

    print(
        "Saved Stage-3 feature cache:",
        cache_path
    )

    return feature_cache


stage3_feature_cache = (
    precompute_stage2_features_for_keys(
        sample_keys=stage3_unique_keys,
        cache_path=stage3_cache_path,
        force_rebuild=FORCE_REBUILD_STAGE3_CACHE
    )
)

print(
    "Stage-3 cached samples:",
    len(stage3_feature_cache)
)

In [ ]:
# Stage-3 temporal feature dataset

class Stage3TemporalFeatureDataset(Dataset):
    """
    Dataset of cached Stage-2 features for temporal RSU recovery.

    Returns:
        ego_feature:          [64, 32, 32]
        current_rsu_feature:  [64, 32, 32]
        history_rsu_features: [HISTORY_LEN, 64, 32, 32]
        target:               [256, 256]
        sample_info:          [4]
    """

    def __init__(
        self,
        sequences,
        feature_cache,
        history_len=HISTORY_LEN
    ):
        self.sequences = sequences
        self.feature_cache = feature_cache
        self.history_len = history_len

        if not self.sequences:
            raise ValueError(
                "Stage-3 dataset received zero sequences."
            )

        # Validate sequence/cache consistency
        for seq in self.sequences:

            current_key = seq["current"]
            history_keys = seq["history"]

            if len(history_keys) != self.history_len:
                raise ValueError(
                    "Temporal history length mismatch."
                )

            if current_key not in self.feature_cache:
                raise KeyError(
                    f"Current key missing from cache: {current_key}"
                )

            for history_key in history_keys:
                if history_key not in self.feature_cache:
                    raise KeyError(
                        f"History key missing from cache: {history_key}"
                    )

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):

        seq = self.sequences[idx]

        current_key = seq["current"]
        history_keys = seq["history"]

        current_data = self.feature_cache[
            current_key
        ]

        ego_feature = (
            current_data["ego_feature"]
            .float()
        )

        current_rsu_feature = (
            current_data["aligned_rsu_feature"]
            .float()
        )

        target = (
            current_data["target"]
            .long()
        )

        history_rsu_features = torch.stack(
            [
                self.feature_cache[key]
                ["aligned_rsu_feature"]
                .float()
                for key in history_keys
            ],
            dim=0
        )

        sample_info = torch.tensor(
            current_key,
            dtype=torch.long
        )

        return {
            "ego_feature": ego_feature,
            "current_rsu_feature": current_rsu_feature,
            "history_rsu_features": history_rsu_features,
            "target": target,
            "sample_info": sample_info
        }


stage3_train_dataset = Stage3TemporalFeatureDataset(
    sequences=stage3_train_sequences,
    feature_cache=stage3_feature_cache,
    history_len=HISTORY_LEN
)

stage3_val_dataset = Stage3TemporalFeatureDataset(
    sequences=stage3_val_sequences,
    feature_cache=stage3_feature_cache,
    history_len=HISTORY_LEN
)

stage3_loader_generator = torch.Generator()
stage3_loader_generator.manual_seed(SEED)

stage3_train_loader = DataLoader(
    stage3_train_dataset,
    batch_size=STAGE3_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    generator=stage3_loader_generator
)

stage3_val_loader = DataLoader(
    stage3_val_dataset,
    batch_size=STAGE3_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

print(
    "Stage-3 training sequences:",
    len(stage3_train_dataset)
)

print(
    "Stage-3 validation sequences:",
    len(stage3_val_dataset)
)

print(
    "Stage-3 batch size:",
    STAGE3_BATCH_SIZE
)

In [ ]:
# Temporal RSU feature recovery network

class TemporalRSURecoveryNet(nn.Module):
    """
    Recover the current RSU BEV feature from:
        1. current ego BEV feature
        2. K previous aligned RSU BEV features
    Inputs:
        ego_feature: [B, C, H, W]
        rsu_history: [B, K, C, H, W]
    Output:
        recovered_rsu_feature: [B, C, H, W]
    """

    def __init__(
        self,
        feature_channels=64,
        history_len=3
    ):
        super().__init__()

        self.feature_channels = feature_channels
        self.history_len = history_len

        in_channels = (
            feature_channels
            * (history_len + 1)
        )

        self.temporal_encoder = nn.Sequential(
            ConvBNReLU(
                in_channels,
                128,
                kernel_size=3
            ),
            ConvBNReLU(
                128,
                128,
                kernel_size=3
            ),
            ConvBNReLU(
                128,
                64,
                kernel_size=3
            )
        )

        self.residual_head = nn.Conv2d(
            64,
            feature_channels,
            kernel_size=1
        )

        self.gate_head = nn.Sequential(
            nn.Conv2d(
                64,
                feature_channels,
                kernel_size=1
            ),
            nn.Sigmoid()
        )

        # Initialize recovery close to the most recent
        # historical RSU feature.
        nn.init.zeros_(
            self.residual_head.weight
        )
        nn.init.zeros_(
            self.residual_head.bias
        )

        nn.init.constant_(
            self.gate_head[0].bias,
            -2.0
        )

    def forward(
        self,
        ego_feature,
        rsu_history
    ):

        B, K, C, H, W = rsu_history.shape

        if K != self.history_len:
            raise ValueError(
                f"Expected {self.history_len} history frames, "
                f"received {K}."
            )

        history_flat = rsu_history.reshape(
            B,
            K * C,
            H,
            W
        )

        x = torch.cat(
            [
                ego_feature,
                history_flat
            ],
            dim=1
        )

        temporal_feature = self.temporal_encoder(
            x
        )

        residual = self.residual_head(
            temporal_feature
        )

        gate = self.gate_head(
            temporal_feature
        )

        last_history_feature = (
            rsu_history[:, -1]
        )

        recovered_rsu_feature = (
            last_history_feature
            + gate * residual
        )

        return recovered_rsu_feature


temporal_recovery_net = TemporalRSURecoveryNet(
    feature_channels=64,
    history_len=HISTORY_LEN
).to(device)

print("Temporal RSU recovery network initialized.")

In [ ]:
# Stage-3 temporal recovery V2I model

class Stage3TemporalRecoveryV2IModel(nn.Module):
    """
    Stage-3 V2I model with temporal RSU feature recovery.

    When the current aligned RSU feature is unavailable,
    the model substitutes a temporally recovered RSU feature
    estimated from the current ego feature and K previous
    aligned RSU features.

    Inputs:
        ego_feature:          [B, 64, 32, 32]
        current_rsu_feature:  [B, 64, 32, 32]
        history_rsu_features: [B, K, 64, 32, 32]
        drop_mask:            [B] or broadcastable equivalent

    Output:
        segmentation_logits:  [B, num_classes, 256, 256]
    """

    def __init__(
        self,
        feature_channels=64,
        num_classes=23,
        history_len=3
    ):
        super().__init__()

        self.feature_channels = feature_channels
        self.num_classes = num_classes
        self.history_len = history_len

        self.recovery_net = TemporalRSURecoveryNet(
            feature_channels=feature_channels,
            history_len=history_len
        )

        self.v2i_fusion = InitialV2IFusion(
            feature_channels=feature_channels
        )

        self.seg_decoder = BEVSegmentationDecoder(
            in_channels=feature_channels,
            num_classes=num_classes
        )

    def forward(
        self,
        ego_feature,
        current_rsu_feature,
        history_rsu_features,
        drop_mask
    ):

        B = ego_feature.shape[0]

        if drop_mask.ndim <= 2:
            drop_mask = drop_mask.view(
                B, 1, 1, 1
            )

        drop_mask = drop_mask.to(
            device=ego_feature.device,
            dtype=ego_feature.dtype
        )

        recovered_rsu_feature = self.recovery_net(
            ego_feature=ego_feature,
            rsu_history=history_rsu_features
        )

        # drop_mask = 1 -> current RSU unavailable
        # drop_mask = 0 -> current RSU available
        selected_rsu_feature = (
            drop_mask * recovered_rsu_feature
            + (1.0 - drop_mask) * current_rsu_feature
        )

        v2i_out = self.v2i_fusion(
            ego_feature=ego_feature,
            rsu_feature=selected_rsu_feature
        )

        segmentation_logits = self.seg_decoder(
            v2i_out["cooperative_feature"]
        )

        return {
            "segmentation_logits": segmentation_logits,
            "recovered_rsu_feature": recovered_rsu_feature,
            "selected_rsu_feature": selected_rsu_feature,
            "v2i_out": v2i_out,
            "drop_mask": drop_mask
        }


stage3_model = Stage3TemporalRecoveryV2IModel(
    feature_channels=64,
    num_classes=NUM_CLASSES,
    history_len=HISTORY_LEN
).to(device)


# Initialize cooperative fusion and decoder
# from the best trained Stage-2 model.
stage3_model.v2i_fusion.load_state_dict(
    stage2_model.v2i_fusion.state_dict()
)

stage3_model.seg_decoder.load_state_dict(
    stage2_model.seg_decoder.state_dict()
)

print("Stage-3 temporal recovery model initialized.")

In [ ]:
# Freeze Stage-2 modules for Stage-3 training

for parameter in stage3_model.v2i_fusion.parameters():
    parameter.requires_grad = False

for parameter in stage3_model.seg_decoder.parameters():
    parameter.requires_grad = False

for parameter in stage3_model.recovery_net.parameters():
    parameter.requires_grad = True


trainable_params = sum(
    p.numel()
    for p in stage3_model.parameters()
    if p.requires_grad
)

frozen_params = sum(
    p.numel()
    for p in stage3_model.parameters()
    if not p.requires_grad
)

print("Stage-3 trainable parameters:", trainable_params)
print("Stage-3 frozen parameters:", frozen_params)
print("Only the temporal recovery network is trainable.")

In [ ]:
batch = next(iter(stage3_train_loader))

ego_feature = batch["ego_feature"].to(device)
current_rsu_feature = batch["current_rsu_feature"].to(device)
history_rsu_features = batch["history_rsu_features"].to(device)

B = ego_feature.shape[0]

# Test with all RSU features dropped
drop_mask = torch.ones(B, device=device)

stage3_model.eval()

with torch.no_grad():
    out = stage3_model(
        ego_feature=ego_feature,
        current_rsu_feature=current_rsu_feature,
        history_rsu_features=history_rsu_features,
        drop_mask=drop_mask
    )

print("Stage 3 forward test:")
print("segmentation_logits:", out["segmentation_logits"].shape)
print("recovered_rsu_feature:", out["recovered_rsu_feature"].shape)
print("selected_rsu_feature:", out["selected_rsu_feature"].shape)
print("drop_mask:", out["drop_mask"].shape)

In [ ]:
# Stage-3 training/validation functions


def move_stage3_batch_to_device(batch, device):
    return {
        "ego_feature": batch["ego_feature"].to(
            device,
            non_blocking=True
        ),
        "current_rsu_feature": batch["current_rsu_feature"].to(
            device,
            non_blocking=True
        ),
        "history_rsu_features": batch["history_rsu_features"].to(
            device,
            non_blocking=True
        ),
        "target": batch["target"].long().to(
            device,
            non_blocking=True
        ),
        "sample_info": batch["sample_info"]
    }


def make_drop_mask(
    batch_size,
    drop_prob,
    device
):
    """
    drop_mask = 1: current RSU feature is unavailable
    drop_mask = 0: current RSU feature is available
    """

    if drop_prob <= 0.0:
        return torch.zeros(
            (batch_size, 1, 1, 1),
            dtype=torch.float32,
            device=device
        )

    if drop_prob >= 1.0:
        return torch.ones(
            (batch_size, 1, 1, 1),
            dtype=torch.float32,
            device=device
        )

    return (
        torch.rand(
            (batch_size, 1, 1, 1),
            device=device
        ) < drop_prob
    ).float()


def run_one_epoch_stage3(
    model,
    loader,
    optimizer=None,
    train=True,
    drop_prob=0.5,
    ignore_index=IGNORE_INDEX,
    class_weights=None,
    recovery_loss_weight=0.1
):
    """
    Run one Stage-3 training or validation epoch.

    Objective:
        segmentation loss
        + recovery_loss_weight * feature-recovery loss

    Returns:
        avg_loss
        avg_seg_loss
        avg_rec_loss
        pixel_acc
        actual_drop_rate
    """

    model.train() if train else model.eval()

    model_device = next(
        model.parameters()
    ).device

    weights = (
        class_weights.to(model_device)
        if class_weights is not None
        else None
    )

    segmentation_criterion = nn.CrossEntropyLoss(
        weight=weights,
        ignore_index=ignore_index
    )

    recovery_criterion = nn.MSELoss()

    total_loss = 0.0
    total_seg_loss = 0.0
    total_rec_loss = 0.0

    total_correct = 0
    total_valid_pixels = 0

    total_dropped = 0
    total_samples = 0
    used_batches = 0

    with torch.set_grad_enabled(train):

        for batch_idx, batch in enumerate(loader):

            batch = move_stage3_batch_to_device(
                batch,
                model_device
            )

            batch_size = batch[
                "ego_feature"
            ].shape[0]

            drop_mask = make_drop_mask(
                batch_size=batch_size,
                drop_prob=drop_prob,
                device=model_device
            )

            target = batch["target"]

            valid_mask = (
                target != ignore_index
            )

            valid_pixels = (
                valid_mask.sum().item()
            )

            if valid_pixels == 0:
                continue

            if train:
                optimizer.zero_grad(
                    set_to_none=True
                )

            out = model(
                ego_feature=batch["ego_feature"],
                current_rsu_feature=batch[
                    "current_rsu_feature"
                ],
                history_rsu_features=batch[
                    "history_rsu_features"
                ],
                drop_mask=drop_mask
            )

            logits = out[
                "segmentation_logits"
            ]

            recovered_rsu_feature = out[
                "recovered_rsu_feature"
            ]

            seg_loss = segmentation_criterion(
                logits,
                target
            )

            # Recover the current aligned RSU representation
            rec_loss = recovery_criterion(
                recovered_rsu_feature,
                batch["current_rsu_feature"]
            )

            loss = (
                seg_loss
                + recovery_loss_weight
                * rec_loss
            )

            if train:
                loss.backward()

                torch.nn.utils.clip_grad_norm_(
                    [
                        p
                        for p in model.parameters()
                        if p.requires_grad
                    ],
                    max_norm=5.0
                )

                optimizer.step()

            with torch.no_grad():

                pred = torch.argmax(
                    logits,
                    dim=1
                )

                correct = (
                    (pred == target)
                    & valid_mask
                ).sum().item()

            total_loss += loss.item()
            total_seg_loss += seg_loss.item()
            total_rec_loss += rec_loss.item()

            total_correct += correct
            total_valid_pixels += valid_pixels

            total_dropped += int(
                drop_mask.sum().item()
            )

            total_samples += batch_size
            used_batches += 1

            if train and (batch_idx + 1) % 100 == 0:
                print(
                    f"Batch {batch_idx + 1}/{len(loader)} | "
                    f"Loss: {loss.item():.4f} | "
                    f"Seg: {seg_loss.item():.4f} | "
                    f"Rec: {rec_loss.item():.6f}"
                )

    avg_loss = (
        total_loss
        / max(used_batches, 1)
    )

    avg_seg_loss = (
        total_seg_loss
        / max(used_batches, 1)
    )

    avg_rec_loss = (
        total_rec_loss
        / max(used_batches, 1)
    )

    pixel_acc = (
        total_correct
        / max(total_valid_pixels, 1)
    )

    actual_drop_rate = (
        total_dropped
        / max(total_samples, 1)
    )

    return (
        avg_loss,
        avg_seg_loss,
        avg_rec_loss,
        pixel_acc,
        actual_drop_rate
    )


print("Stage-3 training functions ready.")

In [ ]:
def evaluate_stage3_at_drop_rate(
    model,
    loader,
    drop_prob,
    num_classes,
    ignore_index=IGNORE_INDEX
):
    """
    Evaluate Stage-3 perception at a fixed current-RSU
    packet-drop probability.

    drop_prob = 0.0:
        current aligned RSU feature is always available.

    drop_prob = 1.0:
        current aligned RSU feature is always unavailable,
        so temporal recovery is always used.
    """

    model.eval()
    model_device = next(model.parameters()).device

    confusion = torch.zeros(
        (num_classes, num_classes),
        dtype=torch.int64
    )

    recovery_criterion = nn.MSELoss()

    total_valid_pixels = 0
    total_correct_pixels = 0
    total_recovery_loss = 0.0

    total_dropped = 0
    total_samples = 0
    used_batches = 0

    with torch.inference_mode():

        for batch in loader:

            batch = move_stage3_batch_to_device(
                batch,
                model_device
            )

            batch_size = batch[
                "ego_feature"
            ].shape[0]

            drop_mask = make_drop_mask(
                batch_size=batch_size,
                drop_prob=drop_prob,
                device=model_device
            )

            target = batch["target"]

            valid_mask = (
                target != ignore_index
            )

            if not valid_mask.any():
                continue

            out = model(
                ego_feature=batch["ego_feature"],
                current_rsu_feature=batch[
                    "current_rsu_feature"
                ],
                history_rsu_features=batch[
                    "history_rsu_features"
                ],
                drop_mask=drop_mask
            )

            logits = out["segmentation_logits"]

            pred = torch.argmax(
                logits,
                dim=1
            )

            recovery_loss = recovery_criterion(
                out["recovered_rsu_feature"],
                batch["current_rsu_feature"]
            )

            valid_target = target[
                valid_mask
            ]

            valid_pred = pred[
                valid_mask
            ]

            total_valid_pixels += int(
                valid_target.numel()
            )

            total_correct_pixels += int(
                (
                    valid_target
                    == valid_pred
                ).sum().item()
            )

            valid_target_cpu = (
                valid_target
                .detach()
                .cpu()
                .long()
            )

            valid_pred_cpu = (
                valid_pred
                .detach()
                .cpu()
                .long()
            )

            indices = (
                num_classes
                * valid_target_cpu
                + valid_pred_cpu
            )

            confusion += torch.bincount(
                indices,
                minlength=(
                    num_classes
                    * num_classes
                )
            ).reshape(
                num_classes,
                num_classes
            )

            total_recovery_loss += float(
                recovery_loss.item()
            )

            total_dropped += int(
                drop_mask.sum().item()
            )

            total_samples += batch_size
            used_batches += 1

    intersection = torch.diag(
        confusion
    ).float()

    gt_pixels_per_class = (
        confusion.sum(dim=1).float()
    )

    pred_pixels_per_class = (
        confusion.sum(dim=0).float()
    )

    union = (
        gt_pixels_per_class
        + pred_pixels_per_class
        - intersection
    )

    gt_present_classes = (
        gt_pixels_per_class > 0
    )

    iou_per_class = torch.zeros(
        num_classes,
        dtype=torch.float32
    )

    iou_per_class[
        gt_present_classes
    ] = (
        intersection[
            gt_present_classes
        ]
        /
        union[
            gt_present_classes
        ].clamp(min=1)
    )

    if gt_present_classes.any():
        miou = float(
            iou_per_class[
                gt_present_classes
            ].mean().item()
        )
    else:
        miou = 0.0

    pixel_acc = (
        total_correct_pixels
        / max(total_valid_pixels, 1)
    )

    avg_recovery_loss = (
        total_recovery_loss
        / max(used_batches, 1)
    )

    actual_drop_rate = (
        total_dropped
        / max(total_samples, 1)
    )

    return {
        "drop_prob": float(drop_prob),
        "actual_drop_rate": float(actual_drop_rate),
        "pixel_acc": float(pixel_acc),
        "miou": float(miou),
        "avg_recovery_loss": float(avg_recovery_loss),
        "total_valid_pixels": int(total_valid_pixels),
        "total_correct_pixels": int(total_correct_pixels),

        "valid_classes": [
            int(i)
            for i in range(num_classes)
            if bool(gt_present_classes[i])
        ],

        "per_class_iou": {
            str(i):
                float(iou_per_class[i].item())
            for i in range(num_classes)
            if bool(gt_present_classes[i])
        }
    }

In [ ]:
# Train Stage-3 temporal recovery model

stage3_history_path = (
    STAGE3_DIR
    / "stage3_training_history.json"
)

stage3_last_path = (
    STAGE3_DIR
    / "stage3_temporal_last_checkpoint.pth"
)

stage3_best_path = (
    STAGE3_DIR
    / "stage3_temporal_best_checkpoint.pth"
)


# Only the temporal recovery network is trainable.
optimizer_stage3 = torch.optim.AdamW(
    stage3_model.recovery_net.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler_stage3 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_stage3,
    mode="min",
    factor=0.5,
    patience=1
)

best_val_loss = float("inf")
epochs_without_improvement = 0
stage3_history = []

print("Starting Stage-3 temporal recovery training.")


for epoch in range(1, STAGE3_NUM_EPOCHS + 1):

    print(
        f"\nStage-3 Epoch "
        f"{epoch}/{STAGE3_NUM_EPOCHS}"
    )

    (
        train_loss,
        train_seg_loss,
        train_rec_loss,
        train_acc,
        train_drop_actual
    ) = run_one_epoch_stage3(
        model=stage3_model,
        loader=stage3_train_loader,
        optimizer=optimizer_stage3,
        train=True,
        drop_prob=STAGE3_TRAIN_DROP_PROB,
        ignore_index=IGNORE_INDEX,
        class_weights=class_weights,
        recovery_loss_weight=STAGE3_RECOVERY_LOSS_WEIGHT
    )

    (
        val_loss,
        val_seg_loss,
        val_rec_loss,
        val_acc,
        val_drop_actual
    ) = run_one_epoch_stage3(
        model=stage3_model,
        loader=stage3_val_loader,
        optimizer=None,
        train=False,
        drop_prob=STAGE3_TRAIN_DROP_PROB,
        ignore_index=IGNORE_INDEX,
        class_weights=class_weights,
        recovery_loss_weight=STAGE3_RECOVERY_LOSS_WEIGHT
    )

    scheduler_stage3.step(
        val_loss
    )

    current_lr = (
        optimizer_stage3
        .param_groups[0]["lr"]
    )

    print(
        f"Train Loss: {train_loss:.4f} | "
        f"Seg: {train_seg_loss:.4f} | "
        f"Rec: {train_rec_loss:.6f} | "
        f"Acc: {train_acc:.4f} | "
        f"Drop: {train_drop_actual:.3f}"
    )

    print(
        f"Val Loss: {val_loss:.4f} | "
        f"Seg: {val_seg_loss:.4f} | "
        f"Rec: {val_rec_loss:.6f} | "
        f"Acc: {val_acc:.4f} | "
        f"Drop: {val_drop_actual:.3f} | "
        f"LR: {current_lr:.8f}"
    )

    epoch_record = {
        "epoch": int(epoch),

        "train_loss": float(train_loss),
        "train_seg_loss": float(train_seg_loss),
        "train_rec_loss": float(train_rec_loss),
        "train_acc": float(train_acc),
        "train_drop_actual": float(train_drop_actual),

        "val_loss": float(val_loss),
        "val_seg_loss": float(val_seg_loss),
        "val_rec_loss": float(val_rec_loss),
        "val_acc": float(val_acc),
        "val_drop_actual": float(val_drop_actual),

        "learning_rate": float(current_lr),
        "history_len": int(HISTORY_LEN),
        "train_drop_prob": float(STAGE3_TRAIN_DROP_PROB),
        "recovery_loss_weight": float(
            STAGE3_RECOVERY_LOSS_WEIGHT
        )
    }

    stage3_history.append(
        epoch_record
    )

    with open(
        stage3_history_path,
        "w"
    ) as f:
        json.dump(
            stage3_history,
            f,
            indent=4
        )

    checkpoint = {
        "epoch": int(epoch),

        "model_state_dict":
            stage3_model.state_dict(),

        "optimizer_state_dict":
            optimizer_stage3.state_dict(),

        "train_loss": float(train_loss),
        "train_seg_loss": float(train_seg_loss),
        "train_rec_loss": float(train_rec_loss),
        "train_acc": float(train_acc),
        "train_drop_actual": float(train_drop_actual),

        "val_loss": float(val_loss),
        "val_seg_loss": float(val_seg_loss),
        "val_rec_loss": float(val_rec_loss),
        "val_acc": float(val_acc),
        "val_drop_actual": float(val_drop_actual),

        "num_classes": int(NUM_CLASSES),

        "class_weights":
            class_weights.detach().cpu(),

        "history":
            stage3_history,

        "history_len":
            int(HISTORY_LEN),

        "train_drop_prob":
            float(STAGE3_TRAIN_DROP_PROB),

        "recovery_loss_weight":
            float(STAGE3_RECOVERY_LOSS_WEIGHT)
    }

    # Save most recent checkpoint
    torch.save(
        checkpoint,
        stage3_last_path
    )

    # Save checkpoint with minimum validation loss
    if val_loss < best_val_loss:

        best_val_loss = val_loss
        epochs_without_improvement = 0

        torch.save(
            checkpoint,
            stage3_best_path
        )

        print(
            "Saved new best Stage-3 checkpoint."
        )

    else:

        epochs_without_improvement += 1

        print(
            "No validation improvement: "
            f"{epochs_without_improvement}/"
            f"{STAGE3_EARLY_STOP_PATIENCE}"
        )

    if (
        epochs_without_improvement
        >= STAGE3_EARLY_STOP_PATIENCE
    ):
        print(
            "Stage-3 early stopping triggered."
        )
        break


print("\nStage-3 training completed.")
print(
    "Best validation loss:",
    best_val_loss
)

In [ ]:
# Load best Stage-3 checkpoint

best_stage3_path = (
    STAGE3_DIR
    / "stage3_temporal_best_checkpoint.pth"
)

assert best_stage3_path.exists(), (
    f"Best Stage-3 checkpoint not found: "
    f"{best_stage3_path}"
)

stage3_checkpoint = torch.load(
    best_stage3_path,
    map_location=device
)

stage3_model.load_state_dict(
    stage3_checkpoint["model_state_dict"]
)

stage3_model.to(device)
stage3_model.eval()

print("Loaded best Stage-3 checkpoint.")
print("Best epoch:", stage3_checkpoint["epoch"])
print("Validation loss:", stage3_checkpoint["val_loss"])
print("Validation pixel accuracy:", stage3_checkpoint["val_acc"])

print(
    "History length:",
    stage3_checkpoint.get("history_len")
)

print(
    "Training drop probability:",
    stage3_checkpoint.get("train_drop_prob")
)

print(
    "Recovery-loss weight:",
    stage3_checkpoint.get("recovery_loss_weight")
)

In [ ]:
# Held-out test sequences

stage3_test_sequences = [
    seq
    for seq in all_stage3_sequences
    if seq["current"][2] in test_scenes
]

assert len(stage3_test_sequences) > 0

stage3_test_scene_set = {
    seq["current"][2]
    for seq in stage3_test_sequences
}

assert stage3_test_scene_set.isdisjoint(
    train_scene_set_stage3
)

assert stage3_test_scene_set.isdisjoint(
    val_scene_set_stage3
)

# Collect all current and historical samples needed by test sequences
stage3_test_unique_keys = set()

for seq in stage3_test_sequences:

    stage3_test_unique_keys.add(
        seq["current"]
    )

    stage3_test_unique_keys.update(
        seq["history"]
    )

stage3_test_unique_keys = sorted(
    stage3_test_unique_keys
)

print(
    "Stage-3 test sequences:",
    len(stage3_test_sequences)
)

print(
    "Stage-3 unique test cache samples:",
    len(stage3_test_unique_keys)
)

print("No Stage-3 test scene leakage detected.")

In [ ]:
# Precompute Stage-2 features for Stage-3 test set

stage3_test_cache_path = (
    STAGE3_DIR
    / f"stage3_test_feature_cache_H{HISTORY_LEN}.pt"
)

stage3_test_feature_cache = (
    precompute_stage2_features_for_keys(
        sample_keys=stage3_test_unique_keys,
        cache_path=stage3_test_cache_path,
        force_rebuild=False
    )
)

print(
    "Stage-3 test cache size:",
    len(stage3_test_feature_cache)
)

In [ ]:
# Held-out test DataLoader

stage3_test_dataset = Stage3TemporalFeatureDataset(
    sequences=stage3_test_sequences,
    feature_cache=stage3_test_feature_cache,
    history_len=HISTORY_LEN
)

stage3_test_loader = DataLoader(
    stage3_test_dataset,
    batch_size=STAGE3_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

print(
    "Stage-3 test sequences:",
    len(stage3_test_dataset)
)

In [ ]:
# Evaluation under packet-drop rates

STAGE3_DROP_RATES = [
    0.0,
    0.3,
    0.5,
    0.7,
    1.0
]

stage3_drop_results = {}
stage3_drop_confusions = {}


for drop_rate in STAGE3_DROP_RATES:

    print(
        f"\nEvaluating Stage-3 at "
        f"packet-drop rate = {drop_rate:.1f}"
    )

    result = evaluate_stage3_at_drop_rate(
        model=stage3_model,
        loader=stage3_val_loader,
        drop_prob=drop_rate,
        num_classes=NUM_CLASSES,
        ignore_index=IGNORE_INDEX
    )

    key = f"{drop_rate:.1f}"

    stage3_drop_results[key] = result

    print(
        "Pixel accuracy:",
        result["pixel_acc"]
    )

    print(
        "mIoU:",
        result["miou"]
    )

    print(
        "Actual drop rate:",
        result["actual_drop_rate"]
    )

    print(
        "Recovery loss:",
        result["avg_recovery_loss"]
    )


# validation results


stage3_drop_results_path = (
    STAGE3_DIR
    / "stage3_validation_packet_drop_results.json"
)

with open(
    stage3_drop_results_path,
    "w"
) as f:
    json.dump(
        stage3_drop_results,
        f,
        indent=4
    )

print(
    "\nSaved Stage-3 packet-drop validation results:",
    stage3_drop_results_path
)

In [ ]:
# Temporal recovery visualization
# qualitative visualization under full RSU packet drop

import numpy as np
import torch
from PIL import Image, ImageDraw

assert "OUTPUT_DIR" in globals(), "OUTPUT_DIR is not defined."
assert "stage3_model" in globals(), "stage3_model is not defined."
assert "stage3_val_loader" in globals(), "stage3_val_loader is not defined."
assert "move_stage3_batch_to_device" in globals(), "move_stage3_batch_to_device is not defined."
assert "num_classes" in globals(), "num_classes is not defined."

FIG_DIR = OUTPUT_DIR / "paper_figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("Stage 3 figure directory:")
print(FIG_DIR)


def feature_to_pil_stage3(feature_tensor, size=(256, 256)):
    feat = feature_tensor.detach().cpu()

    if feat.ndim == 4:
        feat = feat[0]

    arr = feat.mean(dim=0).numpy()
    arr = arr - arr.min()

    if arr.max() > 0:
        arr = arr / arr.max()

    arr = (arr * 255).astype(np.uint8)

    img = Image.fromarray(arr, mode="L").convert("RGB")
    img = img.resize(size, Image.NEAREST)

    return img


def label_to_pil_stage3(label, num_classes, ignore_index=255, size=(256, 256)):
    arr = label.detach().cpu().numpy().astype(np.int32)

    preview = np.zeros_like(arr, dtype=np.uint8)

    valid = arr != ignore_index
    if valid.any():
        # fixed grayscale mapping for consistency
        preview[valid] = (
            arr[valid].astype(np.float32) / max(num_classes - 1, 1) * 255.0
        ).astype(np.uint8)

    img = Image.fromarray(preview, mode="L").convert("RGB")
    img = img.resize(size, Image.NEAREST)

    return img


def make_titled_canvas_stage3(img, title, cell_w=280, cell_h=320):
    canvas = Image.new("RGB", (cell_w, cell_h), color=(255, 255, 255))
    img = img.resize((256, 256), Image.NEAREST)
    canvas.paste(img, (12, 45))

    draw = ImageDraw.Draw(canvas)
    draw.text((10, 12), title, fill=(0, 0, 0))

    return canvas


def save_grid_stage3(items, save_path, cols=3, cell_w=280, cell_h=320):
    rows = int(np.ceil(len(items) / cols))

    grid = Image.new(
        "RGB",
        (cols * cell_w, rows * cell_h),
        color=(255, 255, 255)
    )

    for i, (title, img) in enumerate(items):
        x = (i % cols) * cell_w
        y = (i // cols) * cell_h

        canvas = make_titled_canvas_stage3(
            img,
            title,
            cell_w=cell_w,
            cell_h=cell_h
        )

        grid.paste(canvas, (x, y))

    grid.save(save_path)
    print("Saved:", save_path)


#  Get one validation batch

batch = next(iter(stage3_val_loader))
model_device = next(stage3_model.parameters()).device
batch_device = move_stage3_batch_to_device(batch, model_device)

stage3_model.eval()

# full packet drop
drop_mask = torch.ones(
    (batch_device["ego_feature"].shape[0], 1, 1, 1),
    dtype=torch.float32,
    device=model_device
)

with torch.no_grad():
    out = stage3_model(
        ego_feature=batch_device["ego_feature"],
        current_rsu_feature=batch_device["current_rsu_feature"],
        history_rsu_features=batch_device["history_rsu_features"],
        drop_mask=drop_mask
    )

logits = out["segmentation_logits"]
pred = torch.argmax(logits, dim=1)[0].detach().cpu()
target = batch["target"][0].detach().cpu()

ego_feat = batch_device["ego_feature"][0]
current_rsu = batch_device["current_rsu_feature"][0]
recovered_rsu = out["recovered_rsu_feature"][0]

history_1 = batch_device["history_rsu_features"][0, 0]
history_2 = batch_device["history_rsu_features"][0, 1]
history_3 = batch_device["history_rsu_features"][0, 2]

recovery_error = torch.abs(
    recovered_rsu.detach().cpu() - current_rsu.detach().cpu()
)

pred_masked = pred.clone()
pred_masked[target == 255] = 255

sample_info = batch["sample_info"][0].tolist()
print("Stage 3 visualization sample_info:", sample_info)
print("Prediction unique classes:", torch.unique(pred))
print("Target unique classes:", torch.unique(target))

items = [
    ("Ego Feature", feature_to_pil_stage3(ego_feat)),
    ("History RSU t-3", feature_to_pil_stage3(history_1)),
    ("History RSU t-2", feature_to_pil_stage3(history_2)),
    ("History RSU t-1", feature_to_pil_stage3(history_3)),
    ("True Current RSU", feature_to_pil_stage3(current_rsu)),
    ("Recovered RSU", feature_to_pil_stage3(recovered_rsu)),
    ("Recovery Error", feature_to_pil_stage3(recovery_error)),
    ("GT BEV Label", label_to_pil_stage3(target, num_classes=num_classes)),
    ("Prediction", label_to_pil_stage3(pred_masked, num_classes=num_classes)),
]

save_path = FIG_DIR / "stage3_temporal_recovery_visualization.png"

save_grid_stage3(
    items,
    save_path=save_path,
    cols=3,
    cell_w=280,
    cell_h=320
)

print("\nStage 3 temporal recovery visualization saved to:")
print(save_path)

In [ ]:
# Stage-4 configuration
# D3QN-based adaptive fusion selection
from collections import deque


# Stage-4 output directory


STAGE4_DIR = (
    OUTPUT_DIR
    / "stage4_d3qn_adaptive_fusion"
)

STAGE4_DIR.mkdir(
    parents=True,
    exist_ok=True
)


#  Action space

STAGE4_NUM_ACTIONS = 4

ACTION_NAMES = {
    0: "ego_only",
    1: "current_rsu_fusion",
    2: "recovered_rsu_fusion",
    3: "mixed_current_recovered_rsu"
}


#  Packet-drop settings

STAGE4_TRAIN_DROP_RATES = [
    0.0,
    0.3,
    0.5,
    0.7,
    1.0
]

STAGE4_EVAL_DROP_RATES = [
    0.0,
    0.3,
    0.5,
    0.7,
    1.0
]


# D3QN training hyperparameters

STAGE4_NUM_EPOCHS = 20

STAGE4_BATCH_SIZE_REPLAY = 64
STAGE4_REPLAY_CAPACITY = 10000

# One-step contextual decision
STAGE4_GAMMA = 0.0

STAGE4_LR = 1e-4

STAGE4_EPS_START = 1.0
STAGE4_EPS_END = 0.05
STAGE4_EPS_DECAY = 0.80


# Reward parameters

REWARD_LOSS_WEIGHT = 0.15
INVALID_CURRENT_RSU_PENALTY = 0.5


# Load and freeze best Stage-3 perception model

best_stage3_path = (
    STAGE3_DIR
    / "stage3_temporal_best_checkpoint.pth"
)

assert best_stage3_path.exists(), (
    f"Best Stage-3 checkpoint not found: "
    f"{best_stage3_path}"
)

stage3_checkpoint = torch.load(
    best_stage3_path,
    map_location=device
)

stage3_model.load_state_dict(
    stage3_checkpoint["model_state_dict"]
)

stage3_model.to(device)
stage3_model.eval()

for parameter in stage3_model.parameters():
    parameter.requires_grad = False


# Summary

print("Stage-4 adaptive fusion configuration")
print(" ")

print(
    "Actions:",
    ACTION_NAMES
)

print(
    "Training drop rates:",
    STAGE4_TRAIN_DROP_RATES
)

print(
    "Evaluation drop rates:",
    STAGE4_EVAL_DROP_RATES
)

print(
    "D3QN gamma:",
    STAGE4_GAMMA
)

print(
    "Replay capacity:",
    STAGE4_REPLAY_CAPACITY
)

print(
    "Replay batch size:",
    STAGE4_BATCH_SIZE_REPLAY
)

print(
    "Learning rate:",
    STAGE4_LR
)

print(
    "Loaded Stage-3 checkpoint epoch:",
    stage3_checkpoint.get("epoch")
)

In [ ]:
# Stage-4 D3QN state extraction

def feature_mean(x):
    return x.mean(dim=(1, 2, 3))


def feature_std(x):
    return x.std(
        dim=(1, 2, 3),
        unbiased=False
    )


def feature_energy(x):
    return x.abs().mean(
        dim=(1, 2, 3)
    )


def feature_mse(a, b):
    return ((a - b) ** 2).mean(
        dim=(1, 2, 3)
    )


def build_stage4_state(
    ego_feature,
    current_rsu_feature,
    recovered_rsu_feature,
    history_rsu_features,
    drop_mask,
    drop_prob
):
    """
    Construct the 18-D state used by the D3QN selector.

    The current RSU feature is masked when its packet is
    unavailable to prevent information leakage.
    """

    batch_size = ego_feature.shape[0]

    drop_flag = drop_mask.reshape(
        batch_size
    ).to(
        device=ego_feature.device,
        dtype=ego_feature.dtype
    )

    availability = 1.0 - drop_flag

    # Hide unavailable current-RSU features.
    current_rsu_observed = (
        current_rsu_feature
        * availability.view(
            batch_size,
            1,
            1,
            1
        )
    )

    last_history = (
        history_rsu_features[:, -1]
    )

    if history_rsu_features.shape[1] > 1:

        history_motion = (
            (
                history_rsu_features[:, 1:]
                - history_rsu_features[:, :-1]
            ) ** 2
        ).mean(
            dim=(1, 2, 3, 4)
        )

    else:

        history_motion = torch.zeros(
            batch_size,
            device=ego_feature.device,
            dtype=ego_feature.dtype
        )

    drop_prob_tensor = torch.full(
        (batch_size,),
        float(drop_prob),
        device=ego_feature.device,
        dtype=ego_feature.dtype
    )

    state = torch.stack(
        [
            # Communication status
            drop_flag,
            drop_prob_tensor,
            availability,

            # Ego representation
            feature_mean(ego_feature),
            feature_std(ego_feature),
            feature_energy(ego_feature),

            # Observed current-RSU representation
            feature_mean(current_rsu_observed),
            feature_std(current_rsu_observed),
            feature_energy(current_rsu_observed),

            # Recovered-RSU representation
            feature_mean(recovered_rsu_feature),
            feature_std(recovered_rsu_feature),
            feature_energy(recovered_rsu_feature),

            # Historical RSU information
            feature_energy(last_history),

            # Cross-feature consistency
            feature_mse(
                current_rsu_observed,
                recovered_rsu_feature
            ),
            feature_mse(
                last_history,
                recovered_rsu_feature
            ),
            feature_mse(
                ego_feature,
                current_rsu_observed
            ),
            feature_mse(
                ego_feature,
                recovered_rsu_feature
            ),

            # Temporal variation
            history_motion,
        ],
        dim=1
    )

    state = torch.nan_to_num(
        state,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    return state


STAGE4_STATE_DIM = 18

print(
    "Stage-4 state dimension:",
    STAGE4_STATE_DIM
)

In [ ]:
# Stage-4 action execution

def compute_stage4_all_action_logits(
    stage3_model,
    ego_feature,
    current_rsu_feature,
    history_rsu_features,
    drop_mask,
    mixed_alpha=0.5
):
    """
    Compute segmentation logits for all four Stage-4 actions.

    Action 0: ego-only
    Action 1: current-RSU fusion
    Action 2: recovered-RSU fusion
    Action 3: mixed current/recovered-RSU fusion

    When the current RSU packet is unavailable, its feature is
    masked to zero to prevent information leakage.
    """

    batch_size = ego_feature.shape[0]

    drop_mask = drop_mask.reshape(
        batch_size,
        1,
        1,
        1
    ).to(
        device=ego_feature.device,
        dtype=ego_feature.dtype
    )

    availability = 1.0 - drop_mask

    current_rsu_observed = (
        current_rsu_feature
        * availability
    )

    zero_rsu = torch.zeros_like(
        current_rsu_feature
    )

    # Temporal RSU recovery

    recovered_rsu_feature = (
        stage3_model.recovery_net(
            ego_feature=ego_feature,
            rsu_history=history_rsu_features
        )
    )

    # Action 0: ego-only

    ego_only_out = stage3_model.v2i_fusion(
        ego_feature=ego_feature,
        rsu_feature=zero_rsu
    )

    logits_ego_only = stage3_model.seg_decoder(
        ego_only_out["cooperative_feature"]
    )

    # Action 1: current RSU

    current_out = stage3_model.v2i_fusion(
        ego_feature=ego_feature,
        rsu_feature=current_rsu_observed
    )

    logits_current = stage3_model.seg_decoder(
        current_out["cooperative_feature"]
    )

    # Action 2: recovered RSU

    recovered_out = stage3_model.v2i_fusion(
        ego_feature=ego_feature,
        rsu_feature=recovered_rsu_feature
    )

    logits_recovered = stage3_model.seg_decoder(
        recovered_out["cooperative_feature"]
    )

    # Action 3: mixed current + recovered RSU

    mixed_rsu_feature = (
        mixed_alpha
        * current_rsu_observed
        + (1.0 - mixed_alpha)
        * recovered_rsu_feature
    )

    mixed_out = stage3_model.v2i_fusion(
        ego_feature=ego_feature,
        rsu_feature=mixed_rsu_feature
    )

    logits_mixed = stage3_model.seg_decoder(
        mixed_out["cooperative_feature"]
    )

    logits_all = torch.stack(
        [
            logits_ego_only,
            logits_current,
            logits_recovered,
            logits_mixed
        ],
        dim=1
    )

    return {
        "logits_all": logits_all,
        "recovered_rsu_feature":
            recovered_rsu_feature,
        "current_rsu_observed":
            current_rsu_observed,
        "mixed_rsu_feature":
            mixed_rsu_feature,
        "drop_mask":
            drop_mask,
        "availability":
            availability
    }


def select_logits_by_action(
    logits_all,
    actions
):
    """
    Select segmentation logits corresponding to the D3QN
    action selected for each sample.
    """

    batch_size, num_actions, channels, height, width = (
        logits_all.shape
    )

    actions = actions.to(
        device=logits_all.device,
        dtype=torch.long
    )

    assert actions.shape == (batch_size,)

    assert (
        actions.min().item() >= 0
        and actions.max().item() < num_actions
    )

    action_index = actions.view(
        batch_size,
        1,
        1,
        1,
        1
    ).expand(
        batch_size,
        1,
        channels,
        height,
        width
    )

    selected_logits = torch.gather(
        logits_all,
        dim=1,
        index=action_index
    ).squeeze(1)

    return selected_logits

In [ ]:
# Stage-4 immediate reward

def compute_stage4_reward(
    logits,
    target,
    actions,
    drop_mask,
    class_weights=None,
    ignore_index=IGNORE_INDEX
):
    """
    Compute the immediate reward for each selected fusion action.

    Reward:
        pixel accuracy
        - weighted segmentation-loss penalty
        - invalid current-RSU action penalty

    Action 1 is invalid when the current RSU packet is unavailable.
    """

    batch_size = logits.shape[0]

    actions = actions.to(
        device=logits.device,
        dtype=torch.long
    )

    target = target.to(
        device=logits.device,
        dtype=torch.long
    )

    drop_flag = drop_mask.reshape(
        batch_size
    ).to(
        device=logits.device,
        dtype=logits.dtype
    )

    weights = (
        class_weights.to(logits.device)
        if class_weights is not None
        else None
    )

    # Per-sample segmentation loss

    ce_map = F.cross_entropy(
        logits,
        target,
        weight=weights,
        ignore_index=ignore_index,
        reduction="none"
    )

    valid_mask = (
        target != ignore_index
    )

    valid_pixels = (
        valid_mask
        .reshape(batch_size, -1)
        .sum(dim=1)
        .clamp(min=1)
    )

    sample_loss = (
        ce_map
        .reshape(batch_size, -1)
        .sum(dim=1)
        / valid_pixels.float()
    )

    # Per-sample pixel accuracy

    pred = torch.argmax(
        logits,
        dim=1
    )

    sample_correct = (
        (pred == target)
        & valid_mask
    ).reshape(
        batch_size,
        -1
    ).sum(dim=1)

    sample_acc = (
        sample_correct.float()
        / valid_pixels.float()
    )

    # Invalid-action penalty

    dropped = (
        drop_flag > 0.5
    )

    invalid_current_action = (
        (actions == 1)
        & dropped
    )

    invalid_penalty = (
        INVALID_CURRENT_RSU_PENALTY
        * invalid_current_action.float()
    )

    # Immediate reward

    reward = (
        sample_acc
        - REWARD_LOSS_WEIGHT * sample_loss
        - invalid_penalty
    )

    reward = torch.nan_to_num(
        reward,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    return {
        "reward": reward.detach(),
        "sample_loss": sample_loss.detach(),
        "sample_acc": sample_acc.detach(),
        "invalid_action":
            invalid_current_action.detach(),
        "invalid_penalty":
            invalid_penalty.detach()
    }


print(
    "Stage-4 reward: "
    "pixel accuracy - loss penalty - invalid-action penalty"
)

In [ ]:
# Stage-4 replay buffer

class ReplayBuffer:

    def __init__(self, capacity):
        self.buffer = deque(
            maxlen=capacity
        )

    def __len__(self):
        return len(self.buffer)

    def push_batch(
        self,
        states,
        actions,
        rewards,
        next_states,
        dones
    ):
        """
        Store a batch of transitions in the replay buffer.
        """

        states = torch.nan_to_num(
            states.detach().cpu().float(),
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        )

        next_states = torch.nan_to_num(
            next_states.detach().cpu().float(),
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        )

        actions = (
            actions.detach()
            .cpu()
            .long()
            .reshape(-1)
        )

        rewards = (
            rewards.detach()
            .cpu()
            .float()
            .reshape(-1)
        )

        dones = (
            dones.detach()
            .cpu()
            .float()
            .reshape(-1)
        )

        batch_size = states.shape[0]

        assert actions.shape[0] == batch_size
        assert rewards.shape[0] == batch_size
        assert next_states.shape[0] == batch_size
        assert dones.shape[0] == batch_size

        for i in range(batch_size):

            self.buffer.append(
                (
                    states[i],
                    actions[i],
                    rewards[i],
                    next_states[i],
                    dones[i]
                )
            )

    def sample(self, batch_size):
        """
        Randomly sample a minibatch of transitions.
        """

        assert len(self.buffer) >= batch_size

        samples = random.sample(
            self.buffer,
            batch_size
        )

        (
            states,
            actions,
            rewards,
            next_states,
            dones
        ) = zip(*samples)

        return {
            "states":
                torch.stack(states).float(),

            "actions":
                torch.stack(actions).long().reshape(-1),

            "rewards":
                torch.stack(rewards).float().reshape(-1),

            "next_states":
                torch.stack(next_states).float(),

            "dones":
                torch.stack(dones).float().reshape(-1)
        }


stage4_replay_buffer = ReplayBuffer(
    capacity=STAGE4_REPLAY_CAPACITY
)

print(
    "Stage-4 replay buffer capacity:",
    STAGE4_REPLAY_CAPACITY
)

In [ ]:
# Dueling Q-network and D3QN agent

class DuelingQNetwork(nn.Module):
    def __init__(
        self,
        state_dim,
        num_actions,
        hidden_dim=128
    ):
        super().__init__()

        self.state_dim = state_dim
        self.num_actions = num_actions

        self.feature = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(inplace=True)
        )

        self.value_stream = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, 1)
        )

        self.advantage_stream = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, num_actions)
        )

    def forward(self, state):

        assert state.ndim == 2, (
            f"state should be [B, state_dim], got {state.shape}"
        )

        assert state.shape[1] == self.state_dim, (
            f"Expected state_dim={self.state_dim}, "
            f"got {state.shape[1]}"
        )

        state = torch.nan_to_num(
            state.float(),
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        )

        x = self.feature(state)

        value = self.value_stream(x)
        advantage = self.advantage_stream(x)

        q_values = (
            value
            + advantage
            - advantage.mean(
                dim=1,
                keepdim=True
            )
        )

        return q_values


class D3QNAgent:
    def __init__(
        self,
        state_dim,
        num_actions,
        device,
        lr=1e-4,
        gamma=0.0
    ):
        self.state_dim = state_dim
        self.num_actions = num_actions
        self.device = device
        self.gamma = gamma

        self.q_net = DuelingQNetwork(
            state_dim=state_dim,
            num_actions=num_actions
        ).to(device)

        self.target_net = DuelingQNetwork(
            state_dim=state_dim,
            num_actions=num_actions
        ).to(device)

        self.target_net.load_state_dict(
            self.q_net.state_dict()
        )

        self.optimizer = torch.optim.AdamW(
            self.q_net.parameters(),
            lr=lr,
            weight_decay=1e-4
        )

    def select_action(
        self,
        states,
        epsilon=0.0,
        invalid_action_mask=None
    ):

        states = states.to(
            self.device
        ).float()

        batch_size = states.shape[0]

        with torch.no_grad():

            q_values = self.q_net(
                states
            )

            if invalid_action_mask is not None:

                invalid_action_mask = (
                    invalid_action_mask.to(
                        device=self.device,
                        dtype=torch.bool
                    )
                )

                q_values = q_values.masked_fill(
                    invalid_action_mask,
                    -1e9
                )

            greedy_actions = torch.argmax(
                q_values,
                dim=1
            )

        random_actions = torch.randint(
            low=0,
            high=self.num_actions,
            size=(batch_size,),
            device=self.device
        )

        if invalid_action_mask is not None:

            for i in range(batch_size):

                valid_actions = torch.where(
                    ~invalid_action_mask[i]
                )[0]

                if len(valid_actions) > 0:

                    random_index = torch.randint(
                        low=0,
                        high=len(valid_actions),
                        size=(1,),
                        device=self.device
                    )[0]

                    random_actions[i] = (
                        valid_actions[random_index]
                    )

        explore_mask = (
            torch.rand(
                batch_size,
                device=self.device
            )
            < epsilon
        )

        actions = torch.where(
            explore_mask,
            random_actions,
            greedy_actions
        )

        return actions

    def update(
        self,
        replay_buffer,
        batch_size
    ):

        if len(replay_buffer) < batch_size:
            return None

        batch = replay_buffer.sample(
            batch_size
        )

        states = batch["states"].to(
            self.device
        )

        actions = batch["actions"].to(
            self.device
        )

        rewards = batch["rewards"].to(
            self.device
        )

        q_values = self.q_net(
            states
        )

        current_q = q_values.gather(
            dim=1,
            index=actions.view(-1, 1)
        ).squeeze(1)

        # One-step contextual decision:
        # gamma = 0, therefore y_t = r_t
        target_q = rewards

        loss = F.smooth_l1_loss(
            current_q,
            target_q
        )

        self.optimizer.zero_grad(
            set_to_none=True
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            self.q_net.parameters(),
            max_norm=5.0
        )

        self.optimizer.step()

        return float(
            loss.item()
        )

    def update_target_network(self):

        self.target_net.load_state_dict(
            self.q_net.state_dict()
        )

    def save(self, path):

        torch.save(
            {
                "q_net_state_dict":
                    self.q_net.state_dict(),

                "target_net_state_dict":
                    self.target_net.state_dict(),

                "optimizer_state_dict":
                    self.optimizer.state_dict(),

                "state_dim":
                    self.state_dim,

                "num_actions":
                    self.num_actions,

                "gamma":
                    self.gamma
            },
            path
        )

    def load(
        self,
        path,
        map_location=None
    ):

        checkpoint = torch.load(
            path,
            map_location=(
                map_location
                if map_location is not None
                else self.device
            )
        )

        self.q_net.load_state_dict(
            checkpoint[
                "q_net_state_dict"
            ]
        )

        self.target_net.load_state_dict(
            checkpoint[
                "target_net_state_dict"
            ]
        )

        self.optimizer.load_state_dict(
            checkpoint[
                "optimizer_state_dict"
            ]
        )

        self.q_net.to(
            self.device
        )

        self.target_net.to(
            self.device
        )


stage4_agent = D3QNAgent(
    state_dim=STAGE4_STATE_DIM,
    num_actions=STAGE4_NUM_ACTIONS,
    device=device,
    lr=STAGE4_LR,
    gamma=STAGE4_GAMMA
)

print("D3QN agent ready.")
print("State dimension:", STAGE4_STATE_DIM)
print("Number of actions:", STAGE4_NUM_ACTIONS)
print("Gamma:", STAGE4_GAMMA)
print("Device:", device)

In [ ]:
# Train Stage-4 D3QN adaptive fusion controller

stage4_history = []

epsilon = STAGE4_EPS_START
best_avg_reward = -float("inf")

history_path = (
    STAGE4_DIR
    / "stage4_d3qn_training_history.json"
)

last_path = (
    STAGE4_DIR
    / "stage4_d3qn_last_checkpoint.pth"
)

best_path = (
    STAGE4_DIR
    / "stage4_d3qn_best_checkpoint.pth"
)

model_device = next(
    stage3_model.parameters()
).device


print("Starting Stage-4 D3QN training.")
print(
    "Training epochs:",
    STAGE4_NUM_EPOCHS
)
print(
    "Training drop rates:",
    STAGE4_TRAIN_DROP_RATES
)
print(
    "Replay capacity:",
    STAGE4_REPLAY_CAPACITY
)
print(
    "Replay batch size:",
    STAGE4_BATCH_SIZE_REPLAY
)
print(
    "Gamma:",
    STAGE4_GAMMA
)


# Training loop

for epoch in range(
    1,
    STAGE4_NUM_EPOCHS + 1
):

    print(
        f"\nStage-4 D3QN Epoch "
        f"{epoch}/{STAGE4_NUM_EPOCHS}"
    )

    print(
        f"Epsilon: {epsilon:.4f}"
    )

    stage4_agent.q_net.train()
    stage3_model.eval()

    total_reward = 0.0
    total_sample_loss = 0.0
    total_sample_acc = 0.0

    total_invalid_actions = 0

    total_q_loss = 0.0
    q_updates = 0

    total_samples = 0

    action_counts = {
        action: 0
        for action in range(
            STAGE4_NUM_ACTIONS
        )
    }


    # Iterate over Stage-3 training feature sequences

    for batch_idx, batch in enumerate(
        stage3_train_loader
    ):

        batch_device = (
            move_stage3_batch_to_device(
                batch,
                model_device
            )
        )

        batch_size = (
            batch_device[
                "ego_feature"
            ].shape[0]
        )


        # Sample packet-drop condition

        drop_prob = random.choice(
            STAGE4_TRAIN_DROP_RATES
        )

        drop_mask = make_drop_mask(
            batch_size=batch_size,
            drop_prob=drop_prob,
            device=model_device
        )


        # Frozen perception model:

        with torch.inference_mode():

            action_out = (
                compute_stage4_all_action_logits(
                    stage3_model=stage3_model,

                    ego_feature=
                        batch_device[
                            "ego_feature"
                        ],

                    current_rsu_feature=
                        batch_device[
                            "current_rsu_feature"
                        ],

                    history_rsu_features=
                        batch_device[
                            "history_rsu_features"
                        ],

                    drop_mask=drop_mask
                )
            )

            recovered_rsu_feature = (
                action_out[
                    "recovered_rsu_feature"
                ]
            )


            # Construct 18-D D3QN state


            states = build_stage4_state(
                ego_feature=
                    batch_device[
                        "ego_feature"
                    ],

                current_rsu_feature=
                    batch_device[
                        "current_rsu_feature"
                    ],

                recovered_rsu_feature=
                    recovered_rsu_feature,

                history_rsu_features=
                    batch_device[
                        "history_rsu_features"
                    ],

                drop_mask=drop_mask,
                drop_prob=drop_prob
            )



        # Epsilon-greedy action selection

        actions = stage4_agent.select_action(
            states=states,
            epsilon=epsilon,
            invalid_action_mask=None
        )



        # Select segmentation output of chosen action

        selected_logits = (
            select_logits_by_action(
                logits_all=
                    action_out[
                        "logits_all"
                    ],

                actions=actions
            )
        )


        # Immediate reward

        reward_out = (
            compute_stage4_reward(
                logits=selected_logits,

                target=
                    batch_device[
                        "target"
                    ],

                actions=actions,
                drop_mask=drop_mask,

                class_weights=
                    class_weights,

                ignore_index=
                    IGNORE_INDEX
            )
        )

        rewards = (
            reward_out["reward"]
        )


        # One-step contextual decision
        #
        # gamma = 0.
        # next_state and done are retained only to preserve
        # the standard replay-buffer transition format.

        next_states = states.clone()

        dones = torch.ones(
            batch_size,
            dtype=torch.float32,
            device=model_device
        )


        # Replay buffer

        stage4_replay_buffer.push_batch(
            states=states,
            actions=actions,
            rewards=rewards,
            next_states=next_states,
            dones=dones
        )


        # D3QN optimization

        q_loss = stage4_agent.update(
            replay_buffer=
                stage4_replay_buffer,

            batch_size=
                STAGE4_BATCH_SIZE_REPLAY
        )

        if q_loss is not None:

            total_q_loss += (
                float(q_loss)
            )

            q_updates += 1



        # Training statistics


        total_reward += float(
            rewards.sum().item()
        )

        total_sample_loss += float(
            reward_out[
                "sample_loss"
            ].sum().item()
        )

        total_sample_acc += float(
            reward_out[
                "sample_acc"
            ].sum().item()
        )

        total_invalid_actions += int(
            reward_out[
                "invalid_action"
            ].sum().item()
        )

        total_samples += int(
            batch_size
        )


        for action in range(
            STAGE4_NUM_ACTIONS
        ):

            action_counts[
                action
            ] += int(
                (
                    actions
                    == action
                ).sum().item()
            )



        # Progress report


        if (
            batch_idx + 1
        ) % 50 == 0:

            running_reward = (
                total_reward
                / max(
                    total_samples,
                    1
                )
            )

            running_acc = (
                total_sample_acc
                / max(
                    total_samples,
                    1
                )
            )

            running_invalid_rate = (
                total_invalid_actions
                / max(
                    total_samples,
                    1
                )
            )

            print(
                f"Batch "
                f"{batch_idx + 1}/"
                f"{len(stage3_train_loader)} | "
                f"Reward: "
                f"{running_reward:.4f} | "
                f"Acc: "
                f"{running_acc:.4f} | "
                f"Invalid: "
                f"{running_invalid_rate:.4f} | "
                f"Replay: "
                f"{len(stage4_replay_buffer)}"
            )


    # End-of-epoch statistics

    stage4_agent.update_target_network()


    avg_reward = (
        total_reward
        / max(
            total_samples,
            1
        )
    )

    avg_loss = (
        total_sample_loss
        / max(
            total_samples,
            1
        )
    )

    avg_acc = (
        total_sample_acc
        / max(
            total_samples,
            1
        )
    )

    avg_q_loss = (
        total_q_loss
        / max(
            q_updates,
            1
        )
    )

    invalid_action_rate = (
        total_invalid_actions
        / max(
            total_samples,
            1
        )
    )


    print("\nAction distribution:")

    for action in range(
        STAGE4_NUM_ACTIONS
    ):

        print(
            f"Action {action} "
            f"({ACTION_NAMES[action]}): "
            f"{action_counts[action]}"
        )


    print(
        f"Average reward: "
        f"{avg_reward:.4f} | "
        f"Segmentation loss: "
        f"{avg_loss:.4f} | "
        f"Pixel accuracy: "
        f"{avg_acc:.4f} | "
        f"Q-loss: "
        f"{avg_q_loss:.6f} | "
        f"Invalid-action rate: "
        f"{invalid_action_rate:.4f}"
    )


    # Training history

    epoch_record = {

        "epoch":
            int(epoch),

        "epsilon":
            float(epsilon),

        "avg_reward":
            float(avg_reward),

        "avg_seg_loss":
            float(avg_loss),

        "avg_acc":
            float(avg_acc),

        "avg_q_loss":
            float(avg_q_loss),

        "q_updates":
            int(q_updates),

        "replay_size":
            int(
                len(
                    stage4_replay_buffer
                )
            ),

        "invalid_action_rate":
            float(
                invalid_action_rate
            ),

        "action_counts": {
            str(k): int(v)
            for k, v
            in action_counts.items()
        }
    }


    stage4_history.append(
        epoch_record
    )


    with open(
        history_path,
        "w"
    ) as f:

        json.dump(
            stage4_history,
            f,
            indent=4
        )

    # Checkpoint

    checkpoint = {

        "epoch":
            int(epoch),

        "q_net_state_dict":
            stage4_agent
            .q_net
            .state_dict(),

        "target_net_state_dict":
            stage4_agent
            .target_net
            .state_dict(),

        "optimizer_state_dict":
            stage4_agent
            .optimizer
            .state_dict(),

        "history":
            stage4_history,

        "state_dim":
            int(
                STAGE4_STATE_DIM
            ),

        "num_actions":
            int(
                STAGE4_NUM_ACTIONS
            ),

        "action_names":
            ACTION_NAMES,

        "gamma":
            float(
                STAGE4_GAMMA
            ),

        "train_drop_rates": [
            float(p)
            for p in
            STAGE4_TRAIN_DROP_RATES
        ],

        "epsilon":
            float(epsilon),

        "avg_reward":
            float(avg_reward),

        "avg_seg_loss":
            float(avg_loss),

        "avg_acc":
            float(avg_acc),

        "avg_q_loss":
            float(avg_q_loss),

        "invalid_action_rate":
            float(
                invalid_action_rate
            )
    }


    # Most recent checkpoint
    torch.save(
        checkpoint,
        last_path
    )


    # Best training-reward checkpoint
    if avg_reward > best_avg_reward:

        best_avg_reward = (
            avg_reward
        )

        torch.save(
            checkpoint,
            best_path
        )

        print(
            "Saved new best Stage-4 "
            "training-reward checkpoint."
        )

    else:

        print(
            "Average training reward "
            "did not improve."
        )


    # Epsilon decay


    epsilon = max(
        STAGE4_EPS_END,
        epsilon
        * STAGE4_EPS_DECAY
    )


print("\nStage-4 D3QN training completed.")

print(
    "Best average training reward:",
    best_avg_reward
)

print(
    "Training history:",
    history_path
)

print(
    "Best checkpoint:",
    best_path
)

print(
    "Last checkpoint:",
    last_path
)

In [ ]:
# Load best Stage-4 D3QN checkpoint

best_stage4_path = (
    STAGE4_DIR
    / "stage4_d3qn_best_checkpoint.pth"
)

assert best_stage4_path.exists(), (
    f"Best Stage-4 checkpoint not found: "
    f"{best_stage4_path}"
)

stage4_checkpoint = torch.load(
    best_stage4_path,
    map_location=device
)

stage4_agent.q_net.load_state_dict(
    stage4_checkpoint[
        "q_net_state_dict"
    ]
)

stage4_agent.target_net.load_state_dict(
    stage4_checkpoint[
        "target_net_state_dict"
    ]
)

stage4_agent.q_net.to(device)
stage4_agent.target_net.to(device)

stage4_agent.q_net.eval()
stage4_agent.target_net.eval()


print(
    "Loaded best Stage-4 "
    "training-reward checkpoint."
)

print(
    "Epoch:",
    stage4_checkpoint.get("epoch")
)

print(
    "Average training reward:",
    stage4_checkpoint.get("avg_reward")
)

print(
    "Average segmentation loss:",
    stage4_checkpoint.get("avg_seg_loss")
)

print(
    "Average pixel accuracy:",
    stage4_checkpoint.get("avg_acc")
)

print(
    "Average Q-loss:",
    stage4_checkpoint.get("avg_q_loss")
)

print(
    "Training invalid-action rate:",
    stage4_checkpoint.get(
        "invalid_action_rate"
    )
)

print(
    "State dimension:",
    stage4_checkpoint.get("state_dim")
)

print(
    "Number of actions:",
    stage4_checkpoint.get("num_actions")
)

print(
    "Gamma:",
    stage4_checkpoint.get("gamma")
)

In [ ]:
#  Stage-4 evaluation functions
# D3QN adaptive selector and fixed-action baselines

def update_stage4_confusion(
    confusion,
    pred,
    target,
    num_classes=NUM_CLASSES,
    ignore_index=IGNORE_INDEX
):
    """
    Update the confusion matrix using valid target pixels only.
    """

    pred = pred.reshape(-1)
    target = target.reshape(-1)

    valid = target != ignore_index

    pred = pred[valid]
    target = target[valid]

    if target.numel() == 0:
        return 0, 0

    correct = int(
        (pred == target).sum().item()
    )

    total = int(
        target.numel()
    )

    indices = (
        target * num_classes
        + pred
    )

    confusion += torch.bincount(
        indices.detach().cpu(),
        minlength=(
            num_classes
            * num_classes
        )
    ).reshape(
        num_classes,
        num_classes
    )

    return correct, total


def compute_stage4_metrics_from_confusion(
    confusion,
    total_correct,
    total_valid,
    num_classes=NUM_CLASSES
):
    """
    Compute pixel accuracy and mIoU over classes
    present in the ground truth.
    """

    intersection = torch.diag(
        confusion
    ).float()

    gt_pixels_per_class = (
        confusion.sum(dim=1).float()
    )

    pred_pixels_per_class = (
        confusion.sum(dim=0).float()
    )

    union = (
        gt_pixels_per_class
        + pred_pixels_per_class
        - intersection
    )

    gt_present_classes = (
        gt_pixels_per_class > 0
    )

    iou_per_class = torch.zeros(
        num_classes,
        dtype=torch.float32
    )

    iou_per_class[
        gt_present_classes
    ] = (
        intersection[
            gt_present_classes
        ]
        /
        union[
            gt_present_classes
        ].clamp(min=1)
    )

    if gt_present_classes.any():
        miou = float(
            iou_per_class[
                gt_present_classes
            ].mean().item()
        )
    else:
        miou = 0.0

    pixel_accuracy = (
        total_correct
        / max(total_valid, 1)
    )

    return {
        "pixel_accuracy":
            float(pixel_accuracy),

        "mIoU":
            float(miou),

        "total_valid_pixels":
            int(total_valid),

        "total_correct_pixels":
            int(total_correct),

        "valid_classes": [
            int(i)
            for i in range(num_classes)
            if bool(
                gt_present_classes[i]
            )
        ],

        "per_class_iou": {
            str(i):
                float(
                    iou_per_class[i].item()
                )
            for i in range(num_classes)
            if bool(
                gt_present_classes[i]
            )
        },

        "gt_pixels_per_class": {
            str(i):
                int(
                    gt_pixels_per_class[i].item()
                )
            for i in range(num_classes)
            if bool(
                gt_present_classes[i]
            )
        },

        "pred_pixels_per_class": {
            str(i):
                int(
                    pred_pixels_per_class[i].item()
                )
            for i in range(num_classes)
            if bool(
                gt_present_classes[i]
            )
        }
    }


def evaluate_stage4_policy(
    stage4_agent,
    stage3_model,
    loader,
    drop_prob,
    policy_type="d3qn",
    fixed_action=None,
    num_classes=NUM_CLASSES,
    ignore_index=IGNORE_INDEX
):
    """
    Evaluate either:

        policy_type="d3qn":
            learned adaptive fusion selector

        policy_type="fixed":
            fixed-action baseline

    Fixed actions:
        0 = ego-only
        1 = current-RSU fusion
        2 = recovered-RSU fusion
        3 = mixed current/recovered-RSU fusion
    """

    assert policy_type in {
        "d3qn",
        "fixed"
    }

    if policy_type == "fixed":

        assert fixed_action is not None

        assert (
            0 <= int(fixed_action)
            < STAGE4_NUM_ACTIONS
        )

    stage4_agent.q_net.eval()
    stage3_model.eval()

    model_device = next(
        stage3_model.parameters()
    ).device

    confusion = torch.zeros(
        (
            num_classes,
            num_classes
        ),
        dtype=torch.int64
    )

    action_counts = {
        action: 0
        for action in range(
            STAGE4_NUM_ACTIONS
        )
    }

    total_reward = 0.0
    total_loss = 0.0
    total_sample_acc = 0.0

    total_samples = 0
    total_dropped = 0
    total_invalid_actions = 0

    total_correct = 0
    total_valid = 0


    with torch.inference_mode():

        for batch in loader:

            batch_device = (
                move_stage3_batch_to_device(
                    batch,
                    model_device
                )
            )

            batch_size = (
                batch_device[
                    "ego_feature"
                ].shape[0]
            )



            # Packet-drop realization

            drop_mask = make_drop_mask(
                batch_size=batch_size,
                drop_prob=drop_prob,
                device=model_device
            )


            # Candidate action outputs

            action_out = (
                compute_stage4_all_action_logits(
                    stage3_model=stage3_model,

                    ego_feature=
                        batch_device[
                            "ego_feature"
                        ],

                    current_rsu_feature=
                        batch_device[
                            "current_rsu_feature"
                        ],

                    history_rsu_features=
                        batch_device[
                            "history_rsu_features"
                        ],

                    drop_mask=drop_mask
                )
            )



            # D3QN state

            states = build_stage4_state(
                ego_feature=
                    batch_device[
                        "ego_feature"
                    ],

                current_rsu_feature=
                    batch_device[
                        "current_rsu_feature"
                    ],

                recovered_rsu_feature=
                    action_out[
                        "recovered_rsu_feature"
                    ],

                history_rsu_features=
                    batch_device[
                        "history_rsu_features"
                    ],

                drop_mask=drop_mask,
                drop_prob=drop_prob
            )



            # Policy action


            if policy_type == "d3qn":

                # Greedy evaluation.
                # No hard invalid-action mask is applied so
                # the learned invalid-action rate remains
                # measurable.
                actions = (
                    stage4_agent.select_action(
                        states=states,
                        epsilon=0.0,
                        invalid_action_mask=None
                    )
                    .to(model_device)
                )

            else:

                actions = torch.full(
                    (batch_size,),
                    int(fixed_action),
                    dtype=torch.long,
                    device=model_device
                )



            # Selected segmentation output


            selected_logits = (
                select_logits_by_action(
                    logits_all=
                        action_out[
                            "logits_all"
                        ],

                    actions=actions
                )
            )


            # Reward/statistics


            reward_out = (
                compute_stage4_reward(
                    logits=
                        selected_logits,

                    target=
                        batch_device[
                            "target"
                        ],

                    actions=
                        actions,

                    drop_mask=
                        drop_mask,

                    class_weights=
                        class_weights,

                    ignore_index=
                        ignore_index
                )
            )


            pred = torch.argmax(
                selected_logits,
                dim=1
            )


            correct, valid = (
                update_stage4_confusion(
                    confusion=
                        confusion,

                    pred=
                        pred.detach().cpu(),

                    target=
                        batch_device[
                            "target"
                        ].detach().cpu(),

                    num_classes=
                        num_classes,

                    ignore_index=
                        ignore_index
                )
            )


            total_correct += correct
            total_valid += valid


            for action in range(
                STAGE4_NUM_ACTIONS
            ):

                action_counts[
                    action
                ] += int(
                    (
                        actions
                        == action
                    ).sum().item()
                )


            total_reward += float(
                reward_out[
                    "reward"
                ].sum().item()
            )

            total_loss += float(
                reward_out[
                    "sample_loss"
                ].sum().item()
            )

            total_sample_acc += float(
                reward_out[
                    "sample_acc"
                ].sum().item()
            )

            total_invalid_actions += int(
                reward_out[
                    "invalid_action"
                ].sum().item()
            )

            total_samples += int(
                batch_size
            )

            total_dropped += int(
                drop_mask.sum().item()
            )


    # Aggregate metrics

    metric_result = (
        compute_stage4_metrics_from_confusion(
            confusion=
                confusion,

            total_correct=
                total_correct,

            total_valid=
                total_valid,

            num_classes=
                num_classes
        )
    )


    result = {

        "policy_type":
            policy_type,

        "fixed_action":
            (
                None
                if fixed_action is None
                else int(fixed_action)
            ),

        "drop_prob":
            float(drop_prob),

        "actual_drop_rate":
            float(
                total_dropped
                / max(total_samples, 1)
            ),

        "avg_reward":
            float(
                total_reward
                / max(total_samples, 1)
            ),

        "avg_sample_loss":
            float(
                total_loss
                / max(total_samples, 1)
            ),

        "avg_sample_acc":
            float(
                total_sample_acc
                / max(total_samples, 1)
            ),

        "invalid_action_rate":
            float(
                total_invalid_actions
                / max(total_samples, 1)
            ),

        "action_counts": {
            str(k): int(v)
            for k, v
            in action_counts.items()
        },

        **metric_result
    }

    return result, confusion


print(
    "Stage-4 evaluation functions ready."
)

In [ ]:
# Cell 112: Evaluate Stage 4 D3QN and fixed-action baselines

stage4_eval_results = {}
stage4_eval_confusions = {}

evaluation_loader = stage3_test_loader


def reset_stage4_eval_seed(drop_rate):
    eval_seed = SEED + int(round(float(drop_rate) * 1000))

    torch.manual_seed(eval_seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(eval_seed)


stage4_eval_results["d3qn"] = {}
stage4_eval_confusions["d3qn"] = {}

for drop_rate in STAGE4_EVAL_DROP_RATES:

    print(
        f"\nEvaluating D3QN adaptive policy at "
        f"drop rate {drop_rate:.1f}"
    )

    reset_stage4_eval_seed(drop_rate)

    result, confusion = evaluate_stage4_policy(
        stage4_agent=stage4_agent,
        stage3_model=stage3_model,
        loader=evaluation_loader,
        drop_prob=drop_rate,
        policy_type="d3qn",
        fixed_action=None,
        num_classes=NUM_CLASSES,
        ignore_index=IGNORE_INDEX
    )

    key = f"{drop_rate:.1f}"

    stage4_eval_results["d3qn"][key] = result
    stage4_eval_confusions["d3qn"][key] = confusion

    print("Actual drop rate:", result["actual_drop_rate"])
    print("Pixel accuracy:", result["pixel_accuracy"])
    print("mIoU:", result["mIoU"])
    print("Average reward:", result["avg_reward"])
    print("Invalid action rate:", result["invalid_action_rate"])

    print("Action counts:")

    for action_id in range(STAGE4_NUM_ACTIONS):
        print(
            f"Action {action_id} "
            f"({ACTION_NAMES[action_id]}): "
            f"{result['action_counts'][str(action_id)]}"
        )


stage4_eval_results["fixed"] = {}
stage4_eval_confusions["fixed"] = {}

for action_id in range(STAGE4_NUM_ACTIONS):

    action_name = ACTION_NAMES[action_id]

    stage4_eval_results["fixed"][action_name] = {}
    stage4_eval_confusions["fixed"][action_name] = {}

    for drop_rate in STAGE4_EVAL_DROP_RATES:

        print(
            f"\nEvaluating fixed action {action_id} "
            f"({action_name}) at drop rate {drop_rate:.1f}"
        )

        reset_stage4_eval_seed(drop_rate)

        result, confusion = evaluate_stage4_policy(
            stage4_agent=stage4_agent,
            stage3_model=stage3_model,
            loader=evaluation_loader,
            drop_prob=drop_rate,
            policy_type="fixed",
            fixed_action=action_id,
            num_classes=NUM_CLASSES,
            ignore_index=IGNORE_INDEX
        )

        key = f"{drop_rate:.1f}"

        stage4_eval_results["fixed"][action_name][key] = result
        stage4_eval_confusions["fixed"][action_name][key] = confusion

        print("Actual drop rate:", result["actual_drop_rate"])
        print("Pixel accuracy:", result["pixel_accuracy"])
        print("mIoU:", result["mIoU"])
        print("Average reward:", result["avg_reward"])
        print("Invalid action rate:", result["invalid_action_rate"])


stage4_eval_json_path = (
    STAGE4_DIR
    / "stage4_test_policy_eval_results.json"
)

stage4_eval_confusion_path = (
    STAGE4_DIR
    / "stage4_test_policy_eval_confusions.pt"
)

with open(stage4_eval_json_path, "w") as f:
    json.dump(
        stage4_eval_results,
        f,
        indent=4
    )

torch.save(
    stage4_eval_confusions,
    stage4_eval_confusion_path
)

print("\nSaved Stage 4 test evaluation results:")
print(stage4_eval_json_path)
print(stage4_eval_confusion_path)


print("\nStage 4 D3QN test results")

print(
    f"{'Drop':>8} | "
    f"{'Actual':>8} | "
    f"{'Pixel Acc':>10} | "
    f"{'mIoU':>10} | "
    f"{'Invalid':>10}"
)

for drop_rate in STAGE4_EVAL_DROP_RATES:

    key = f"{drop_rate:.1f}"

    result = stage4_eval_results["d3qn"][key]

    print(
        f"{result['drop_prob']:8.1f} | "
        f"{result['actual_drop_rate']:8.3f} | "
        f"{result['pixel_accuracy']:10.4f} | "
        f"{result['mIoU']:10.4f} | "
        f"{result['invalid_action_rate']:10.4f}"
    )


print("\nStage 4 fixed-policy test results")

for action_id in range(STAGE4_NUM_ACTIONS):

    action_name = ACTION_NAMES[action_id]

    print(
        f"\nAction {action_id}: "
        f"{action_name}"
    )

    for drop_rate in STAGE4_EVAL_DROP_RATES:

        key = f"{drop_rate:.1f}"

        result = (
            stage4_eval_results["fixed"]
            [action_name][key]
        )

        print(
            f"Drop {drop_rate:.1f} | "
            f"mIoU: {result['mIoU']:.4f} | "
            f"Pixel Acc: {result['pixel_accuracy']:.4f} | "
            f"Invalid: {result['invalid_action_rate']:.4f}"
        )